In [ ]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats
import random
import re
from types import SimpleNamespace
import warnings
warnings.filterwarnings('ignore')

MIN_SAMPLE = 15
# The fetch pulls TOP_N archetypes; the first CORE_N by use rate are the "core"
# meta that every standard tab analyses, and the remainder are sleeper
# candidates judged only on their matchups against that core.
CORE_N = 25
SMOOTHING_ALPHA = 2
N_SIMS = 10_000
N_PLAYERS = 256
SWISS_ROUNDS = 8
CUT_THRESHOLD = 16
# Format-shape tabs: the thinnest edge a rock-paper-scissors triangle may
# rest on, and how many strategic families to cut the profile tree into.
STRUCTURE_MIN_TRIAD_GAMES = 50
STRUCTURE_FAMILIES = 5
# Skill Curve tab. A slope through three thin bands extrapolates to absurd end
# points, so the chart takes only decks with enough bands and enough games.
SKILL_MIN_BAND_GAMES = 60
SKILL_MIN_BANDS = 3
SKILL_REPORT_BANDS = 4
SKILL_REPORT_GAMES = 800
# Meta Roles reads a share trend as well as a snapshot. Fewer periods than
# this and a slope is not worth fitting, so the roles fall back to the
# trajectory-free rules (in-person majors are a few discrete weekends).
MIN_TREND_PERIODS = 6
# Resamples behind each deck's "how often does it keep this role" figure.
ROLE_BOOT = 200
# Prediction ledger: stamped on every snapshot. Bump it whenever a rule that
# produces a verdict changes, so the Track Record never silently pools scores made
# under different rules.
LEDGER_RULES_VERSION = '2026-09-19'   # finishing tiers from standings (ledger.STANDINGS_TIERS_SINCE)
# Mispriced Decks: posterior draws behind every equilibrium figure (a single LP
# solution is a vertex and not to be trusted alone), and the games a deck needs
# before its matchup row is worth solving a game over.
NASH_BOOT = 400
NASH_MIN_GAMES = 200
# Share of draws that must agree before a deck is called under- or overplayed.
NASH_CONFIDENCE = 90
# Top Cut / Late Rounds tabs: decided games a deck needs in the slice to be shown.
MIN_CUT_GAMES = 100
MIN_LATE_GAMES = 100
# Conversion tab: the finishing layers every deck is measured at. Any mix works:
# 'U<pct>' is the upper pct% of each event, 'Top<N>' its top N places (counted only
# in events of at least 2N players). Layers are drawn in order of measured depth, and
# each build's own split-half test decides which of them carry signal; the rest are
# greyed rather than hidden.
CONVERSION_LAYERS = ['U50', 'U25', 'U10', 'Top64', 'Top32', 'Top16', 'Top8']
CONV_SPLITS, CONV_PERM = 40, 200     # random event halves, and shuffles behind each layer's p-value
CONV_SHOW = 10                       # most-played decks drawn by default; the rest sit in the legend

DECK_PALETTE = [
    '#4338ca', '#0d9488', '#d97706', '#dc2626', '#7c3aed',
    '#2563eb', '#059669', '#ea580c', '#e11d48', '#8b5cf6',
    '#0284c7', '#16a34a', '#ca8a04', '#be123c', '#6d28d9',
    '#0891b2', '#65a30d', '#c2410c', '#9f1239', '#5b21b6',
    '#0e7490', '#4d7c0f', '#9a3412', '#881337', '#4c1d95',
    '#155e75', '#3f6212', '#7c2d12', '#701a75', '#312e81',
]

# --- The one palette --------------------------------------------------------------
# Every colour the dashboard draws comes from here: Python figures read UI.<name>, the
# page's CSS reads var(--<name>), and its JavaScript reads UI.<name>. Another look (or a
# dark mode) is a second dict with the same keys. "Tournament" look: navy chrome, electric
# blue accent, yellow stripe, Barlow type.
THEME = {
    # chrome + neutrals (navy-tinted slate)
    'navy': '#111a2f', 'navy2': '#182c77', 'stripe': '#ffcb05', 'on_navy': '#adb8cf',
    'ink': '#111a2f', 'text': '#2b3757', 'text2': '#3c4a6d', 'muted': '#55648a',
    'faint': '#64718e', 'rule': '#adb8cf', 'border': '#d2d9e7', 'grid': '#e7ebf4',
    'wash': '#e7ebf4', 'bg': '#f3f5fa', 'surface': '#ffffff',
    'noise': 'rgba(231,235,244,0.72)', 'scrim': 'rgba(17,26,47,0.35)',
    # accent (electric blue)
    'accent': '#1f47f0', 'accent_dark': '#1736c4', 'accent2': '#5b82ff', 'accent_rule': '#bccdff',
    'accent_wash': '#eef3ff', 'accent_wash2': '#dbe5ff', 'accent_ring': 'rgba(31,71,240,0.18)',
    # semantic: values unchanged by the redesign (colorblind-checked, meanings fixed)
    'fav': '#16a34a', 'unfav': '#dc2626', 'fav_bg': '#d1fae5', 'fav_ink': '#047857',
    'unfav_bg': '#fee2e2', 'unfav_ink': '#b91c1c', 'warn': '#b45309', 'warn_bg': '#fef3c7',
    'amber': '#d97706', 'cut': '#e11d48',
    'rose': '#be123c', 'div_neg_soft': '#f87171', 'div_mid': '#f5f5f5', 'div_pos_soft': '#6ee7b7',
    'emerald': '#059669',
    # role / verdict hues
    'teal': '#0d9488', 'cyan': '#0891b2', 'orange': '#ea580c', 'violet': '#7c3aed',
    'violet_ink': '#6d28d9', 'violet_bg': '#ede9fe', 'gold': '#eab308', 'gold_ink': '#a16207',
    'gold_bg': '#fef9c3', 'tier_c': '#f59e0b', 'tier_d': '#ef4444',
}
UI = SimpleNamespace(**THEME)
THEME_CSS = (':root {\n' + ''.join(f'  --{k.replace("_", "-")}: {v};\n' for k, v in THEME.items())
             + "  --font: 'Barlow', 'Segoe UI', system-ui, sans-serif;\n"
             + "  --font-display: 'Barlow Condensed', 'Barlow', 'Segoe UI', sans-serif;\n"
             + '  --radius: 2px;\n}\n')

TIER_COLORS = {'All': UI.accent, 'Upper50': UI.teal, 'Upper25': UI.amber}
TIER_RATING_COLORS = {'S': UI.gold, 'A': UI.emerald, 'B': UI.accent2, 'C': UI.tier_c, 'D': UI.tier_d}
FAVORABLE_COLOR = UI.fav
UNFAVORABLE_COLOR = UI.unfav

HEATMAP_SCALE = [[0, UI.rose], [0.25, UI.div_neg_soft], [0.5, UI.div_mid], [0.75, UI.div_pos_soft], [1, UI.emerald]]

FONT_STACK = "'Barlow', 'Segoe UI', system-ui, -apple-system, sans-serif"
FONT_DISPLAY = "'Barlow Condensed', 'Barlow', 'Segoe UI', sans-serif"

META_TEMPLATE = go.layout.Template(
    layout=go.Layout(
        font=dict(family=FONT_STACK, color=UI.text, size=13),
        title=dict(font=dict(family=FONT_DISPLAY, size=19, color=UI.ink), x=0.02, xanchor='left'),
        paper_bgcolor='rgba(0,0,0,0)',
        plot_bgcolor=UI.surface,
        xaxis=dict(
            gridcolor=UI.grid, gridwidth=1, linecolor=UI.rule, linewidth=1,
            zeroline=False, tickfont=dict(size=11, color=UI.muted),
            title_font=dict(size=13, color=UI.text2),
        ),
        yaxis=dict(
            gridcolor=UI.grid, gridwidth=1, linecolor=UI.rule, linewidth=1,
            zeroline=False, tickfont=dict(size=11, color=UI.muted),
            title_font=dict(size=13, color=UI.text2),
        ),
        hoverlabel=dict(
            bgcolor='white', bordercolor=UI.border,
            font=dict(family=FONT_STACK, size=12, color=UI.ink),
        ),
        legend=dict(
            bgcolor='rgba(255,255,255,0.9)', bordercolor=UI.border, borderwidth=1,
            font=dict(size=11),
        ),
    )
)

import plotly.io as pio
pio.templates['meta'] = META_TEMPLATE
pio.templates.default = 'meta'

def make_short_name(name, max_len=14):
    parts = name.split('-')
    if len(parts) <= 2:
        return name.title()
    abbrev = '-'.join(p[:4].title() for p in parts[:2])
    suffix = '-' + parts[-1] if parts[-1] in ('ex', 'box', 'dri', 'twm', 'scr') else ''
    result = abbrev + suffix
    return result[:max_len] if len(result) > max_len else result

def unique_short_names(decks, max_len=14):
    """Short labels guaranteed unique across a deck set.

    make_short_name() only ever sees one deck and builds its label from the first
    two words, so two decks sharing those words collapse to the same string --
    ogerpon-meganium-arboliva and ogerpon-meganium-hydrapple both came out as
    'Oger-Mega', which put two identical rows and columns on the matchup heatmap.
    Only colliding labels are touched, disambiguated by the first word that
    actually differs, so every other deck keeps exactly the name it had.
    """
    names = {d: make_short_name(d, max_len) for d in decks}
    groups = {}
    for deck, label in names.items():
        groups.setdefault(label, []).append(deck)
    for label, group in groups.items():
        if len(group) < 2:
            continue
        split = [d.split('-') for d in group]
        shared = 0
        while (shared < min(len(parts) for parts in split)
               and len({parts[shared] for parts in split}) == 1):
            shared += 1
        for i, (deck, parts) in enumerate(zip(group, split)):
            extra = parts[shared][:4].title() if shared < len(parts) else str(i + 1)
            names[deck] = f'{label}-{extra}'
    # Belt and braces: if two labels still tie, number them so the map is injective.
    seen = {}
    for deck in sorted(names):
        label = names[deck]
        if label in seen:
            seen[label] += 1
            names[deck] = f'{label}-{seen[label]}'
        else:
            seen[label] = 1
    return names


print("Configuration loaded (v2 Top 50 — polished theme).")

In [ ]:
# === Trainer Hill auto-fetch: setup ===
# Defines paths/constants and decides whether the next cell needs to fetch fresh
# CSVs (NEEDS_FETCH). Two backends write the same nine files:
#
#   'api'     tools.trainerhill.com JSON API, via scripts/th_csv.py. Default.
#             A tier is a server-side `placement` filter, so it cannot silently
#             come back as a copy of another tier -- which is exactly what the
#             scraper did, making every Upper50 export a duplicate of its All
#             counterpart. Also the only backend that populates `inperson`.
#   'scrape'  Playwright against www.trainerhill.com/meta. That site is now
#             deprecated (it banners its replacement) and its DOM has moved
#             under this notebook twice, but it is kept as a cross-check.
# Filenames are static (no date), so each fetch overwrites the prior copy.
# Freshness is determined by file mtime: if every slice file and every trend
# file was modified today, we skip the fetch.
#
# The All slice and the finishing tiers (FILTERS + SKILL_FILTERS) are no longer
# Trainer Hill's: they are built from the events' final standings and pairings
# (scripts/tiers.py, NEEDS_MATCHES below), whichever backend ran. Trainer Hill's
# `placement` filter is the top N PLACES of each division at majors and runs
# 13-28% short of its name online (tasks/todo.md section 10). Trainer Hill still
# supplies the event list, use rates, the trend and the long-window slices.
import json
import os
import sys
import subprocess
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path('scripts').resolve()))

FETCH_BACKEND = 'api'          # 'api' | 'scrape'

# Own data folder so the 25-deck notebook's CSVs stay untouched.
# META_DATA_DIR / META_FETCH_END_DATE (below) let scripts/run_notebook.py build
# the dashboard as it would have looked on an earlier date, into its own folder,
# for the prediction ledger. Unset, the notebook behaves exactly as before.
DATA_DIR = Path(os.environ.get('META_DATA_DIR') or 'data/top50')
DATA_DIR.mkdir(parents=True, exist_ok=True)
FILTERS = ['All', 'Upper50', 'Upper25']
# Two finer placement cuts, used only by the Skill Curve tab. Kept out of
# FILTERS so the three tier-based tabs keep meaning exactly what they did.
# Placement slices nest, so All/Upper50/Upper25 plus these give five DISJOINT
# bands once differenced: 0-5, 5-10, 10-25, 25-50, 50-100.
# A game belongs to a tier when BOTH players finished in the top p% of their event
# (th_csv.TIER_PLACEMENT gives p), ranked as on the Conversion tab.
SKILL_FILTERS = ['P5', 'P10']
ALL_FILTERS = FILTERS + SKILL_FILTERS
PLATFORMS = ['all', 'online', 'inperson']
# Which platform slice the standard analysis cells operate on. Change to
# 'online' or 'inperson' and re-run the analysis cells to switch views.
PLATFORM = 'all'
TOP_N = 50

# --- API backend settings -------------------------------------------------
# The window ends today by default, so the dashboard tracks the current meta.
# Pin FETCH_END_DATE to an ISO date to freeze a slice and make a run
# reproducible; cell 2 records whichever window it used in a manifest.
FETCH_END_DATE = os.environ.get('META_FETCH_END_DATE') or None   # None -> today
FETCH_DAYS = 30
# Velocity needs history, not a snapshot: 30 days is only four weekly buckets,
# which is too few to fit a slope through. Matchups and trend therefore run on
# different windows on purpose.
TREND_DAYS = 90
VELOCITY_BOOT = 300        # multinomial resamples behind each deck's interval
MIN_PLAYERS = 32
API_CACHE = DATA_DIR / '_api_cache'
FETCH_MANIFEST = DATA_DIR / 'fetch-manifest.json'

# --- Prediction ledger --------------------------------------------------------
# Every build records the verdicts it shows (data/ledger/snapshots), and the Track
# Record tab scores each one once the window after it has finished. Historical
# builds (scripts/run_notebook.py) write into the same ledger. The long trend is
# what finds set releases, which no score may straddle.
LEDGER_DIR = Path(os.environ.get('META_LEDGER_DIR') or 'data/ledger')
LEDGER_TREND_DAYS = 300

# --- Long-window slices: Top Cut and Late Rounds tabs ------------------------
# Top-cut and late-round games are ~5% and ~8% of all games, so a 30-day window
# leaves most decks with too few to say anything. They share the trend's longer
# window, over the same deck list as the 30-day CSVs.
import swiss_sim as _swiss_sim
LONG_DAYS = TREND_DAYS
# Top cut is the single-elimination phase, and its phase number depends on the
# source (probed 2026-09-12, 90 days): online events run Swiss then cut, so the
# cut is phase 2; majors run a second Swiss day first -- phase 2 there is 2,586
# Day-2 Swiss pairings, the cut is phase 3 (60 pairings).
CUT_PHASE = {'play': 2, 'labs': 3}
# Late rounds: round >= LATE_ROUND, only in events big enough to reach it, so
# the event mix is the same on both sides of the comparison. Round numbers
# continue into the cut (98% of cut pairings in 129+ player events are numbered
# 8 or later), so the cut is subtracted back out.
LATE_ROUND = 8
LATE_MIN_PLAYERS = _swiss_sim.min_players_for_rounds(LATE_ROUND)
LONG_FILTERS = {
    'LAll':     {},
    'LCut':     {'phase': CUT_PHASE},
    'LBig':     {'min_players': LATE_MIN_PLAYERS},
    'LBigCut':  {'min_players': LATE_MIN_PLAYERS, 'phase': CUT_PHASE},
    'LLate':    {'min_players': LATE_MIN_PLAYERS, 'round_number': LATE_ROUND},
    'LLateCut': {'min_players': LATE_MIN_PLAYERS, 'round_number': LATE_ROUND, 'phase': CUT_PHASE},
}

# --- Conversion tab: final standings ------------------------------------------
# Trainer Hill has no per-player placings, so the Conversion tab reads each event's
# final standings from Limitless (scripts/standings.py), for exactly the events
# Trainer Hill lists in this build's window. The per-event cache (data/standings/)
# is shared by every build, because a finished event never changes; each build
# keeps a one-row-per-player extract. Its freshness is tracked separately, so adding
# it to an already-fetched day does not re-fetch the matchup slices.
import th_api as _th_api_conv
STANDINGS_PATH = DATA_DIR / 'standings.csv'
STANDINGS_MANIFEST = DATA_DIR / 'standings-manifest.json'
STANDINGS_WINDOW = _th_api_conv.window(FETCH_DAYS, FETCH_END_DATE)

def _standings_fresh():
    try:
        m = json.loads(STANDINGS_MANIFEST.read_text(encoding='utf-8'))
    except (OSError, ValueError):
        return False
    return STANDINGS_PATH.exists() and m.get('window') == STANDINGS_WINDOW

NEEDS_STANDINGS = not _standings_fresh()

def _csv_path(tier, platform='all'):
    return DATA_DIR / f'trainerhill-meta-matchups {tier} {platform}.csv'

def _is_fresh(p):
    if not p.exists():
        return False
    return date.fromtimestamp(p.stat().st_mtime) == date.today()

def _manifest_end():
    try:
        return json.loads(FETCH_MANIFEST.read_text(encoding='utf-8'))['window']['end_date']
    except (OSError, ValueError, KeyError):
        return None

# The trend and share files count toward freshness too, otherwise adding them
# to an already-fetched day would leave those inputs permanently missing. The All
# and tier CSVs do not: they are built from standings (NEEDS_MATCHES, below).
_input_files = ([_csv_path(t, p) for t in LONG_FILTERS for p in PLATFORMS]
                + [DATA_DIR / f'meta-trend {p}.json' for p in PLATFORMS]
                + [DATA_DIR / f'archetype-share {p}.json' for p in PLATFORMS])
if FETCH_END_DATE:
    # A pinned window is a fixed snapshot: it does not go stale with the calendar,
    # but the files on disk must be for THAT window.
    NEEDS_FETCH = (not all(p.exists() for p in _input_files)
                   or _manifest_end() != FETCH_END_DATE)
else:
    NEEDS_FETCH = (not all(_is_fresh(p) for p in _input_files)
                   or _manifest_end() != date.today().isoformat())

# --- All and finishing-tier matchup slices: standings + pairings --------------
# Rebuilt whenever what they rest on changes: a new Trainer Hill fetch (it picks the
# decks), new standings, a different tier rule or deck list, or a missing file.
import th_csv as _th_csv_m
import tiers as _tiers_m
MATCHES_MANIFEST = DATA_DIR / 'matches-manifest.json'

def _matches_fresh():
    m = _tiers_m.read_manifest(MATCHES_MANIFEST)
    decks = {p: _th_csv_m.read_decks(DATA_DIR, p, TOP_N) for p in PLATFORMS}
    return (m.get('window') == STANDINGS_WINDOW and m.get('rule') == _tiers_m.RULE
            and m.get('tiers') == ALL_FILTERS and m.get('decks') == _tiers_m.decks_digest(decks)
            and all(_csv_path(t, p).exists() for t in ALL_FILTERS for p in PLATFORMS))

NEEDS_MATCHES = NEEDS_FETCH or NEEDS_STANDINGS or not _matches_fresh()

if NEEDS_FETCH and FETCH_BACKEND == 'scrape':
    # Only the scraper needs a browser; the API backend is plain HTTP.
    try:
        import playwright  # noqa: F401
    except ImportError:
        print('Installing playwright...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'playwright'])
    subprocess.run([sys.executable, '-m', 'playwright', 'install', 'chromium'],
                   check=False, capture_output=True)

if NEEDS_FETCH:
    _n_slices = len(LONG_FILTERS)
    print(f'Will fetch fresh data via {FETCH_BACKEND!r} '
          f'({_n_slices * len(PLATFORMS)} files: {_n_slices} slices x {len(PLATFORMS)} platforms)')
else:
    print(f'Using cached CSVs for the window ending {_manifest_end()} ({DATA_DIR})')
print(f'Final standings: {"will fetch" if NEEDS_STANDINGS else "cached"} for '
      f'{STANDINGS_WINDOW["start_date"]} .. {STANDINGS_WINDOW["end_date"]}')
print(f'All + finishing-tier matchup slices: {"will build" if NEEDS_MATCHES else "cached"} '
      f'from standings + pairings')


In [ ]:
# === Trainer Hill auto-fetch: download CSVs ===
# Backend 'api' (default) is a handful of HTTP calls; everything below it is the
# 'scrape' backend, kept as a cross-check. Both write the same nine CSVs and
# both end at the same guard: no two tiers of a platform may be identical.

if NEEDS_FETCH and FETCH_BACKEND == 'api':
    import json as _json

    import th_api
    import th_csv

    _th = th_api.TrainerHill(cache_dir=API_CACHE)
    _window = th_api.window(FETCH_DAYS, FETCH_END_DATE)
    print(f'window {_window["start_date"]} .. {_window["end_date"]}, '
          f'min {MIN_PLAYERS} players, top {TOP_N} archetypes by share\n')

    # Chooses each platform's decks and caches its share of players (Mispriced Decks
    # reads it). No matchup slice here: All and the finishing tiers are built from
    # standings + pairings at the end of this cell.
    _written, _decks_by_platform = th_csv.write_csvs(_th, DATA_DIR, [], PLATFORMS,
                                                     top_n=TOP_N, window=_window,
                                                     min_players=MIN_PLAYERS)

    print(f'\ndaily archetype trend ({TREND_DAYS}d window, for share velocity):')
    _trend_window = th_api.window(TREND_DAYS, FETCH_END_DATE)
    _trend_days = th_csv.write_trend(_th, DATA_DIR, PLATFORMS, _trend_window,
                                     min_players=MIN_PLAYERS)

    # Same decks, longer window: top-cut and late-round games are too few in 30 days.
    # The Top Cut and Late Rounds tabs subtract these slices from one another, but each
    # is its own query and Trainer Hill's data can move between two of them: on
    # 2026-09-26 an online event dropped out mid-fetch, leaving cut slices holding games
    # their totals lacked. So each platform is checked with the same derivation the tabs
    # use, and one that does not nest is fetched again, bypassing the cache.
    import time as _time
    import meta_shift as _mshift
    print(f'\nlong-window slices ({LONG_DAYS}d, for Top Cut and Late Rounds):')
    _long_window = th_api.window(LONG_DAYS, FETCH_END_DATE)
    for _p in PLATFORMS:
        for _attempt in range(3):
            _th.refresh = _attempt > 0   # a retry must not be handed the cached copy
            _long_written, _ = th_csv.write_csvs(_th, DATA_DIR, list(LONG_FILTERS), [_p],
                                                 top_n=TOP_N, window=_long_window,
                                                 min_players=MIN_PLAYERS, tier_filters=LONG_FILTERS,
                                                 decks_by_platform=_decks_by_platform)
            try:
                _mshift.pressure_slices({t: pd.read_csv(_csv_path(t, _p)) for t in LONG_FILTERS})
                break
            except ValueError as _exc:
                print(f'  {_p}: slices do not nest -- {_exc}')
                if _attempt < 2:
                    print(f'  {_p}: re-fetching in 30s')
                    _time.sleep(30)
        else:
            # The next run must fetch again rather than trust today's files.
            FETCH_MANIFEST.unlink(missing_ok=True)
            raise RuntimeError(f'{_p}: long-window slices still do not nest after 3 fetches; '
                               f"Trainer Hill's data is still changing. Re-run in a few minutes.")
        _written.update(_long_written)
    _th.refresh = False

    # The check that would have caught the Upper50 duplication at fetch time.
    # Long slices are checked filter by filter -- each against the slice it
    # narrows -- because some other pairs are legitimately equal: every in-person
    # major reaches round 8, so there LAll == LBig and LBigCut == LLateCut.
    # (All and the finishing tiers get the same guard where they are built.)
    for _pair in (['LAll', 'LCut'], ['LBig', 'LBigCut'], ['LBig', 'LLate'], ['LLate', 'LLateCut']):
        th_csv.assert_tiers_differ(DATA_DIR, _pair, PLATFORMS)
    print('\nguard: no two slices of a platform are identical -- OK')

    # Provenance, so a dashboard built from these CSVs can say what it covers.
    FETCH_MANIFEST.write_text(_json.dumps({
        'backend': 'api', 'fetched_on': date.today().isoformat(),
        'window': _window, 'min_players': MIN_PLAYERS, 'top_n': TOP_N,
        'platform_sources': th_csv.PLATFORM_SOURCES,
        'matchup_slices': 'All and finishing tiers from standings + pairings: matches-manifest.json',
        'rows': {f'{t}|{p}': n for (t, p), n in _written.items()},
        'trend_window': _trend_window, 'trend_days': _trend_days,
        'long_window': _long_window, 'long_filters': LONG_FILTERS,
    }, indent=1), encoding='utf-8')
    print(f'wrote {FETCH_MANIFEST.name} '
          f'({_th.calls} network calls, {_th.cache_hits} cache hits)')

    # Prediction ledger, live builds only: refresh the long trend (it locates set
    # releases) and measure what happened after any snapshot whose next window has
    # now finished. A historical build has nothing to add -- it is itself the
    # "what happened next" for the build before it.
    if not FETCH_END_DATE:
        import ledger as _ledger
        _th_ledger = th_api.TrainerHill(cache_dir=LEDGER_DIR / '_api_cache')
        _ledger_trend_w = th_api.window(LEDGER_TREND_DAYS)
        LEDGER_DIR.mkdir(parents=True, exist_ok=True)
        (LEDGER_DIR / 'meta-trend.json').write_text(_json.dumps({
            'window': _ledger_trend_w,
            'daily_trend': _th_ledger.query('meta_trend', min_players=MIN_PLAYERS,
                                            **_ledger_trend_w)['daily_trend'],
        }), encoding='utf-8')
        _measured = [s['end'] for s in _ledger.load_snapshots(LEDGER_DIR)
                     if not (LEDGER_DIR / 'outcomes' / f"{s['end']}.json").exists()
                     and _ledger.build_outcome(s, LEDGER_DIR, th=_th_ledger) is not None]
        print(f'ledger: trend refreshed; outcomes measured for {_measured or "no new snapshots"} '
              f'({_th_ledger.calls} network calls)')


# --- Conversion tab: final standings for the same window -----------------------
# Independent of the backend above: the event list is Trainer Hill's, the standings
# are Limitless's, and only events not already in data/standings/ cost a request.
# Limitless allows 50 requests per 5 minutes, which the fetcher waits out itself.
if NEEDS_STANDINGS:
    import json as _json_st

    import standings as _sd

    _th_st = _th_api_conv.TrainerHill(cache_dir=API_CACHE)
    _st_events = _sd.events_in_window(_th_st, STANDINGS_WINDOW, min_players=MIN_PLAYERS)
    print(f'\nfinal standings: {len(_st_events)} events, '
          f'{STANDINGS_WINDOW["start_date"]} .. {STANDINGS_WINDOW["end_date"]}')
    _st = _sd.Standings()
    _st_players = _st.load(_st_events, every=50)
    _st_cov = _sd.coverage(_st_players, _st_events)
    # An event still running, or published late, comes back short: used, never cached,
    # and re-requested by the next build.
    _st_short = _st_cov[_st_cov['row_ratio'] < _sd.MIN_COVERAGE]
    _st_players.to_csv(STANDINGS_PATH, index=False)
    STANDINGS_MANIFEST.write_text(_json_st.dumps({
        'window': STANDINGS_WINDOW, 'fetched_on': date.today().isoformat(),
        'events': len(_st_events), 'players': len(_st_players),
        'short_events': list(_st_short.index), 'network_calls': _st.calls,
    }, indent=1), encoding='utf-8')
    print(f'  {len(_st_players):,} players ({_st.calls} fetched, {_st.cache_hits} cached); '
          f'{len(_st_short)} event(s) not fully published yet')


# --- scrape backend -------------------------------------------------------
# Drives /meta with Playwright. The platform filter is set via the URL query
# parameter (?platform=all|online|inperson) which requires a page reload, so
# each platform iteration starts from a clean DOM. Per platform:
#   1. Goto URL with the platform query param.
#   2. Top up Archetype Selection to TOP_N (usage-ordered).
#   3. For each Placement value (All / Upper 50% / Upper 25%): set the
#      dropdown, click "Export data (csv)", save the download.
# Result: 9 files in data/.

if NEEDS_FETCH and FETCH_BACKEND == 'scrape':
    import re
    from playwright.async_api import async_playwright

    BASE_URL = 'https://www.trainerhill.com/meta'
    EXPECTED_HEADER = 'deck1,deck2,wins,losses,ties,total,win_rate'
    PLACEMENT_LABELS = {'All': 'All', 'Upper50': 'Upper 50%', 'Upper25': 'Upper 25%'}
    SKIP_DECKS = {'Other'}
    api_calls = set()
    export_failures = []  # (platform, tier) slices the site had nothing to export for

    async def _fetch():
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            ctx = await browser.new_context(
                viewport={'width': 1400, 'height': 900},
                accept_downloads=True,
            )
            page = await ctx.new_page()

            def _log(req):
                u = req.url
                if 'trainerhill' in u and any(s in u for s in ('/api/', '.json', 'matchup', 'usage', 'meta')):
                    api_calls.add(f'{req.method} {u}')
            page.on('request', _log)

            async def find_select(label_text):
                handle = await page.evaluate_handle(
                    "labelText => {"
                    "  const labels = Array.from(document.querySelectorAll('label, .form-label, h5, h6, strong'));"
                    "  const lbl = labels.find(l => (l.innerText || '').trim().toLowerCase() === labelText.toLowerCase());"
                    "  if (!lbl) return null;"
                    "  let n = lbl;"
                    "  for (let i = 0; i < 6 && n; i++) {"
                    "    const ctrl = n.querySelector ? n.querySelector('.Select-control') : null;"
                    "    if (ctrl) return ctrl;"
                    "    n = n.parentElement;"
                    "  }"
                    "  return null;"
                    "}",
                    label_text,
                )
                return handle.as_element()

            async def open_select(ctrl, label_for_diag=''):
                async def _wait():
                    await page.wait_for_selector('.Select-menu-outer', timeout=1500)
                async def _mousedown(el):
                    if not el: return False
                    try:
                        await el.scroll_into_view_if_needed()
                        await el.dispatch_event('mousedown', {'button': 0, 'buttons': 1, 'which': 1})
                        await el.dispatch_event('mouseup', {'button': 0, 'buttons': 0, 'which': 1})
                        await _wait(); return True
                    except Exception: return False
                async def _click(el):
                    if not el: return False
                    try:
                        await el.scroll_into_view_if_needed()
                        await el.click(force=True, timeout=2000)
                        await _wait(); return True
                    except Exception: return False
                for attempt in range(3):
                    arrow = await ctrl.query_selector('.Select-arrow-zone')
                    si = await ctrl.query_selector('.Select-input')
                    inp = await ctrl.query_selector('input')
                    if await _mousedown(arrow): return
                    if await _mousedown(si): return
                    if await _click(arrow): return
                    if await _click(si): return
                    if inp:
                        try:
                            await inp.scroll_into_view_if_needed()
                            await inp.focus()
                            await page.keyboard.press('ArrowDown')
                            await _wait(); return
                        except Exception: pass
                    if arrow:
                        try:
                            await arrow.scroll_into_view_if_needed()
                            box = await arrow.bounding_box()
                            if box:
                                await page.mouse.click(box['x'] + box['width']/2, box['y'] + box['height']/2)
                                await _wait(); return
                        except Exception: pass
                    await page.wait_for_timeout(400 + attempt * 400)
                cls = await ctrl.get_attribute('class')
                outer = await ctrl.evaluate('el => el.outerHTML.slice(0, 800)')
                raise RuntimeError(f"open_select({label_for_diag!r}) failed.\nclass={cls}\nHTML: {outer}")

            async def expand_filters():
                """Open the collapsed half of the Tournament Filters panel.

                Placement lives behind a 'Show more' toggle. A collapsed control
                still opens, but its react-virtualized menu renders with width 0
                and therefore zero option rows, so pick_option matches nothing.
                """
                for _ in range(3):
                    try:
                        btn = page.get_by_text(re.compile(r'^\s*show more\s*$', re.I)).first
                        if not await btn.is_visible():
                            return
                        await btn.click(timeout=1500)
                        await page.wait_for_timeout(400)
                    except Exception:
                        return

            async def topup_decks():
                """Bring Archetype Selection up to TOP_N decks (usage-ordered)."""
                async def chip_count():
                    return await page.evaluate(
                        "() => document.querySelectorAll('.Select-multi-value-wrapper .Select-value-label .ms-1').length"
                    )
                if await chip_count() >= TOP_N:
                    return
                arch_ctrl = await find_select('Archetype Selection')
                if not arch_ctrl:
                    raise RuntimeError("Couldn't locate the 'Archetype Selection' dropdown.")
                try:
                    await open_select(arch_ctrl, 'Archetype Selection')
                except Exception as exc:
                    # A platform can have fewer archetypes than TOP_N; once every
                    # option is selected the menu stops opening at all. Keep what
                    # is already selected rather than failing the whole fetch.
                    print(f'  WARN: archetype list would not open '
                          f'({type(exc).__name__}); continuing with '
                          f'{await chip_count()} decks')
                    return
                attempts = 0
                while await chip_count() < TOP_N and attempts < TOP_N * 4:
                    attempts += 1
                    try:
                        await page.wait_for_selector('.Select-menu-outer', timeout=300)
                    except Exception:
                        arch_ctrl = await find_select('Archetype Selection')
                        try:
                            await open_select(arch_ctrl, 'Archetype Selection')
                        except Exception:
                            break  # nothing left to add
                        await page.wait_for_timeout(250)
                    grid = await page.query_selector('.Select-menu-outer .ReactVirtualized__Grid')
                    if grid:
                        await grid.evaluate('el => el.scrollTo(0, 0)')
                        await page.wait_for_timeout(80)
                    pick_text = await page.evaluate(
                        "skip => {"
                        "  const skipSet = new Set(skip);"
                        "  const opts = Array.from(document.querySelectorAll('.Select-menu-outer .VirtualizedSelectOption'));"
                        "  for (const o of opts) {"
                        "    const t = (o.innerText || '').trim();"
                        "    if (t && !skipSet.has(t)) return t;"
                        "  }"
                        "  return null;"
                        "}",
                        list(SKIP_DECKS),
                    )
                    if not pick_text:
                        break
                    opt = await page.query_selector(
                        f'.Select-menu-outer .VirtualizedSelectOption:has-text("{pick_text}")'
                    )
                    if not opt:
                        continue
                    await opt.click(force=True)
                    await page.wait_for_timeout(400)
                await page.keyboard.press('Escape')
                await page.wait_for_timeout(500)
                final = await chip_count()
                if final < TOP_N:
                    print(f'  WARN: only {final}/{TOP_N} decks selected')

            async def pick_option(option_text):
                # The menu is react-virtualized: it renders no rows at all until
                # it has been given a width, so "zero options" is a real state
                # rather than a slow one. Wait for rows before concluding the
                # option is missing, and say which failure it was.
                for _ in range(20):
                    n_opts = await page.evaluate(
                        "() => document.querySelectorAll("
                        "'.Select-menu-outer .VirtualizedSelectOption').length")
                    if n_opts:
                        break
                    await page.wait_for_timeout(150)
                else:
                    raise RuntimeError(
                        f'menu rendered 0 options while selecting {option_text!r} '
                        f'(virtualized grid has no width -- filter panel collapsed?)')
                opt = await page.query_selector(
                    f'.Select-menu-outer .VirtualizedSelectOption:has-text("{option_text}")'
                )
                if not opt:
                    grid = await page.query_selector('.Select-menu-outer .ReactVirtualized__Grid')
                    for _ in range(50):
                        if not grid: break
                        await grid.evaluate('el => el.scrollBy(0, 200)')
                        await page.wait_for_timeout(50)
                        opt = await page.query_selector(
                            f'.Select-menu-outer .VirtualizedSelectOption:has-text("{option_text}")'
                        )
                        if opt: break
                if not opt:
                    raise RuntimeError(f'Option {option_text!r} not found')
                await opt.click(force=True)
                await page.wait_for_timeout(300)

            # ------------------------------------------------------------
            # Outer loop: each platform gets a fresh page load via the
            # ?platform=X query param. Top up decks then iterate placements.
            # ------------------------------------------------------------
            for platform in PLATFORMS:
                url = f'{BASE_URL}?platform={platform}'
                print(f'\n=== Platform: {platform!r}  ({url}) ===')
                try:
                    await page.goto(url, wait_until='networkidle', timeout=60_000)
                    await page.wait_for_timeout(2500)
                    await expand_filters()
                    await topup_decks()
                except Exception as exc:
                    print(f'  WARN: {platform} page setup failed '
                          f'({type(exc).__name__}: {exc}); treating its tiers as no entries')
                    export_failures.extend((platform, t) for t in FILTERS)
                    continue

                for tier in FILTERS:
                    target = _csv_path(tier, platform)
                    try:
                        placement_ctrl = await find_select('Placement')
                        if not placement_ctrl:
                            raise RuntimeError("Couldn't locate the 'Placement' dropdown.")
                        await open_select(placement_ctrl, 'Placement')
                        await pick_option(PLACEMENT_LABELS[tier])
                        # pick_option clicks with force=True, which skips
                        # actionability checks -- a click on a detached or
                        # zero-size option is a silent no-op, leaves the previous
                        # tier selected, and the export then simply repeats it.
                        # That is how every Upper50 CSV became a copy of All, so
                        # never trust the click; read the control back.
                        _ctrl = await find_select('Placement')
                        _shown = (await _ctrl.inner_text()).strip() if _ctrl else ''
                        if PLACEMENT_LABELS[tier].lower() not in _shown.lower():
                            raise RuntimeError(
                                f'Placement reads {_shown!r} after selecting '
                                f'{PLACEMENT_LABELS[tier]!r} -- selection did not apply')
                        await page.wait_for_load_state('networkidle')
                        await page.wait_for_timeout(1000)

                        async with page.expect_download(timeout=30_000) as dl_info:
                            await page.get_by_role('button', name=re.compile(r'export', re.I)).first.click()
                        dl = await dl_info.value
                        await dl.save_as(str(target))
                        print(f'  saved {target.name}  (site suggested: {dl.suggested_filename})')
                    except Exception as exc:
                        # A platform slice can legitimately have no entries (e.g. no
                        # in-person events yet for the format), and the site then has
                        # nothing to export. Record it; the empty CSV is written after
                        # the crawl so a site-wide breakage can't wipe cached data.
                        export_failures.append((platform, tier))
                        print(f'  WARN: no export for {tier}/{platform} '
                              f'({type(exc).__name__}: {exc}); treating as no entries')

            await browser.close()

    await _fetch()

    # Failing *every* slice means the page layout changed, not that the meta is
    # empty -- raise and leave the previously fetched CSVs untouched.
    if len(export_failures) == len(PLATFORMS) * len(FILTERS):
        raise RuntimeError(
            'No slice exported successfully; the Trainer Hill layout has probably '
            'changed. Existing CSVs left untouched.'
        )
    for _platform, _tier in export_failures:
        _csv_path(_tier, _platform).write_text(EXPECTED_HEADER + '\n', encoding='utf-8')
        print(f'  wrote empty {_csv_path(_tier, _platform).name} (no entries for '
              f'{_tier}/{_platform})')

    # --- Validate exports ---
    # An empty (header-only) export is a valid outcome for a platform slice with
    # no entries, so only a header that is present *and* wrong is an error.
    print('\n=== Validation ===')
    for platform in PLATFORMS:
        for tier in FILTERS:
            path = _csv_path(tier, platform)
            if not path.exists():
                print(f'  {path.name}: [MISSING] -- loader will treat it as no data')
                continue
            with open(path, encoding='utf-8') as fh:
                header = fh.readline().strip()
                row_count = sum(1 for line in fh if line.strip())
            if header and header != EXPECTED_HEADER:
                raise ValueError(f'{path}: header mismatch')
            tag = 'NO DATA' if row_count == 0 else ('OK' if row_count >= 20 else 'LOW DATA')
            print(f'  {path.name}: {row_count} rows  [{tag}]')

    import th_csv as _th_csv
    _th_csv.assert_tiers_differ(DATA_DIR, FILTERS, PLATFORMS)
    print('  guard: no two tiers of a platform are identical -- OK')

    if api_calls:
        print('\n--- XHR endpoints observed (for a future requests-only refactor) ---')
        for c in sorted(api_calls):
            print(' ', c)


# --- All and finishing-tier matchup slices: standings + pairings ------------------
# Trainer Hill's `placement` filter does not mean its name: at majors it is the top N
# PLACES of each division (Upper 25% = 3% of a Worlds field), online it runs 13-28%
# short (tasks/todo.md section 10). So every slice the tier tabs compare is built here,
# from the events in the standings extract and their pairings (scripts/tiers.py): a
# game counts in Upper p% when BOTH players finished in the top p% of their event,
# ranked as on the Conversion tab. All comes from the same pairings, because the tiers
# are differenced against it and subtracted slices must come from one fetch; over the
# same events it is byte-identical to Trainer Hill's All. Pairings cost one call per
# new online event (plus a standings re-fetch to map names to rows) and one per round
# at a major, cached forever; whatever runs after the scraper, these are the slices.
if NEEDS_MATCHES:
    import json as _json_m
    import standings as _sd_m

    _m_players = pd.read_csv(STANDINGS_PATH, dtype={'event_id': str, 'source': str, 'deck': str})
    _m_events = _m_players.drop_duplicates('event_id')[['event_id', 'source']]
    _m_st = _sd_m.Standings()
    _m_pairs, _m_missing = _m_st.load_pairings(_m_events, every=50)
    # An event without pairings (still running when its standings were read, or corrected
    # since) sits out every per-match slice, All included, so the slices stay one set.
    _m_players = _m_players[~_m_players['event_id'].isin(_m_missing)].reset_index(drop=True)
    _m_decks = {p: _th_csv_m.read_decks(DATA_DIR, p, TOP_N) for p in PLATFORMS}
    _m_tiers = {t: _tiers_m.TIERS[t] for t in ALL_FILTERS}
    print(f'\nAll + finishing tiers from standings + pairings: {len(_m_events) - len(_m_missing)} of '
          f'{len(_m_events)} events ({len(_m_pairs):,} pairings)')
    _m_written = _tiers_m.write_csvs(DATA_DIR, _m_players, _m_pairs, PLATFORMS, _m_decks, tiers=_m_tiers)

    # Guards: each tier inside the one before it (the skill bands and the weighted tier
    # difference them), and no two slices of a platform identical.
    _m_order = sorted(_m_tiers, key=lambda t: -(_m_tiers[t] or 100))
    _tiers_m.assert_nested(DATA_DIR, _m_order, PLATFORMS)
    _th_csv_m.assert_tiers_differ(DATA_DIR, ALL_FILTERS, PLATFORMS)
    _m_depth = {}
    for _p in PLATFORMS:
        _q = _m_players[_m_players['source'].isin(_th_csv_m.PLATFORM_SOURCES[_p])]
        _d = {t: _tiers_m.depth(_q, pct) for t, pct in _m_tiers.items()} if len(_q) else {}
        _m_depth[_p] = {t: round(v, 4) for t, v in _d.items() if np.isfinite(v)}
    MATCHES_MANIFEST.write_text(_json_m.dumps({
        'window': STANDINGS_WINDOW, 'built_on': date.today().isoformat(), 'rule': _tiers_m.RULE,
        'tiers': ALL_FILTERS, 'placement': _m_tiers, 'decks': _tiers_m.decks_digest(_m_decks),
        'events': len(_m_events), 'events_without_pairings': _m_missing,
        'standings_changed_since_cached': _m_st.not_joined, 'pairings': len(_m_pairs),
        'network_calls': _m_st.calls, 'share_of_players': _m_depth,
        'rows': {f'{t}|{p}': n for (t, p), n in _m_written.items()},
    }, indent=1), encoding='utf-8')
    print(f'  guards: nested and distinct -- OK ({_m_st.calls} network calls). Share of players in '
          + '/'.join(_m_order) + ': '
          + '; '.join(f'{p} ' + '/'.join(f'{d[t]:.1%}' for t in _m_order) for p, d in _m_depth.items() if d))


In [ ]:
import os

data_dir = 'data'
MATCHUP_COLUMNS = ['deck1', 'deck2', 'wins', 'losses', 'ties', 'total', 'win_rate']


# Every platform is analysed twice: as itself (sample-adjusted win rates, the
# default) and as a raw VIEW, the same games with matchup shrinkage switched off,
# which the dashboard's "Raw win rates" checkbox shows. A view key is the platform
# plus RAW_SUFFIX; everything downstream that is keyed by platform is keyed by view.
RAW_SUFFIX = '-raw'


def raw_view(platform):
    return platform + RAW_SUFFIX


def base_platform(view):
    return view[:-len(RAW_SUFFIX)] if view.endswith(RAW_SUFFIX) else view


def is_adjusted(view):
    return not view.endswith(RAW_SUFFIX)


def _eb_shrinkage(rates, counts, adjust=True):
    """How hard should each matchup be pulled toward the format's neutral rate?

    Small samples make matchups look more one-sided than they are -- a 6-game
    5-1 is not an 83% matchup. Rather than a fixed prior, split the spread of
    matchups around the neutral into real signal and sampling noise (method of
    moments) and express the answer as K "prior games":

        shrunk = (n * observed + K * neutral) / (n + K)

    so a matchup needs roughly K games before its own record outweighs the
    format baseline, and a matchup with none sits exactly at that baseline.

    The target is deliberately the **neutral** -- the same line that separates a
    winning matchup from a losing one. A blend of the observation with that line
    can never cross it, so shrinkage can soften how lopsided a matchup looks but
    can never turn a losing matchup into a winning one. (Shrinking toward each
    deck's own rate did exactly that: it dragged a strong deck's bad matchups
    above 50% and a weak deck's good ones below.) Returns (neutral, K).

    `adjust=False` is the raw view: the neutral is still estimated -- it is the
    break-even line, not an adjustment -- but K is 0, so nothing is pulled.
    """
    n = np.asarray(counts, dtype=float)
    p = np.asarray(rates, dtype=float)
    ok = np.isfinite(n) & np.isfinite(p) & (n > 0)
    n, p = n[ok], p[ok]
    if len(n) < 5:
        # Not enough matchups to estimate from; fall back to a weak explicit prior.
        return (float(np.average(p, weights=n)) if len(n) else 0.5), (float(MIN_SAMPLE) if adjust else 0.0)
    neutral = float(np.average(p, weights=n))
    if not adjust:
        return neutral, 0.0
    obs_var = float(np.average((p - neutral) ** 2, weights=n))
    samp_var = float(np.average(neutral * (1 - neutral) / n, weights=n))
    true_var = obs_var - samp_var
    if true_var <= 1e-9:
        # Nothing beyond noise: treat every matchup as neutral until proven otherwise.
        return neutral, float(n.sum())
    return neutral, max(1.0, neutral * (1 - neutral) / true_var)


def _shrink_column(df, src, out, neutral, k):
    """(n*observed + K*neutral) / (n + K); no-game rows land on the neutral."""
    obs = df[src].fillna(neutral)
    n = df['actual_total']
    # With K = 0 (raw view) a no-game row would be 0/0; it stays on the neutral.
    df[out] = ((n * obs + k * neutral) / (n + k)).mask(n + k == 0, neutral)
    return df


def _mirror_baselines(df, fallback_pct):
    """Per-deck break-even: what an even matchup is actually worth to *this* deck.

    A mirror is even by construction -- the export's wins always equal its losses
    -- so the only thing it measures is how often the deck draws, and draws pull a
    3-pt rate below 50%: an even matchup with tie rate t is worth 50 - t/6. Decks
    differ here for real (Slowking ties 1% of its mirrors, Dragapult ex 10%), which
    is why the favorable/unfavorable split is drawn per deck rather than on one
    format-wide line -- a deck that draws a lot scores below 50% in every matchup
    it plays, including the even ones.

    Two corrections on the raw mirror row:

      games     a mirror match is recorded once, counted as both a win and a loss
                but as only one tie, so `wins + losses + ties` overstates the games
                and halves the tie share. `total` is the match count, and
                `ties / total` the real tie rate.
      shrinkage a tie rate is a plain binomial proportion, so the same
                method-of-moments prior used for matchups applies unchanged.
                Without it, a deck whose one recorded mirror happened to be a draw
                reads as a 33% break-even -- and then every matchup it has ever
                played scores as favorable against it.

    Returns {deck: baseline %}, {deck: mirror matches}, and the prior baseline that
    decks with no mirror of their own fall back to.
    """
    m = df[df['is_mirror'] & ~df['is_other'] & (df['total'] > 0)]
    if len(m) == 0:
        return {}, {}, float(fallback_pct)
    games = m['total'].astype(float)
    tie_rate = (m['ties'] / games).astype(float)
    prior_tie, k = _eb_shrinkage(tie_rate, games)
    shrunk = (games * tie_rate + k * prior_tie) / (games + k)
    return ({d: float((0.5 - s / 6) * 100) for d, s in zip(m['deck1'], shrunk)},
            {d: int(n) for d, n in zip(m['deck1'], games)},
            float((0.5 - prior_tie / 6) * 100))


def _read_matchup_csv(path, adjust=True):
    """Read one matchup export into a typed frame.

    A platform/tier slice can legitimately have no entries (e.g. no in-person
    events yet for the format), which the site exports as a header-only CSV --
    and a slice that could not be exported at all leaves no file behind. Both
    cases return an empty frame that still carries the full schema, so every
    later `df['col']` lookup keeps working instead of raising KeyError.

    `adjust=False` builds the raw view: the smoothed_* columns equal the raw rates."""
    if path.exists():
        df = pd.read_csv(path)
    else:
        print(f'  NOTE: {path.name} not found; treating that slice as no data.')
        df = pd.DataFrame(columns=MATCHUP_COLUMNS)
    for col in MATCHUP_COLUMNS:
        if col not in df.columns:
            df[col] = pd.Series(dtype='object' if col in ('deck1', 'deck2') else 'float64')
    num_cols = ['wins', 'losses', 'ties', 'total', 'win_rate']
    df[num_cols] = df[num_cols].apply(pd.to_numeric, errors='coerce')
    df = df.dropna(subset=['deck1', 'deck2']).reset_index(drop=True)
    df['actual_total'] = df['wins'] + df['losses'] + df['ties']
    df['is_mirror'] = df['deck1'] == df['deck2']
    df['is_other'] = (df['deck1'] == 'other') | (df['deck2'] == 'other')
    df['win_prob'] = df['wins'] / df['actual_total']
    df['tie_prob'] = df['ties'] / df['actual_total']
    df['loss_prob'] = df['losses'] / df['actual_total']
    # Tie-aware match-point win rate (W=3, T=1, L=0) -- the same thing the site's
    # own win_rate column reports.
    df['wr3'] = (3 * df['wins'] + df['ties']) / (3 * df['actual_total'])
    # Shrink every rate by its sample size, with the strength estimated from this
    # slice's own matchups. Mirrors and 'other' are left out of the estimate (they
    # are not real matchups) but still get shrunk values.
    est = df[~df['is_mirror'] & ~df['is_other'] & (df['actual_total'] > 0)]
    shrink_info = {}
    for src, out in (('wr3', 'smoothed_wr3'),
                     ('win_prob', 'smoothed_wr'),
                     ('tie_prob', 'smoothed_tie_prob')):
        neutral, k = _eb_shrinkage(est[src], est['actual_total'], adjust)
        _shrink_column(df, src, out, neutral, k)
        # `mean` is the format's tie-aware neutral -- both the shrink target and
        # the break-even line used elsewhere; `k` is how many games a matchup
        # needs before its own record outweighs that baseline.
        shrink_info[src] = {'mean': neutral, 'k': k}
    return df, shrink_info


# Build raw_by_platform[platform][tier] from the static-named CSVs.
# Filename pattern: trainerhill-meta-matchups {tier} {platform}.csv
raw_by_platform = {}
clean_by_platform = {}
matrices_by_platform = {}
popularity_by_platform = {}
popularity_weights_by_platform = {}
shrinkage_by_platform = {}
mirror_by_platform = {}

for platform in PLATFORMS:
    raw_p, shrink_p, mirror_p = {}, {}, {}
    for tier in FILTERS:
        raw_p[tier], shrink_p[tier] = _read_matchup_csv(_csv_path(tier, platform))
        base, mgames, prior = _mirror_baselines(raw_p[tier], shrink_p[tier]['wr3']['mean'] * 100)
        mirror_p[tier] = {'baseline': base, 'games': mgames, 'prior': prior}
    raw_by_platform[platform] = raw_p
    shrinkage_by_platform[platform] = shrink_p
    mirror_by_platform[platform] = mirror_p

# A platform with no rows in any tier is inactive: it is skipped by the analysis
# pipeline and shown as unavailable in the dashboard, rather than producing
# charts built from nothing.
ACTIVE_PLATFORMS = [p for p in PLATFORMS
                    if any(len(raw_by_platform[p][t]) > 0 for t in FILTERS)]
EMPTY_PLATFORMS = [p for p in PLATFORMS if p not in ACTIVE_PLATFORMS]
if not ACTIVE_PLATFORMS:
    raise RuntimeError('Every platform/tier CSV is empty; re-run the fetch cell.')
DEFAULT_PLATFORM = 'all' if 'all' in ACTIVE_PLATFORMS else ACTIVE_PLATFORMS[0]
if PLATFORM not in ACTIVE_PLATFORMS:
    print(f'PLATFORM {PLATFORM!r} has no entries; switching to {DEFAULT_PLATFORM!r}.')
    PLATFORM = DEFAULT_PLATFORM

# The raw view of each active platform: the same CSVs, read with shrinkage off.
# The per-deck break-even lines are shared, not re-derived: they keep their
# sample-size adjustment in both views, so a deck whose one recorded mirror was a
# draw still does not read as favourable in every matchup. Only the matchup
# rates measured against those lines go raw.
VIEWS = []
for platform in ACTIVE_PLATFORMS:
    _view = raw_view(platform)
    raw_by_platform[_view], shrinkage_by_platform[_view] = {}, {}
    for tier in FILTERS:
        raw_by_platform[_view][tier], shrinkage_by_platform[_view][tier] = \
            _read_matchup_csv(_csv_path(tier, platform), adjust=False)
    mirror_by_platform[_view] = mirror_by_platform[platform]
    VIEWS += [platform, _view]

# Union of decks across ALL platforms / tiers (so every chart can list every
# deck even if data is sparse on a given slice).
all_decks_set = set()
for platform in PLATFORMS:
    for tier in FILTERS:
        all_decks_set |= set(raw_by_platform[platform][tier]['deck1'].unique())
        all_decks_set |= set(raw_by_platform[platform][tier]['deck2'].unique())
all_decks = sorted(all_decks_set)
analysis_decks = [d for d in all_decks if d != 'other']

DECK_SHORT_NAMES = unique_short_names(analysis_decks)
DECK_COLORS = {d: DECK_PALETTE[i % len(DECK_PALETTE)] for i, d in enumerate(analysis_decks)}

# CORE_DECKS / SLEEPER_DECKS are split by use rate at the end of the archetype
# cell, which is where the Limitless meta table (the use-rate source) is loaded.

def _build_matrix(tier_df, value_col, decks):
    matrix = pd.DataFrame(np.nan, index=decks, columns=decks)
    for _, row in tier_df.iterrows():
        if row['deck1'] in decks and row['deck2'] in decks:
            matrix.loc[row['deck1'], row['deck2']] = row[value_col]
    return matrix

for platform in raw_by_platform:      # every platform, plus the raw views
    clean_p = {}
    matrices_p = {}
    for tier, df in raw_by_platform[platform].items():
        cdf = df[~df['is_mirror'] & ~df['is_other']].copy()
        clean_p[tier] = cdf
        matrices_p[tier] = {
            'win_prob': _build_matrix(cdf, 'smoothed_wr', analysis_decks),
            'tie_prob': _build_matrix(cdf, 'smoothed_tie_prob', analysis_decks),
            'smoothed_wr': _build_matrix(cdf, 'smoothed_wr', analysis_decks),
            'smoothed_wr3': _build_matrix(cdf, 'smoothed_wr3', analysis_decks),
            'raw_wr3': _build_matrix(cdf, 'wr3', analysis_decks),
            'sample_n': _build_matrix(cdf, 'actual_total', analysis_decks),
            'raw_wr': _build_matrix(cdf, 'win_rate', analysis_decks),
        }
    clean_by_platform[platform] = clean_p
    matrices_by_platform[platform] = matrices_p
    pop_p = clean_p['All'].groupby('deck1')['actual_total'].sum().reindex(analysis_decks, fill_value=0)
    popularity_by_platform[platform] = pop_p
    s = pop_p.sum()
    popularity_weights_by_platform[platform] = (pop_p / s).values if s > 0 else np.zeros(len(analysis_decks))

# Daily archetype trend per platform, for the share-velocity inputs to Meta
# Roles. Absent or too-short is a normal outcome (in-person majors are a few
# discrete weekends), so this stays an empty list and the roles code falls back.
import sys as _sys
from pathlib import Path as _Path
_sys.path.insert(0, str(_Path('scripts').resolve()))
import th_csv as _th_csv

# Finer placement slices for the Skill Curve tab. Loaded separately from the
# three tiers so nothing else in the notebook can accidentally read them as a
# fourth and fifth tier.
skill_raw_by_platform = {}
for platform in PLATFORMS:
    _sp = {}
    for tier in SKILL_FILTERS:
        _df, _ = _read_matchup_csv(_csv_path(tier, platform))
        _sp[tier] = _df[_df['deck1'].isin(analysis_decks)
                        & _df['deck2'].isin(analysis_decks)].copy()
    skill_raw_by_platform[platform] = _sp
_skill_note = ', '.join(
    f'{p}: ' + '/'.join(str(len(skill_raw_by_platform[p][t])) for t in SKILL_FILTERS)
    for p in PLATFORMS)
print(f'Skill-curve slices ({"/".join(SKILL_FILTERS)} rows) -- {_skill_note}')

# Long-window slices for the Top Cut and Late Rounds tabs: a different window
# from everything above, so kept apart for the same reason.
long_raw_by_platform = {}
for platform in PLATFORMS:
    _lp = {}
    for tier in LONG_FILTERS:
        _df, _ = _read_matchup_csv(_csv_path(tier, platform))
        _lp[tier] = _df[_df['deck1'].isin(analysis_decks)
                        & _df['deck2'].isin(analysis_decks)].copy()
    long_raw_by_platform[platform] = _lp
_long_note = ', '.join(
    f'{p}: ' + '/'.join(f"{int(long_raw_by_platform[p][t]['total'].sum() / 2):,}" for t in LONG_FILTERS)
    for p in PLATFORMS)
print(f'Long-window slices ({"/".join(LONG_FILTERS)} matches) -- {_long_note}')

trend_by_platform = {p: _th_csv.read_trend(DATA_DIR, p) for p in PLATFORMS}
_trend_note = ', '.join(f'{p}: {len(d)}d' for p, d in trend_by_platform.items())

# Counts, not rates: nothing here is shrunk, so a raw view reads the same data.
for platform in ACTIVE_PLATFORMS:
    for _by_platform in (skill_raw_by_platform, long_raw_by_platform, trend_by_platform):
        _by_platform[raw_view(platform)] = _by_platform[platform]

# Backward-compat aliases for the analysis cells: point at PLATFORM slice.
raw = raw_by_platform[PLATFORM]
clean = clean_by_platform[PLATFORM]
matrices = matrices_by_platform[PLATFORM]
popularity = popularity_by_platform[PLATFORM]
popularity_weights = popularity_weights_by_platform[PLATFORM]

print(f"Loaded {len(analysis_decks)} decks across {len(ACTIVE_PLATFORMS)} platforms x {len(FILTERS)} tiers.")
if EMPTY_PLATFORMS:
    print(f"No entries for: {', '.join(EMPTY_PLATFORMS)} (skipped everywhere downstream).")
_shrink_all = shrinkage_by_platform[PLATFORM]['All']['wr3']
print(f"Matchup shrinkage ({PLATFORM}/All): matchups pulled toward the tie-aware "
      f"neutral {_shrink_all['mean'] * 100:.1f}% with a weight of {_shrink_all['k']:.0f} games.")
_mirror_all = mirror_by_platform[PLATFORM]['All']
print(f"Per-deck break-even ({PLATFORM}/All): {len(_mirror_all['baseline'])} decks have a mirror "
      f"record; thin ones are pulled toward {_mirror_all['prior']:.1f}%.")
print(f"Raw win-rate views (no matchup shrinkage; break-even lines shared): "
      f"{', '.join(v for v in VIEWS if not is_adjusted(v))}")
print(f"Active PLATFORM = {PLATFORM!r}")
print(f"Matchup rows (active platform): { {t: len(clean[t]) for t in clean} }")
print(f"Daily trend loaded ({_trend_note}).")


In [ ]:
# === Limitless archetype map: cache deck-variant -> archetype roll-up ===
# Trainerhill provides per-variant matchup data (e.g. dragapult-ex,
# dragapult-blaziken, dragapult-dusknoir, dragapult-dudunsparce as four
# separate decks). Limitless's combined-mode view rolls these up under a
# single "Dragapult" archetype, while keeping Ogerpon Meganium and Ogerpon
# Box as separate archetypes -- which is the differentiation the role
# analysis needs to surface family-level meta share.
#
# This cell builds {trainerhill_slug -> archetype_slug} once, caches it to
# data/deck-archetypes.json, and reuses the cache on subsequent runs. Flip
# REFRESH_ARCHETYPES to True to re-pull from the network.

import datetime as _dt
import json as _json
import re as _re
from pathlib import Path as _Path

REFRESH_ARCHETYPES = False  # flip to True (or delete the JSON) to re-fetch
ARCHETYPE_MAX_AGE_DAYS = 30  # warn past this; a stale map files new decks as singletons
LIMITLESS_BASE = 'https://play.limitlesstcg.com/decks'
LIMITLESS_FORMAT = 'standard'
ARCHETYPE_CACHE_PATH = _Path('data/deck-archetypes.json')

_IMG_RE = _re.compile(r'/pokemon/(?:gen\d+/)?([^./]+)\.(?:png|webp)', _re.I)
_SLUG_RE = _re.compile(r'/decks/([^?/]+)')
# Use-rate column, e.g. "7.83%". Anchored so a signed change column ("+1.2%")
# can't be mistaken for it.
_SHARE_RE = _re.compile(r'^(\d+(?:\.\d+)?)\s*%$')


def _fetch_limitless_html(combine):
    import requests  # local import: cell stays usable from cached file alone
    params = {'format': LIMITLESS_FORMAT}
    if combine:
        params['combine'] = 'true'
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/120.0'}
    r = requests.get(LIMITLESS_BASE, params=params, headers=headers, timeout=30)
    r.raise_for_status()
    return r.text


def _parse_meta_table(html):
    """Yield {rank, name, slug, pokemon} for each data row in the .meta table."""
    from bs4 import BeautifulSoup
    soup = BeautifulSoup(html, 'html.parser')
    table = soup.find('table', class_='meta')
    if not table:
        return []
    rows = []
    for tr in table.find_all('tr')[1:]:
        tds = tr.find_all('td')
        if len(tds) < 3:
            continue
        # tds[1] holds the Pokemon icons, tds[2] holds the linked deck name
        pokemons = []
        for img in tds[1].find_all('img'):
            m = _IMG_RE.search(img.get('src', ''))
            if m:
                pokemons.append(m.group(1).lower())
        name_a = tds[2].find('a')
        name = (name_a.get_text(strip=True) if name_a else tds[2].get_text(strip=True))
        slug_m = _SLUG_RE.search(name_a.get('href', '') if name_a else '')
        slug = slug_m.group(1) if slug_m else None
        if not slug:
            continue
        share = None
        for td in tds:
            share_m = _SHARE_RE.match(td.get_text(strip=True))
            if share_m:
                share = float(share_m.group(1))
                break
        # The table is sorted by share, so row order is the use-rate ranking.
        # 'other' is a bucket, not a deck, and must not consume a rank.
        rank = None if slug == 'other' else len([r for r in rows if r['slug'] != 'other'])
        rows.append({'name': name, 'slug': slug, 'pokemon': pokemons,
                     'rank': rank, 'share': share})
    return rows


def _build_archetype_map_from_network():
    combined_rows = _parse_meta_table(_fetch_limitless_html(combine=True))
    variant_rows = _parse_meta_table(_fetch_limitless_html(combine=False))
    if not combined_rows or not variant_rows:
        raise RuntimeError('Limitless returned no rows; site structure may have changed.')

    combined_slug_set = {r['slug'] for r in combined_rows}
    archetype_name_by_slug = {r['slug']: r['name'] for r in combined_rows}
    # Longest-Pokemon-prefix wins for icon-based matching (so Ogerpon Meganium
    # = [ogerpon, meganium] beats Dragapult = [dragapult] when matching a
    # hypothetical [ogerpon, meganium, dusknoir] variant).
    arch_by_prefix_len = sorted(combined_rows, key=lambda r: -len(r['pokemon']))

    variant_to_archetype = {}
    for v in variant_rows:
        if v['slug'] in combined_slug_set:
            variant_to_archetype[v['slug']] = v['slug']
            continue
        # 1) Slug prefix wins. A variant slug extends its archetype's slug
        #    (ogerpon-meganium -> ogerpon-meganium-arboliva), and that beats icon
        #    matching because Limitless tags such rows with the variant's own
        #    Pokemon (ogerpon + arboliva) rather than the archetype's
        #    (ogerpon + meganium) -- which used to file them under Basic Box.
        matched = None
        for cand in combined_slug_set:
            if v['slug'].startswith(cand + '-') and (matched is None or len(cand) > len(matched)):
                matched = cand
        # 2) Otherwise the Pokemon-icon prefix rule, longest archetype first.
        if matched is None and v['pokemon']:
            for arch in arch_by_prefix_len:
                n = len(arch['pokemon'])
                if n == 0:
                    continue
                if v['pokemon'][:n] == arch['pokemon']:
                    matched = arch['slug']
                    break
        # No looser fallback than that: an unmatched deck becoming its own
        # singleton archetype is better than a confidently wrong parent.
        if matched is not None:
            variant_to_archetype[v['slug']] = matched
    return {
        'fetched_at': _dt.datetime.now(_dt.timezone.utc).isoformat(timespec='seconds'),
        'format': LIMITLESS_FORMAT,
        'source_combined': f'{LIMITLESS_BASE}?format={LIMITLESS_FORMAT}&combine=true',
        'source_uncombined': f'{LIMITLESS_BASE}?format={LIMITLESS_FORMAT}',
        'archetype_name_by_slug': archetype_name_by_slug,
        'variant_to_archetype': variant_to_archetype,
        'variant_rank_by_slug': {r['slug']: r['rank'] for r in variant_rows
                                 if r.get('rank') is not None},
        'variant_share_by_slug': {r['slug']: r['share'] for r in variant_rows
                                  if r.get('share') is not None},
    }


def _cache_age_days(payload):
    ts = payload.get('fetched_at')
    if not ts:
        return None
    try:
        fetched = _dt.datetime.fromisoformat(ts)
    except ValueError:
        return None
    if fetched.tzinfo is None:
        fetched = fetched.replace(tzinfo=_dt.timezone.utc)
    return (_dt.datetime.now(_dt.timezone.utc) - fetched).days


def _archetype_by_slug_prefix(deck, archetype_slugs):
    """Longest archetype slug that is a dash-boundary prefix of `deck`.

    Trainerhill names a new variant by extending the archetype slug
    (`ogerpon-meganium` -> `ogerpon-meganium-arboliva`), so this files variants
    that appeared after the cache was built instead of stranding them as
    singletons. Dash-boundary only, so `toxtricity-box` can never match
    `toxtricity-pfl`.
    """
    best = None
    for slug in archetype_slugs:
        if deck.startswith(slug + '-') and (best is None or len(slug) > len(best)):
            best = slug
    return best


def _load_or_fetch_archetype_map():
    if ARCHETYPE_CACHE_PATH.exists() and not REFRESH_ARCHETYPES:
        payload = _json.loads(ARCHETYPE_CACHE_PATH.read_text(encoding='utf-8'))
        age = _cache_age_days(payload)
        if age is not None and age > ARCHETYPE_MAX_AGE_DAYS:
            print(f"  WARN: archetype cache is {age} days old (fetched {payload.get('fetched_at')}). "
                  f"Decks released since then rely on slug-prefix matching; "
                  f"set REFRESH_ARCHETYPES = True to re-pull.")
        return payload
    print(f"Fetching Limitless archetype map (REFRESH_ARCHETYPES={REFRESH_ARCHETYPES})...")
    payload = _build_archetype_map_from_network()
    ARCHETYPE_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    ARCHETYPE_CACHE_PATH.write_text(_json.dumps(payload, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f"  -> {len(payload['variant_to_archetype'])} variant entries cached to {ARCHETYPE_CACHE_PATH}")
    return payload


_arch_payload = _load_or_fetch_archetype_map()
_variant_to_archetype = _arch_payload['variant_to_archetype']
_archetype_name_by_slug = _arch_payload['archetype_name_by_slug']

# Map our trainerhill `analysis_decks` slugs to archetype slugs. Trainerhill
# and Limitless share the same kebab-case slug convention -- direct equality
# matches every deck we've seen so far. Any unmatched deck becomes its own
# singleton archetype so it never silently drops out of the role analysis.
archetype_by_deck = {}
archetype_display_name = {}
unmatched_trainerhill_decks = []
prefix_matched_decks = []
for _d in analysis_decks:
    arch = _variant_to_archetype.get(_d)
    if arch is None:
        # Not in the cache: a variant newer than the map. Try the slug prefix
        # before giving up and making it its own archetype.
        arch = _archetype_by_slug_prefix(_d, _archetype_name_by_slug.keys())
        if arch is not None:
            prefix_matched_decks.append((_d, arch))
    if arch is None:
        unmatched_trainerhill_decks.append(_d)
        arch = _d  # singleton archetype
    archetype_by_deck[_d] = arch
    archetype_display_name[arch] = _archetype_name_by_slug.get(arch, _d.replace('-', ' ').title())

print(f"Archetype map: {len(set(archetype_by_deck.values()))} archetypes covering {len(archetype_by_deck)} decks.")
if prefix_matched_decks:
    print(f"  Matched by slug prefix (newer than the cache): "
          f"{[f'{d} -> {a}' for d, a in prefix_matched_decks]}")
if unmatched_trainerhill_decks:
    print(f"  Unmatched (treated as singleton archetypes): {unmatched_trainerhill_decks}")
# Per-archetype member preview
from collections import defaultdict as _dd
_members = _dd(list)
for _d, _a in archetype_by_deck.items():
    _members[_a].append(_d)
for _a in sorted(_members, key=lambda a: -len(_members[a])):
    if len(_members[_a]) >= 2:
        print(f"  {_archetype_name_by_slug.get(_a, _a)}: {_members[_a]}")


# --- Core meta vs sleeper candidates -----------------------------------------
# Ranked by Trainer Hill's share of PLAYERS for the same window and sources as
# the matchups (written by the fetch cell). Two reasons this is the source:
#   * games played is NOT use rate -- decks that convert into the top cut play
#     more matches per entry, which is how a 0.53%-share deck once landed at
#     rank 25;
#   * it belongs to the window. Limitless's table is today's ranking, so a build
#     pinned to an earlier date would have picked its core decks with hindsight,
#     and the prediction ledger scores exactly those builds.
# Limitless is still the source of archetype FAMILIES (above). A deck with no
# recorded share falls back to a games-based ordering, behind every ranked deck.
_rank_platform = 'all' if 'all' in ACTIVE_PLATFORMS else ACTIVE_PLATFORMS[0]
_th_share = _th_csv.read_shares(DATA_DIR, _rank_platform)
if not _th_share:
    print(f"  WARN: no archetype-share file for {_rank_platform!r} in {DATA_DIR}; "
          f"ranking by games played until the fetch cell writes one.")

_rank_df = raw_by_platform[_rank_platform]['All']
_rank_df = _rank_df[~_rank_df['is_mirror'] & ~_rank_df['is_other']]
deck_usage_games = (_rank_df.groupby('deck1')['actual_total'].sum()
                    .reindex(analysis_decks, fill_value=0)
                    .sort_values(ascending=False))
_games_order = {d: i for i, d in enumerate(deck_usage_games.index)}
_unranked_decks = [d for d in analysis_decks if d not in _th_share]
ranked_decks = sorted(
    analysis_decks,
    key=lambda d: (0, -_th_share[d]) if d in _th_share
    else (1, _games_order.get(d, len(analysis_decks))),
)
CORE_DECKS = ranked_decks[:CORE_N]
# A candidate with no recorded games has nothing to judge, so it is not a sleeper.
SLEEPER_DECKS = [d for d in ranked_decks[CORE_N:] if float(deck_usage_games.get(d, 0)) > 0]

_games_share = ((deck_usage_games / deck_usage_games.sum() * 100)
                if deck_usage_games.sum() > 0 else deck_usage_games.astype(float))
DECK_USE_RATE = pd.Series({d: float(_th_share[d]) if d in _th_share
                           else float(_games_share.get(d, 0.0))
                           for d in analysis_decks})

print(f"Core meta: {len(CORE_DECKS)} decks | sleeper candidates: {len(SLEEPER_DECKS)} "
      f"(use-rate ranking from Trainer Hill shares, {_rank_platform!r}).")
if _unranked_decks:
    print(f"  No recorded share -- ranked by games played, behind the rest: {_unranked_decks}")


In [ ]:
# === Compute analyses for every platform ===
import sys
from pathlib import Path
sys.path.insert(0, str(Path('scripts').resolve()))
import meta_structure as mst
import meta_skill as msk
import meta_drift as md
# Encapsulates the entire analytical pipeline (cells originally 4-9) in a
# function. Runs once per platform (3x) into `analyses` -- downstream display
# cells and the dashboard pull from this dict.

import json as jsonlib

PLATFORM_LABELS = {'all': 'All Sources', 'online': 'Online Only', 'inperson': 'In-Person Majors'}
# A raw view carries its platform's label: the dashboard's checkbox says which one is shown.
PLATFORM_LABELS.update({raw_view(p): label for p, label in list(PLATFORM_LABELS.items())})
TIER_WEIGHTS = {'All': 1, 'Upper50': 2, 'Upper25': 4}

def _wilson_ci(wins, total, z=1.96):
    if total == 0:
        return 0.5, 0.0, 1.0
    p = wins / total
    denom = 1 + z**2 / total
    center = (p + z**2 / (2 * total)) / denom
    spread = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
    return p, max(0, center - spread), min(1, center + spread)

wilson_ci = _wilson_ci


def _frame(rows, columns):
    """Build a DataFrame from row dicts, keeping the schema when there are none.

    `pd.DataFrame([])` has no columns at all, so a slice with no entries would
    make every downstream `df['tier']` / `df['deck']` lookup raise KeyError."""
    if rows:
        return pd.DataFrame(rows)
    return pd.DataFrame({c: pd.Series(dtype='object') for c in columns})


def compute_platform(platform):
    # These are the "standard" tabs, and they cover the core meta only: matchups
    # are filtered to core-vs-core and popularity renormalised over CORE_DECKS,
    # so fetching 50 archetypes instead of 25 doesn't change what they mean.
    # Decks outside the core are covered by the Sleepers tab and the simulator.
    # `platform` may be a raw view: its matrices are already unshrunk (cell 3).
    plat_label = PLATFORM_LABELS[platform]
    adjust = is_adjusted(platform)
    p_raw = raw_by_platform[platform]
    p_clean = {k: v[v['deck1'].isin(CORE_DECKS) & v['deck2'].isin(CORE_DECKS)].copy()
               for k, v in clean_by_platform[platform].items()}
    p_matrices = {tier: {k: m.reindex(index=CORE_DECKS, columns=CORE_DECKS)
                         for k, m in mats.items()}
                  for tier, mats in matrices_by_platform[platform].items()}
    # Break-even for this slice: the pooled tie-aware rate, not a hardcoded 50%.
    _shrink = shrinkage_by_platform[platform]['All']['wr3']
    neutral_wr = float(_shrink['mean']) * 100
    shrink_k = float(_shrink['k'])
    p_pop = (p_clean['All'].groupby('deck1')['actual_total'].sum()
             .reindex(CORE_DECKS, fill_value=0))
    pop_total = p_pop.sum()
    p_pop_w = (p_pop / pop_total).values if pop_total > 0 else np.zeros(len(CORE_DECKS))
    pop_by_deck = p_pop / pop_total if pop_total > 0 else p_pop

    # 1) Aggregate win rates per (deck, tier)
    agg_rows = []
    for tier in ['All', 'Upper50', 'Upper25']:
        df = p_clean[tier]
        for deck in CORE_DECKS:
            deck_rows = df[df['deck1'] == deck]
            total_w = deck_rows['wins'].sum()
            total_l = deck_rows['losses'].sum()
            total_t = deck_rows['ties'].sum()
            total_g = total_w + total_l + total_t
            if total_g == 0:
                continue
            wr_3pt = (3 * total_w + total_t) / (3 * total_g) * 100
            simple_wr = total_w / total_g
            _, ci_lo, ci_hi = _wilson_ci(total_w, total_g)
            agg_rows.append({
                'deck': deck, 'short_name': DECK_SHORT_NAMES[deck], 'tier': tier,
                'wins': total_w, 'losses': total_l, 'ties': total_t,
                'total_games': total_g, 'wr_3pt': wr_3pt, 'simple_wr': simple_wr * 100,
                'ci_lo': ci_lo * 100, 'ci_hi': ci_hi * 100,
            })
    agg_df = _frame(agg_rows, ['deck', 'short_name', 'tier', 'wins', 'losses', 'ties',
                               'total_games', 'wr_3pt', 'simple_wr', 'ci_lo', 'ci_hi'])

    if len(agg_df):
        sort_order = (agg_df[agg_df['tier'] == 'All']
                      .sort_values('wr_3pt', ascending=True)['deck'].tolist())
    else:
        sort_order = CORE_DECKS[:]

    fig_wr = go.Figure()
    tier_display = {'All': 'All Players', 'Upper50': 'Top 50%', 'Upper25': 'Top 25%'}
    for tier in ['All', 'Upper50', 'Upper25']:
        td = agg_df[agg_df['tier'] == tier].set_index('deck').reindex(sort_order).reset_index()
        fig_wr.add_trace(go.Bar(
            y=[DECK_SHORT_NAMES[d] for d in td['deck']],
            x=td['wr_3pt'], name=tier_display[tier], orientation='h',
            marker=dict(color=TIER_COLORS[tier], line=dict(width=0)),
            error_x=dict(type='data', symmetric=False,
                         array=(td['ci_hi'] - td['simple_wr']).values,
                         arrayminus=(td['simple_wr'] - td['ci_lo']).values,
                         thickness=1.5, width=3, color=UI.faint),
            hovertemplate=('<b>%{y}</b><br>'
                           f'Tier: {tier_display[tier]}<br>'
                           'Win Rate (3pt): %{x:.1f}%<br>'
                           'Games: %{customdata[0]:,.0f}<br>'
                           'Record: %{customdata[1]:.0f}W-%{customdata[2]:.0f}L-%{customdata[3]:.0f}T'
                           '<extra></extra>'),
            customdata=td[['total_games', 'wins', 'losses', 'ties']].values,
        ))
    fig_wr.add_vline(x=neutral_wr, line_dash='dot', line_color=UI.rule, line_width=2)
    fig_wr.update_layout(
        title=f'Overall Win Rate by Deck and Finishing Tier - {plat_label}',
        xaxis_title='Win Rate (3-Point System, %)', yaxis_title='', barmode='group',
        height=max(650, len(CORE_DECKS) * 32),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        margin=dict(l=130, r=20, t=60, b=50), bargap=0.2, bargroupgap=0.05,
    )

    # 2) Per-deck break-even + Weighted tier
    # A deck's own mirror is the only matchup that is even by definition, so it is
    # what separates its favorable matchups from its unfavorable ones -- ties push
    # that line below 50% by different amounts for different decks. `_mirror_baselines`
    # (cell 3) corrects the export's mirror tie accounting and shrinks the result by
    # how many mirrors it rests on, so a deck with one recorded mirror sits on the
    # format's prior instead of on that single result.
    mirror_wr = {}
    for tier in ['All', 'Upper50', 'Upper25']:
        mb = mirror_by_platform[platform][tier]
        mirror_wr[tier] = {d: mb['baseline'].get(d, mb['prior']) for d in CORE_DECKS}
    mirror_wr['Weighted'] = {
        d: sum(mirror_wr[t][d] * w for t, w in TIER_WEIGHTS.items()) / sum(TIER_WEIGHTS.values())
        for d in CORE_DECKS
    }

    # Weighted tier: games weighted toward top finishers, over DISJOINT finishing
    # bands -- bottom 50% x1, 25-50% x2, top 25% x4. It used to add the nested
    # slices themselves (All x1 + Upper50 x2 + Upper25 x4), which counted a top-25%
    # game 7 times and a 25-50% game 3 times, and then fit shrinkage on those
    # inflated counts as if they were 2-7x more games than exist. The weighting only
    # changes the RATE; the sample behind it is the real game count. Like every
    # finishing-tier view it selects on results, so it is not a skill measure.
    band_w = (1.0, 2.0, 4.0)
    keyed = {t: p_clean[t].set_index(['deck1', 'deck2'])[['wins', 'losses', 'ties', 'actual_total']]
             for t in ('All', 'Upper50', 'Upper25')}
    weighted_rows = []
    for (deck, opp), a_row in keyed['All'].iterrows():
        if deck == opp or a_row['actual_total'] <= 0:
            continue
        h_row = keyed['Upper50'].loc[(deck, opp)] if (deck, opp) in keyed['Upper50'].index else 0 * a_row
        q_row = keyed['Upper25'].loc[(deck, opp)] if (deck, opp) in keyed['Upper25'].index else 0 * a_row
        bands = [(a_row - h_row).clip(lower=0), (h_row - q_row).clip(lower=0), q_row]
        ww = sum(m * b['wins'] for m, b in zip(band_w, bands))
        wl = sum(m * b['losses'] for m, b in zip(band_w, bands))
        wt = sum(m * b['ties'] for m, b in zip(band_w, bands))
        if ww + wl + wt <= 0:
            continue
        weighted_rows.append({
            'deck1': deck, 'deck2': opp,
            'wins': ww, 'losses': wl, 'ties': wt, 'actual_total': float(a_row['actual_total']),
            'win_rate': (3 * ww + wt) / (3 * (ww + wl + wt)) * 100,
            'wr3': (3 * ww + wt) / (3 * (ww + wl + wt)),
            'is_mirror': False, 'is_other': False,
        })
    wdf = _frame(weighted_rows, ['deck1', 'deck2', 'wins', 'losses', 'ties', 'actual_total',
                                 'win_rate', 'wr3', 'is_mirror', 'is_other'])
    if len(wdf):
        _w_neutral, _w_k = _eb_shrinkage(wdf['wr3'], wdf['actual_total'], adjust)
        _shrink_column(wdf, 'wr3', 'smoothed_wr3', _w_neutral, _w_k)
    else:
        wdf['smoothed_wr3'] = pd.Series(dtype='float64')
    p_clean['Weighted'] = wdf

    # 3) Consistency
    consistency_rows = []
    matchup_breakdown = {t: {} for t in ['All', 'Upper50', 'Upper25', 'Weighted']}
    for tier in ['All', 'Upper50', 'Upper25', 'Weighted']:
        df = p_clean[tier]
        for deck in CORE_DECKS:
            deck_rows = df[df['deck1'] == deck].copy()
            if len(deck_rows) == 0: continue
            fav_impact = unfav_impact = 0.0
            fav_details = []; unfav_details = []
            weighted_wrs = []; total_pop = 0.0; mu_list = []
            for _, row in deck_rows.iterrows():
                opp = row['deck2']
                if opp not in pop_by_deck.index: continue
                wr = float(row['smoothed_wr3']) * 100  # sample-adjusted
                raw_wr = float(row['win_rate'])
                opp_pop = pop_by_deck[opp]
                total_pop += opp_pop
                weighted_wrs.append((wr, opp_pop))
                baseline = mirror_wr[tier].get(deck, neutral_wr)
                if wr > baseline:
                    contrib = opp_pop * (wr - baseline)
                    fav_impact += contrib
                    fav_details.append((opp, wr, opp_pop, contrib))
                elif wr < baseline:
                    contrib = opp_pop * (baseline - wr)
                    unfav_impact += contrib
                    unfav_details.append((opp, wr, opp_pop, contrib))
                mu_list.append({
                    'opponent': opp, 'opp_short': DECK_SHORT_NAMES.get(opp, opp),
                    'win_rate': round(wr, 1), 'raw_wr': round(raw_wr, 1),
                    'opp_pop': round(float(opp_pop * 100), 2),
                    'games': int(round(row['actual_total'])),   # tier counts can be fractional
                })
            if total_pop == 0: continue
            fav_impact /= total_pop; unfav_impact /= total_pop
            baseline = mirror_wr[tier].get(deck, neutral_wr)
            for mu in mu_list:
                wr = mu['win_rate']
                opp_pop_norm = mu['opp_pop'] / 100 / total_pop
                if wr > baseline:
                    mu['net_impact'] = round(opp_pop_norm * (wr - baseline), 3)
                elif wr < baseline:
                    mu['net_impact'] = round(-opp_pop_norm * (baseline - wr), 3)
                else:
                    mu['net_impact'] = 0.0
            matchup_breakdown[tier][deck] = sorted(mu_list, key=lambda x: x['net_impact'])
            wrs_arr = np.array([w for w, _ in weighted_wrs])
            pops_arr = np.array([p for _, p in weighted_wrs])
            pops_arr /= pops_arr.sum()
            wmean = np.average(wrs_arr, weights=pops_arr)
            wvar = np.average((wrs_arr - wmean)**2, weights=pops_arr)
            wstd = np.sqrt(wvar)
            fav_details.sort(key=lambda x: -x[3])
            unfav_details.sort(key=lambda x: -x[3])
            consistency_rows.append({
                'deck': deck, 'short_name': DECK_SHORT_NAMES[deck], 'tier': tier,
                'fav_impact': fav_impact, 'unfav_impact': unfav_impact,
                'net_impact': fav_impact - unfav_impact,
                'matchup_std': wstd, 'agg_wr': wmean,
                'top_fav': fav_details[0][0] if fav_details else 'none',
                'top_unfav': unfav_details[0][0] if unfav_details else 'none',
                'n_fav': len(fav_details), 'n_unfav': len(unfav_details),
            })
    consist_df = _frame(consistency_rows, ['deck', 'short_name', 'tier', 'fav_impact',
                                           'unfav_impact', 'net_impact', 'matchup_std',
                                           'agg_wr', 'top_fav', 'top_unfav', 'n_fav', 'n_unfav'])
    all_consist = consist_df[consist_df['tier'] == 'All'].copy()

    fig_consist = go.Figure()
    if len(all_consist):
        fig_consist.add_trace(go.Scatter(
            x=all_consist['unfav_impact'], y=all_consist['fav_impact'],
            mode='markers+text', text=all_consist['short_name'],
            textposition='top center', textfont=dict(size=9, color=UI.text2),
            marker=dict(size=16, color=[DECK_COLORS[d] for d in all_consist['deck']],
                        line=dict(width=2, color='white'), opacity=0.9),
            hovertemplate=('<b>%{customdata[0]}</b><br>'
                           'Favorable Impact: %{y:.2f}<br>'
                           'Unfavorable Impact: %{x:.2f}<br>'
                           'Net: %{customdata[1]}<br>'
                           'Best MU: %{customdata[2]}<br>'
                           'Worst MU: %{customdata[3]}<br>'
                           'Favorable/Unfavorable: %{customdata[4]}/%{customdata[5]}<br>'
                           '<i>Click for matchup breakdown</i><extra></extra>'),
            # Signed numbers are pre-formatted: Plotly 2.x rejects '+.2f' in a
            # hovertemplate and silently prints the raw, unrounded value.
            customdata=np.column_stack([
                all_consist['deck'].values, [f'{v:+.2f}' for v in all_consist['net_impact']],
                all_consist[['top_fav', 'top_unfav', 'n_fav', 'n_unfav']].values]),
        ))
        max_val = max(all_consist['fav_impact'].max(), all_consist['unfav_impact'].max()) * 1.1
    else:
        max_val = 1.0
    fig_consist.add_shape(type='line', x0=0, y0=0, x1=max_val, y1=max_val,
                          line=dict(dash='dash', color=UI.rule, width=1.5))
    annotations = [
        dict(x=0.02, y=0.98, text='Good MUs vs Popular<br>Few Bad MUs', xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color=FAVORABLE_COLOR), opacity=0.5),
        dict(x=0.98, y=0.98, text='Polarized:<br>Strong & Weak vs Popular', xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color=UI.amber), xanchor='right', opacity=0.5),
        dict(x=0.02, y=0.02, text='Neutral:<br>Even Across the Board', xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color=UI.faint), opacity=0.6),
        dict(x=0.98, y=0.02, text='Bad MUs vs Popular<br>Few Good MUs', xref='paper', yref='paper',
             showarrow=False, font=dict(size=10, color=UNFAVORABLE_COLOR), xanchor='right', opacity=0.5),
    ]
    fig_consist.update_layout(
        title=f'Matchup Consistency - {plat_label}',
        xaxis_title='Unfavorable Impact (bad MUs weighted by opponent popularity)',
        yaxis_title='Favorable Impact (good MUs weighted by opponent popularity)',
        height=650, annotations=annotations, showlegend=False,
        margin=dict(l=80, r=30, t=80, b=60),
    )

    consist_scatter_data = {}
    for tier in ['All', 'Upper50', 'Upper25', 'Weighted']:
        td = consist_df[consist_df['tier'] == tier].copy()
        if len(td) == 0: continue
        consist_scatter_data[tier] = {
            'x': td['unfav_impact'].round(4).tolist(),
            'y': td['fav_impact'].round(4).tolist(),
            'text': td['short_name'].tolist(),
            'marker_colors': [DECK_COLORS[d] for d in td['deck']],
            'customdata': np.column_stack([
                td['deck'].values, [f'{v:+.2f}' for v in td['net_impact']],
                td[['top_fav', 'top_unfav', 'n_fav', 'n_unfav']].values]).tolist(),
            'max_val': round(max(td['fav_impact'].max(), td['unfav_impact'].max()) * 1.1, 4),
        }

    # 4) Heatmaps + lopsided + skill matchup
    # Restrict heatmap axes to decks that actually have games on this platform.
    wr_sort = [d for d in p_pop.sort_values(ascending=False).index.tolist() if p_pop[d] > 0]
    active_decks = set(wr_sort)
    short_sorted = [DECK_SHORT_NAMES[d] for d in wr_sort]

    DELTA_HEATMAP_SCALE = [[0, UI.rose], [0.5, UI.div_mid], [1, UI.emerald]]
    all_smooth = p_matrices['All']['smoothed_wr'].reindex(index=wr_sort, columns=wr_sort) * 100
    u25_smooth = p_matrices['Upper25']['smoothed_wr'].reindex(index=wr_sort, columns=wr_sort) * 100
    all_sample = p_matrices['All']['sample_n'].reindex(index=wr_sort, columns=wr_sort)
    u25_sample = p_matrices['Upper25']['sample_n'].reindex(index=wr_sort, columns=wr_sort)
    delta_data = u25_smooth - all_smooth

    delta_hover = []
    for i, d1 in enumerate(wr_sort):
        row = []
        for j, d2 in enumerate(wr_sort):
            d = delta_data.iloc[i, j]; a = all_smooth.iloc[i, j]; u = u25_smooth.iloc[i, j]
            na = all_sample.iloc[i, j]; nu = u25_sample.iloc[i, j]
            if pd.isna(d) or pd.isna(a) or pd.isna(u):
                row.append(f'{DECK_SHORT_NAMES[d1]} vs {DECK_SHORT_NAMES[d2]}<br>No data')
            else:
                row.append(f'<b>{DECK_SHORT_NAMES[d1]} vs {DECK_SHORT_NAMES[d2]}</b><br>'
                           f'All WR: {a:.1f}%<br>Top 25% WR: {u:.1f}%<br>'
                           f'Delta: {d:+.1f}%<br>Games (All): {int(na)}<br>'
                           f'Games (Top 25%): {int(nu) if not pd.isna(nu) else "N/A"}')
        delta_hover.append(row)
    if len(wr_sort):
        np.fill_diagonal(delta_data.values, np.nan)
    fig_skill_mu_heatmap = go.Figure(data=go.Heatmap(
        z=delta_data.values, x=short_sorted, y=short_sorted,
        colorscale=DELTA_HEATMAP_SCALE, zmid=0, zmin=-15, zmax=15,
        text=delta_hover, hoverinfo='text',
        colorbar=dict(title=dict(text='WR Delta %', font=dict(size=12, color=UI.text2)),
                      thickness=14, len=0.6, tickfont=dict(size=10, color=UI.muted), outlinewidth=0),
    ))
    fig_skill_mu_heatmap.update_layout(
        title=f'Matchup WR Change: All -> Top 25% - {plat_label}',
        xaxis_title='Opponent', yaxis_title='Deck', height=800, width=900,
        xaxis=dict(tickangle=45, side='bottom'), yaxis=dict(autorange='reversed'),
        margin=dict(l=120, r=60, t=60, b=100),
    )

    skill_mu_rows = []
    seen_pairs = set()
    for d1 in CORE_DECKS:
        for d2 in CORE_DECKS:
            if d1 == d2: continue
            pair = tuple(sorted([d1, d2]))
            if pair in seen_pairs: continue
            seen_pairs.add(pair)
            n_all = p_matrices['All']['sample_n'].loc[d1, d2]
            n_u25 = p_matrices['Upper25']['sample_n'].loc[d1, d2]
            if pd.isna(n_all) or n_all < MIN_SAMPLE or pd.isna(n_u25) or n_u25 < MIN_SAMPLE:
                continue
            wr_all = p_matrices['All']['smoothed_wr'].loc[d1, d2] * 100
            wr_u25 = p_matrices['Upper25']['smoothed_wr'].loc[d1, d2] * 100
            if pd.isna(wr_all) or pd.isna(wr_u25): continue
            delta = wr_u25 - wr_all
            skill_mu_rows.append({
                'deck1': d1, 'deck2': d2,
                'd1_short': DECK_SHORT_NAMES[d1], 'd2_short': DECK_SHORT_NAMES[d2],
                'wr_all': wr_all, 'wr_u25': wr_u25,
                'delta': delta, 'abs_delta': abs(delta),
                # Tier counts can be fractional (dropped players sharing a place): round, not truncate.
                'games_all': int(round(n_all)), 'games_u25': int(round(n_u25)),
            })
    skill_mu_df = _frame(skill_mu_rows, ['deck1', 'deck2', 'd1_short', 'd2_short', 'wr_all',
                                         'wr_u25', 'delta', 'abs_delta', 'games_all', 'games_u25'])
    if len(skill_mu_df):
        top_pos = skill_mu_df.nlargest(15, 'delta')
        top_neg = skill_mu_df.nsmallest(15, 'delta')
        top_skill_mu = pd.concat([top_pos, top_neg]).drop_duplicates().sort_values('delta', ascending=True)
    else:
        top_skill_mu = skill_mu_df

    fig_skill_mu_top = go.Figure()
    if len(top_skill_mu):
        fig_skill_mu_top.add_trace(go.Bar(
            y=[f"{r['d1_short']}  vs  {r['d2_short']}" for _, r in top_skill_mu.iterrows()],
            x=top_skill_mu['delta'], orientation='h',
            marker=dict(color=[FAVORABLE_COLOR if d >= 0 else UNFAVORABLE_COLOR
                               for d in top_skill_mu['delta']], line=dict(width=0)),
            hovertemplate=('<b>%{customdata[0]} vs %{customdata[1]}</b><br>'
                           'All WR: %{customdata[2]:.1f}%<br>'
                           'Top 25% WR: %{customdata[3]:.1f}%<br>'
                           'Delta: %{customdata[6]}%<br>'
                           'Games (All): %{customdata[4]}<br>'
                           'Games (Top 25%): %{customdata[5]}<extra></extra>'),
            customdata=np.column_stack([
                top_skill_mu[['deck1', 'deck2', 'wr_all', 'wr_u25', 'games_all', 'games_u25']].values,
                [f'{v:+.1f}' for v in top_skill_mu['delta']]]),
        ))
    fig_skill_mu_top.add_vline(x=0, line_dash='dash', line_color=UI.rule, line_width=1.5)
    fig_skill_mu_top.update_layout(
        title=f'Matchups that shift most among top finishers - {plat_label}',
        xaxis_title='Win Rate Delta (Top 25% - All) %', yaxis_title='',
        height=max(600, len(top_skill_mu) * 24) if len(top_skill_mu) else 400,
        margin=dict(l=200, r=30, t=60, b=50), showlegend=False,
    )

    heatmap_data = p_matrices['All']['smoothed_wr'].reindex(index=wr_sort, columns=wr_sort) * 100
    sample_data = p_matrices['All']['sample_n'].reindex(index=wr_sort, columns=wr_sort)
    raw_data = p_matrices['All']['raw_wr'].reindex(index=wr_sort, columns=wr_sort)
    # The plotted value is wins / games (smoothed unless this is the raw view);
    # "Raw WR" is the tie-aware 3-point record.
    _plotted = 'Smoothed' if adjust else 'Wins / games'
    hover_text = []
    for i, d1 in enumerate(wr_sort):
        row = []
        for j, d2 in enumerate(wr_sort):
            n = sample_data.iloc[i, j]; raw = raw_data.iloc[i, j]; sm = heatmap_data.iloc[i, j]
            if pd.isna(n):
                row.append(f'{DECK_SHORT_NAMES[d1]} vs {DECK_SHORT_NAMES[d2]}<br>No data')
            else:
                row.append(f'<b>{DECK_SHORT_NAMES[d1]} vs {DECK_SHORT_NAMES[d2]}</b><br>'
                           f'Raw WR: {raw:.1f}%<br>{_plotted}: {sm:.1f}%<br>Games: {int(n)}')
        hover_text.append(row)
    if len(wr_sort):
        np.fill_diagonal(heatmap_data.values, np.nan)
    fig_heatmap = go.Figure(data=go.Heatmap(
        z=heatmap_data.values, x=short_sorted, y=short_sorted,
        colorscale=HEATMAP_SCALE, zmid=neutral_wr, zmin=25, zmax=75,
        text=hover_text, hoverinfo='text',
        colorbar=dict(title=dict(text='Win Rate %', font=dict(size=12, color=UI.text2)),
                      thickness=14, len=0.6, tickfont=dict(size=10, color=UI.muted), outlinewidth=0),
    ))
    fig_heatmap.update_layout(
        title=f'Matchup Win Rates ({"Smoothed" if adjust else "Raw"}) - All Players - {plat_label}',
        xaxis_title='Opponent', yaxis_title='Deck', height=800, width=900,
        xaxis=dict(tickangle=45, side='bottom'), yaxis=dict(autorange='reversed'),
        margin=dict(l=120, r=60, t=60, b=100),
    )

    lopsided = []
    df_all = p_clean['All']
    wr3_m = p_matrices['All']['smoothed_wr3']
    raw_m = p_matrices['All']['raw_wr']
    seen = set()
    for _, row in df_all.iterrows():
        d1, d2 = row['deck1'], row['deck2']
        pair = tuple(sorted([d1, d2]))
        if pair in seen: continue
        seen.add(pair)
        n = row['actual_total']
        if n < MIN_SAMPLE: continue
        wr_fwd, wr_rev = wr3_m.loc[d1, d2], wr3_m.loc[d2, d1]
        if pd.isna(wr_fwd): continue
        wr_fwd *= 100
        wr_rev = (2 * neutral_wr - wr_fwd) if pd.isna(wr_rev) else wr_rev * 100
        if wr_fwd >= wr_rev:
            favored, unfavored, fav_wr = d1, d2, wr_fwd
        else:
            favored, unfavored, fav_wr = d2, d1, wr_rev
        raw_v = raw_m.loc[favored, unfavored]
        lopsided.append({
            'favored': favored, 'unfavored': unfavored,
            'fav_short': DECK_SHORT_NAMES.get(favored, favored),
            'unfav_short': DECK_SHORT_NAMES.get(unfavored, unfavored),
            'win_rate': fav_wr, 'raw_wr': float(raw_v) if not pd.isna(raw_v) else np.nan,
            'deviation': fav_wr - neutral_wr, 'games': n,
        })
    lop_df = _frame(lopsided, ['favored', 'unfavored', 'fav_short', 'unfav_short',
                               'win_rate', 'raw_wr', 'deviation', 'games'])
    lop_df = lop_df.sort_values('deviation', ascending=False).head(20)
    fig_lopsided = go.Figure()
    if len(lop_df):
        fig_lopsided.add_trace(go.Bar(
            y=[f"{r['fav_short']}  -  {r['unfav_short']}" for _, r in lop_df.iterrows()],
            x=lop_df['deviation'], orientation='h',
            marker=dict(color=[DECK_COLORS.get(r['favored'], UI.faint) for _, r in lop_df.iterrows()],
                        line=dict(width=0)),
            # In the raw view the adjusted rate IS the raw one, so it is shown once.
            hovertemplate=('<b>%{customdata[0]} vs %{customdata[1]}</b><br>'
                           + ('Adjusted WR: %{customdata[2]:.1f}%<br>'
                              'Raw WR: %{customdata[4]:.1f}%<br>' if adjust else
                              'Raw WR: %{customdata[2]:.1f}%<br>')
                           + 'Deviation from neutral: +%{x:.1f}%<br>'
                           'Games: %{customdata[3]}<extra></extra>'),
            customdata=lop_df[['favored', 'unfavored', 'win_rate', 'games', 'raw_wr']].values,
        ))
    fig_lopsided.update_layout(
        title=f'Top 20 Most Lopsided Matchups ({"sample-adjusted" if adjust else "raw"}) - {plat_label}',
        xaxis_title=f'Win Rate Deviation from neutral ({neutral_wr:.1f}%)', yaxis_title='',
        height=max(500, len(lop_df) * 28) if len(lop_df) else 300,
        margin=dict(l=200, r=30, t=60, b=50), yaxis=dict(autorange='reversed'),
    )

    # 5) Swiss simulation
    n_decks = len(CORE_DECKS)
    deck_to_idx = {d: i for i, d in enumerate(CORE_DECKS)}
    wp_matrix = np.full((n_decks, n_decks), 0.5)
    tp_matrix = np.zeros((n_decks, n_decks))
    for i, d1 in enumerate(CORE_DECKS):
        for j, d2 in enumerate(CORE_DECKS):
            wp = p_matrices['All']['win_prob'].loc[d1, d2]
            tp = p_matrices['All']['tie_prob'].loc[d1, d2]
            if not np.isnan(wp):
                wp_matrix[i, j] = wp
                tp_matrix[i, j] = tp
    rng = np.random.default_rng(42)
    all_mp = {i: [] for i in range(n_decks)}
    # No games on this slice means there is nothing to draw deck assignments
    # from -- simulating uniform play would invent a meta that does not exist.
    weights_for_sim = p_pop_w
    n_sims_eff = N_SIMS if p_pop_w.sum() > 0 else 0
    for sim in range(n_sims_eff):
        decks_assigned = rng.choice(n_decks, size=N_PLAYERS, p=weights_for_sim)
        match_points = np.zeros(N_PLAYERS, dtype=np.int32)
        for rnd in range(SWISS_ROUNDS):
            order = np.lexsort((rng.random(N_PLAYERS), -match_points))
            for k in range(0, N_PLAYERS - 1, 2):
                p1, p2 = order[k], order[k + 1]
                d1, d2 = decks_assigned[p1], decks_assigned[p2]
                roll = rng.random()
                w = wp_matrix[d1, d2]; t = tp_matrix[d1, d2]
                if roll < w:
                    match_points[p1] += 3
                elif roll < w + t:
                    match_points[p1] += 1; match_points[p2] += 1
                else:
                    match_points[p2] += 3
        for pp in range(N_PLAYERS):
            all_mp[decks_assigned[pp]].append(match_points[pp])

    swiss_rows = []
    for i, deck in enumerate(CORE_DECKS):
        mps = np.array(all_mp[i])
        if len(mps) == 0: continue
        swiss_rows.append({
            'deck': deck, 'short_name': DECK_SHORT_NAMES[deck],
            'mean_mp': mps.mean(), 'median_mp': np.median(mps), 'std_mp': mps.std(),
            'p_cut': (mps >= CUT_THRESHOLD).mean() * 100,
            'p_6_2': (mps >= 18).mean() * 100,
            'p_7_1': (mps >= 21).mean() * 100,
            'n_appearances': len(mps),
            'q25': np.percentile(mps, 25), 'q75': np.percentile(mps, 75),
        })
    swiss_df = _frame(swiss_rows, ['deck', 'short_name', 'mean_mp', 'median_mp', 'std_mp',
                                   'p_cut', 'p_6_2', 'p_7_1', 'n_appearances', 'q25', 'q75'])
    swiss_df = swiss_df.sort_values('mean_mp', ascending=True)

    fig_swiss = make_subplots(rows=1, cols=2, column_widths=[0.6, 0.4],
                              subplot_titles=['MP Distribution: filled = P(MP=k), line = P(MP≤k)',
                                              'Top Cut Probability'],
                              horizontal_spacing=0.08)

    def _hex_to_rgba(h, a):
        h = h.lstrip('#')
        return f'rgba({int(h[0:2],16)},{int(h[2:4],16)},{int(h[4:6],16)},{a})'

    # Ridgeline overlay per deck: PMF as a faint filled area + inclusive
    # lower-tail CDF P(MP <= k) as a bold step line. Both share AMP-height scale.
    mp_axis = np.arange(SWISS_ROUNDS * 3 + 1)
    deck_pmfs = {}
    deck_cdfs = {}
    for _, row in swiss_df.iterrows():
        mps = np.array(all_mp[deck_to_idx[row['deck']]])
        if len(mps) == 0:
            continue
        counts = np.bincount(mps, minlength=len(mp_axis))[:len(mp_axis)]
        pmf = counts / counts.sum()
        deck_pmfs[row['deck']] = pmf
        # Inclusive lower-tail cumulative: P(MP <= k).
        deck_cdfs[row['deck']] = np.cumsum(pmf)

    global_max_pmf = max((p.max() for p in deck_pmfs.values()), default=1.0)
    AMP = 0.9
    n_swiss = len(swiss_df)
    for i, (_, row) in enumerate(swiss_df.iterrows()):
        deck = row['deck']
        if deck not in deck_pmfs: continue
        pmf = deck_pmfs[deck]
        cdf = deck_cdfs[deck]

        # Layer 1: PMF as a filled area, normalized so the tallest peak across
        # decks equals AMP. Faint fill so the cumulative line stays readable.
        y_pmf = i + (pmf / global_max_pmf) * AMP
        fig_swiss.add_trace(go.Scatter(
            x=np.concatenate([mp_axis, mp_axis[::-1]]),
            y=np.concatenate([y_pmf, np.full_like(y_pmf, i)]),
            fill='toself', mode='lines',
            fillcolor=_hex_to_rgba(DECK_COLORS[deck], 0.30),
            line=dict(color=_hex_to_rgba(DECK_COLORS[deck], 0.45), width=0.5),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=1)

        # Layer 2: P(MP <= k) as a step line, scale 0..AMP. Rises from 0 to 1.
        y_cdf = i + cdf * AMP
        fig_swiss.add_trace(go.Scatter(
            x=mp_axis, y=y_cdf, mode='lines',
            line=dict(color=DECK_COLORS[deck], width=2, shape='hv'),
            showlegend=False, hoverinfo='skip',
        ), row=1, col=1)

        # Layer 3: invisible hover markers carrying both views.
        fig_swiss.add_trace(go.Scatter(
            x=mp_axis, y=y_cdf, mode='markers',
            marker=dict(size=10, color=DECK_COLORS[deck], opacity=0),
            customdata=np.column_stack([mp_axis, pmf * 100, cdf * 100]),
            hovertemplate=(f"<b>{row['short_name']}</b><br>"
                           "Match Points: %{customdata[0]}<br>"
                           "P(MP = k): %{customdata[1]:.2f}%<br>"
                           "P(MP ≤ k): %{customdata[2]:.2f}%<extra></extra>"),
            showlegend=False,
        ), row=1, col=1)

    if len(swiss_df):
        fig_swiss.add_trace(go.Bar(
            y=swiss_df['short_name'], x=swiss_df['p_cut'], orientation='h',
            marker=dict(color=[DECK_COLORS[d] for d in swiss_df['deck']], line=dict(width=0)),
            hovertemplate=('<b>%{y}</b><br>'
                           'Cut Rate: %{x:.1f}%<br>'
                           'Mean MP: %{customdata[0]:.1f}<br>'
                           'Appearances: %{customdata[1]:,}<extra></extra>'),
            customdata=swiss_df[['mean_mp', 'n_appearances']].values, showlegend=False,
        ), row=1, col=2)
    fig_swiss.add_vline(x=CUT_THRESHOLD, line_dash='dash', line_color=UI.cut,
                        opacity=0.4, row=1, col=1,
                        annotation_text=f'Cut ({CUT_THRESHOLD} MP)',
                        annotation_font=dict(size=10, color=UI.cut))
    fig_swiss.update_layout(
        title=f'Swiss Sim ({N_SIMS:,} runs, {N_PLAYERS} players, {SWISS_ROUNDS} rounds) - {plat_label}',
        height=max(700, n_swiss * 38), showlegend=False,
        margin=dict(l=130, r=30, t=60, b=50),
    )
    fig_swiss.update_xaxes(title_text='Match Points', row=1, col=1, range=[-0.5, mp_axis[-1] + 0.5])
    fig_swiss.update_yaxes(
        tickmode='array',
        tickvals=list(range(n_swiss)),
        ticktext=swiss_df['short_name'].tolist(),
        range=[-0.5, n_swiss - 0.5 + AMP],
        row=1, col=1,
    )
    fig_swiss.update_xaxes(title_text='Top Cut %', row=1, col=2)
    for ann in fig_swiss.layout.annotations:
        ann.font = dict(size=14, color=UI.ink, family=FONT_STACK)

    # 6) Skill ceiling
    skill_rows = []
    for deck in CORE_DECKS:
        row = {'deck': deck, 'short_name': DECK_SHORT_NAMES[deck]}
        for tier in ['All', 'Upper50', 'Upper25']:
            m = agg_df[(agg_df['deck'] == deck) & (agg_df['tier'] == tier)]
            row[f'wr_{tier}'] = m.iloc[0]['wr_3pt'] if len(m) > 0 else np.nan
        row['delta_all_u25'] = (row['wr_Upper25'] - row['wr_All']
                                if not np.isnan(row.get('wr_All', np.nan)) and not np.isnan(row.get('wr_Upper25', np.nan))
                                else np.nan)
        row['delta_all_u50'] = (row['wr_Upper50'] - row['wr_All']
                                if not np.isnan(row.get('wr_All', np.nan)) and not np.isnan(row.get('wr_Upper50', np.nan))
                                else np.nan)
        skill_rows.append(row)
    skill_df = pd.DataFrame(skill_rows)
    for tier in ['All', 'Upper50', 'Upper25']:
        skill_df[f'rank_{tier}'] = skill_df[f'wr_{tier}'].rank(ascending=False, method='min')
    skill_df['rank_shift'] = skill_df['rank_All'] - skill_df['rank_Upper25']

    fig_bump = go.Figure()
    tier_x = {'All': 0, 'Upper50': 1, 'Upper25': 2}
    tier_labels = ['All', 'Top 50%', 'Top 25%']
    for _, row in skill_df.iterrows():
        ranks = [row['rank_All'], row['rank_Upper50'], row['rank_Upper25']]
        if any(np.isnan(r) for r in ranks): continue
        fig_bump.add_trace(go.Scatter(
            x=list(tier_x.values()), y=ranks,
            mode='lines+markers+text', name=row['short_name'],
            text=[row['short_name'], '', row['short_name']],
            textposition=['middle left', 'middle left', 'middle right'],
            textfont=dict(size=8, color=UI.muted),
            line=dict(color=DECK_COLORS[row['deck']], width=3),
            marker=dict(size=10, line=dict(width=2, color='white')),
            hovertemplate=(f"<b>{row['deck']}</b><br>Tier: %{{customdata}}<br>Rank: %{{y:.0f}}<extra></extra>"),
            customdata=[tier_labels[i] for i in range(3)],
        ))
    fig_bump.update_layout(
        title=f'Deck Rank Across Finishing Tiers - {plat_label}',
        xaxis=dict(tickmode='array', tickvals=[0, 1, 2], ticktext=tier_labels),
        yaxis=dict(autorange='reversed', title='Rank', dtick=1),
        height=max(700, n_decks * 28), showlegend=False,
        margin=dict(l=120, r=120, t=60, b=50),
    )

    skill_valid = skill_df.dropna(subset=['delta_all_u25', 'wr_All'])
    fig_skill = go.Figure()
    if len(skill_valid):
        med_wr_skill = skill_valid['wr_All'].median()
        fig_skill.add_trace(go.Scatter(
            x=skill_valid['delta_all_u25'], y=skill_valid['wr_All'],
            mode='markers+text', text=skill_valid['short_name'],
            textposition='top center', textfont=dict(size=9, color=UI.text2),
            marker=dict(size=16, color=[DECK_COLORS[d] for d in skill_valid['deck']],
                        line=dict(width=2, color='white'), opacity=0.9),
            hovertemplate=('<b>%{customdata[0]}</b><br>'
                           'All WR: %{y:.1f}%<br>'
                           'Top 25% - All: %{customdata[3]}%<br>'
                           'Upper25 WR: %{customdata[1]:.1f}%<br>'
                           'Rank Shift: %{customdata[2]}<extra></extra>'),
            customdata=np.column_stack([
                skill_valid[['deck', 'wr_Upper25']].values,
                [f'{v:+.0f}' for v in skill_valid['rank_shift']],
                [f'{v:+.1f}' for v in skill_valid['delta_all_u25']]]),
        ))
        fig_skill.add_hline(y=med_wr_skill, line_dash='dot', line_color=UI.rule, line_width=1.5)
    fig_skill.add_vline(x=0, line_dash='dash', line_color=UI.rule, line_width=1.5)
    fig_skill.update_layout(
        title=f'Win rate among top-25% finishers vs everyone - {plat_label}',
        xaxis_title='Win Rate Delta (Upper25 - All) %',
        yaxis_title='Aggregate Win Rate (All) %',
        height=600, showlegend=False, margin=dict(l=80, r=30, t=60, b=60),
    )

    # 9) Format shape and deck families -------------------------------------
    # Read the matchup matrix as a flow on a graph rather than a list of decks.
    # A flow is a power ranking exactly when some single score per deck
    # reproduces it; the closest such ranking is a weighted least-squares fit,
    # and what is left over is structure no ranking can express.
    #
    # Cyclicity has no external reference point, so the readable form is the
    # comparison across tiers -- which only works now that the Upper50 export
    # has stopped duplicating All.
    _TIERS = ['All', 'Upper50', 'Upper25']
    _struct_rows = []
    for _tier in _TIERS:
        _m = p_matrices[_tier]
        _wr3 = _m['smoothed_wr3'].to_numpy(dtype=float) * 100
        _games = np.nan_to_num(_m['sample_n'].to_numpy(dtype=float))
        if np.triu(_games, 1).sum() == 0:
            continue
        _h = mst.hodge(np.nan_to_num((_wr3 - _wr3.T) / 2.0), _games)
        _struct_rows.append({
            'tier': _tier, 'cyclic': _h['cyclicity'] * 100,
            'transitive': (1 - _h['cyclicity']) * 100,
            'games': int(np.triu(_games, 1).sum()),
        })
    struct_df = _frame(_struct_rows, ['tier', 'cyclic', 'transitive', 'games'])

    # Triangles, families and twins describe the format as a whole, so they read
    # the All tier; the other two are the same decks on a fraction of the games.
    _wr3_all = p_matrices['All']['smoothed_wr3'].to_numpy(dtype=float) * 100
    _games_all = np.nan_to_num(p_matrices['All']['sample_n'].to_numpy(dtype=float))
    _edge_all = np.nan_to_num((_wr3_all - _wr3_all.T) / 2.0)
    _labels_core = [DECK_SHORT_NAMES[d] for d in CORE_DECKS]
    _has_games = np.triu(_games_all, 1).sum() > 0

    if _has_games:
        _h_all = mst.hodge(_edge_all, _games_all)
        triad_df = mst.triads(_h_all['residual'], _games_all, names=_labels_core,
                              min_weight=STRUCTURE_MIN_TRIAD_GAMES, top=10)
        _sim = mst.profile_similarity(_edge_all)
        _n_fam = int(min(STRUCTURE_FAMILIES, max(2, len(CORE_DECKS) - 1)))
        _fam, _order = mst.cluster_profiles(_sim, n_clusters=_n_fam)
        # Mutual pairs collapse to one row -- listing both directions spent half
        # the table restating the same pairing. One-way rows stay, because
        # "A's nearest is B but B prefers C" is a different fact.
        twin_df = mst.nearest_twins(_sim, names=_labels_core)
        twin_df['family'] = [int(_fam[i]) for i in twin_df['deck_idx']]
    else:
        triad_df = _frame([], ['a', 'b', 'c', 'curl', 'min_games'])
        twin_df = _frame([], ['deck', 'twin', 'r', 'mutual', 'deck_idx',
                              'twin_idx', 'family'])
        _sim, _order, _fam = np.zeros((0, 0)), [], []

    # Stacked to 100%: the question is the split, so the two parts share one bar.
    fig_structure = go.Figure()
    if len(struct_df):
        fig_structure.add_trace(go.Bar(
            y=struct_df['tier'], x=struct_df['transitive'], orientation='h',
            name='Explained by a power ranking', marker_color=UI.accent,
            hovertemplate='<b>%{y}</b><br>transitive %{x:.1f}%<extra></extra>'))
        fig_structure.add_trace(go.Bar(
            y=struct_df['tier'], x=struct_df['cyclic'], orientation='h',
            name='Rock-paper-scissors', marker_color=UI.amber,
            customdata=struct_df[['games']].values,
            hovertemplate=('<b>%{y}</b><br>cyclic %{x:.1f}%<br>'
                           '%{customdata[0]:,} games<extra></extra>')))
        fig_structure.update_layout(barmode='stack')
    _head = (f"{struct_df.iloc[0]['cyclic']:.0f}% of this format is not a power ranking"
             if len(struct_df) else 'No data for this platform')
    fig_structure.update_layout(
        title=(f'What a power ranking can and cannot explain - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">{_head}. '
               'A deck can be favoured against the field and still lose to specific '
               'decks; this is how much of the format works that way</span>'),
        xaxis_title='Share of matchup structure (%)', yaxis_title='',
        height=380, margin=dict(l=110, r=40, t=100, b=60),
        legend=dict(orientation='h', y=-0.3, x=0))

    fig_triads = go.Figure()
    if len(triad_df):
        _t = triad_df.head(8).iloc[::-1]
        fig_triads.add_trace(go.Bar(
            x=_t['curl'], y=[f'{r.a} > {r.b} > {r.c}' for r in _t.itertuples()],
            orientation='h', marker_color=UI.amber,
            customdata=_t[['min_games']].values,
            hovertemplate=('<b>%{y}</b><br>curl %{x:.2f} pp<br>'
                           'thinnest edge %{customdata[0]:,.0f} games<extra></extra>')))
    fig_triads.update_layout(
        title=(f'Strongest rock-paper-scissors triangles - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">Curl is how far the three '
               'matchups close into a loop instead of agreeing with any ranking. Every edge '
               f'rests on at least {STRUCTURE_MIN_TRIAD_GAMES} games</span>'),
        xaxis_title='Curl (percentage points)', yaxis_title='',
        height=460, margin=dict(l=300, r=50, t=100, b=60), bargap=0.3, showlegend=False)

    fig_families = go.Figure()
    if _has_games and len(_order):
        _names_ord = [_labels_core[i] for i in _order]
        fig_families.add_trace(go.Heatmap(
            z=_sim[np.ix_(_order, _order)], x=_names_ord, y=_names_ord,
            zmid=0, zmin=-1, zmax=1, colorscale=HEATMAP_SCALE,
            colorbar=dict(title=dict(text='profile<br>correlation', side='right'),
                          thickness=14),
            hovertemplate='<b>%{y}</b><br>vs <b>%{x}</b><br>r = %{z:.2f}<extra></extra>'))
    fig_families.update_layout(
        title=(f'Which decks answer the format the same way - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">Correlation between decks\' '
               'matchup profiles, each pair\'s own head-to-head excluded. Ordered by '
               'clustering, so families appear as blocks on the diagonal</span>'),
        height=680, margin=dict(l=190, r=90, t=100, b=170),
        xaxis=dict(tickangle=-45, tickfont=dict(size=9)),
        yaxis=dict(tickfont=dict(size=9), autorange='reversed'))

    fig_twins = go.Figure()
    if len(twin_df):
        _tw = twin_df.head(15)
        fig_twins.add_trace(go.Table(
            columnwidth=[32, 32, 12, 12, 12],
            header=dict(values=['Deck', 'Closest strategic twin', 'r', 'Mutual',
                                'Family'],
                        fill_color=UI.wash, align='left',
                        font=dict(size=12, color=UI.ink), height=30),
            cells=dict(
                values=[_tw['deck'], _tw['twin'], _tw['r'].round(3),
                        ['yes' if m else 'one-way' for m in _tw['mutual']],
                        _tw['family']],
                fill_color=[[UI.surface, UI.bg] * len(_tw)], align='left',
                font=dict(size=11), height=26)))
    fig_twins.update_layout(
        title=(f'Closest strategic twins - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">Nearest deck by matchup '
               'profile, using no name or card information. A high r means the two decks '
               'are near-interchangeable answers to the format. Mutual pairs are listed '
               'once; "one-way" means that deck\'s nearest does not pick it back</span>'),
        height=560, margin=dict(l=30, r=30, t=100, b=30))

    # 10) Skill as a curve --------------------------------------------------
    # The Skill Ceiling tab compares two overlapping slices (All vs Upper 25%),
    # which understates uncertainty because one contains the other. Placement
    # slices nest, and match counts are additive, so differencing successive
    # cuts gives genuinely disjoint bands and win rate becomes a curve over
    # finishing position rather than a two-point delta.
    _core_only = lambda d: d[d['deck1'].isin(CORE_DECKS) & d['deck2'].isin(CORE_DECKS)]
    _by_placement = {5: _core_only(skill_raw_by_platform[platform]['P5']),
                     10: _core_only(skill_raw_by_platform[platform]['P10']),
                     25: p_clean['Upper25'], 50: p_clean['Upper50']}
    _have_skill = all(len(v) > 0 for v in _by_placement.values()) and len(p_clean['All']) > 0

    if _have_skill:
        skill_bands = msk.disjoint_bands(_by_placement, p_clean['All'])
        band_df = msk.band_win_rates(skill_bands, CORE_DECKS,
                                     min_games=SKILL_MIN_BAND_GAMES)
        # Differencing must never produce a negative cell; if it does the slices
        # were not nested and every band below is meaningless.
        if len(band_df) and (band_df[['wins', 'losses', 'ties']] < 0).any().any():
            raise ValueError(f'{platform}: differencing placement slices produced '
                             f'negative counts -- slices are not nested')
        _blo, _bhi = msk.wilson(band_df['wins'], band_df['ties'], band_df['games'])
        band_df['wr3_%'] = band_df['wr3'] * 100
        band_df['ci_lo_%'], band_df['ci_hi_%'] = _blo * 100, _bhi * 100
        band_df['name'] = band_df['deck'].map(DECK_SHORT_NAMES)

        skill_curve_df = msk.skill_slopes(band_df, decks=CORE_DECKS,
                                          min_bands=SKILL_MIN_BANDS,
                                          min_games=SKILL_MIN_BAND_GAMES)
        skill_curve_df['name'] = skill_curve_df['deck'].map(DECK_SHORT_NAMES)
        skill_curve_df = skill_curve_df[
            (skill_curve_df['bands'] >= SKILL_REPORT_BANDS)
            & (skill_curve_df['games'] >= SKILL_REPORT_GAMES)].copy()
        _band_order = [b[0] for b in skill_bands]
    else:
        band_df = _frame([], ['deck', 'band', 'lo', 'hi', 'mid', 'wins', 'losses',
                              'ties', 'games', 'wr3', 'enough', 'wr3_%', 'ci_lo_%',
                              'ci_hi_%', 'name'])
        skill_curve_df = _frame([], ['deck', 'bands', 'games', 'slope_pp_per_decade',
                                     'intercept', 'se', 't', 'wr_field', 'wr_elite',
                                     'elite_minus_field', 'name'])
        _band_order = []

    # Curves for the extremes: lines over an ordered categorical axis.
    fig_skill_curve = go.Figure()
    if len(skill_curve_df):
        _pick = pd.concat([skill_curve_df.head(4), skill_curve_df.tail(4)])
        for _, row in _pick.iterrows():
            g = (band_df[band_df['deck'] == row['deck']].set_index('band')
                 .reindex(_band_order).dropna(subset=['wr3_%']))
            up = row['slope_pp_per_decade'] > 0
            fig_skill_curve.add_trace(go.Scatter(
                x=g.index, y=g['wr3_%'], mode='lines+markers', name=row['name'],
                line=dict(color=FAVORABLE_COLOR if up else UNFAVORABLE_COLOR, width=2),
                marker=dict(size=7, line=dict(color='white', width=1.5)),
                error_y=dict(type='data', symmetric=False,
                             array=g['ci_hi_%'] - g['wr3_%'],
                             arrayminus=g['wr3_%'] - g['ci_lo_%'],
                             thickness=1, width=0, color=UI.rule),
                customdata=np.c_[g['games']],
                hovertemplate=('<b>%{fullData.name}</b><br>band %{x}%<br>'
                               'win rate %{y:.1f}%<br>%{customdata[0]:,.0f} games'
                               '<extra></extra>')))
        fig_skill_curve.add_hline(y=neutral_wr, line_dash='dot',
                                  line_color=UI.rule, line_width=1.5)
    fig_skill_curve.update_layout(
        title=(f'Win rate by finishing position - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">Disjoint placement bands, '
               'best finishers on the left. Green climbs with pilot skill, red fades; '
               'bars are Wilson intervals</span>'),
        xaxis_title='Finishing position band (top %)',
        yaxis_title='3-point win rate (%)',
        height=560, margin=dict(l=70, r=170, t=100, b=60),
        legend=dict(font=dict(size=10)))

    fig_skill_slope = go.Figure()
    if len(skill_curve_df):
        _r = skill_curve_df.sort_values('slope_pp_per_decade').tail(16)
        fig_skill_slope.add_trace(go.Bar(
            x=_r['slope_pp_per_decade'], y=_r['name'], orientation='h',
            marker=dict(color=[FAVORABLE_COLOR if v > 0 else UNFAVORABLE_COLOR
                               for v in _r['slope_pp_per_decade']], line=dict(width=0)),
            error_x=dict(type='data', array=_r['se'], thickness=1, width=3,
                         color=UI.faint),
            customdata=np.column_stack([
                _r[['bands', 'games']].values.astype(object),
                [f'{v:+.1f}' for v in _r['elite_minus_field']], _r['t'].values.astype(object),
                [f'{v:+.2f}' for v in _r['slope_pp_per_decade']]]),
            hovertemplate=('<b>%{y}</b><br>%{customdata[4]} pp per decade<br>'
                           '%{customdata[0]:.0f} bands, %{customdata[1]:,.0f} games<br>'
                           'elite minus field %{customdata[2]} pp<br>'
                           't = %{customdata[3]:.1f}<extra></extra>')))
        fig_skill_slope.add_vline(x=0, line_color=UI.rule, line_width=1.5)
    fig_skill_slope.update_layout(
        title=(f'Win rate by finishing position: slope - {plat_label}'
               f'<br><span style="font-size:12px;color:{UI.muted}">Slope of win rate against '
               'finishing position, in points per decade (median player to top 5% is about '
               '1.5 decades). Bars are standard errors on a 3-5 point fit, so read them '
               'as rough</span>'),
        xaxis_title='Win rate slope (pp per decade of finishing position)',
        yaxis_title='', height=600, margin=dict(l=190, r=60, t=110, b=60),
        bargap=0.25, showlegend=False)

    # A platform can be real but too thin for the finest bands -- in-person
    # majors are a handful of championship weekends, so its top 5% is a few
    # dozen games. Say so, rather than rendering an empty pair of axes.
    if not len(skill_curve_df):
        _why = ('No data for this platform.' if not _have_skill else
                f'No deck here clears {SKILL_REPORT_BANDS} bands and '
                f'{SKILL_REPORT_GAMES:,} games.')
        for _f in (fig_skill_curve, fig_skill_slope):
            _f.add_annotation(
                text=(f'Not enough data to fit a skill curve.<br>{_why}<br>'
                      'The finest placement bands need more games than this '
                      'slice holds.'),
                showarrow=False, xref='paper', yref='paper', x=0.5, y=0.5,
                font=dict(size=13, color=UI.faint), align='center')
            _f.update_xaxes(visible=False)
            _f.update_yaxes(visible=False)

    return {
        'agg_df': agg_df, 'fig_wr': fig_wr,
        'consist_df': consist_df, 'fig_consist': fig_consist,
        'matchup_breakdown': matchup_breakdown,
        'consist_scatter_data': consist_scatter_data,
        'fig_skill_mu_heatmap': fig_skill_mu_heatmap,
        'skill_mu_df': skill_mu_df, 'fig_skill_mu_top': fig_skill_mu_top,
        'fig_heatmap': fig_heatmap, 'lop_df': lop_df, 'fig_lopsided': fig_lopsided,
        'swiss_df': swiss_df, 'fig_swiss': fig_swiss,
        'skill_df': skill_df, 'fig_bump': fig_bump, 'fig_skill': fig_skill,
        'neutral_wr': neutral_wr, 'shrink_k': shrink_k,
        'band_df': band_df, 'skill_curve_df': skill_curve_df,
        'fig_skill_curve': fig_skill_curve, 'fig_skill_slope': fig_skill_slope,
        'struct_df': struct_df, 'fig_structure': fig_structure,
        'triad_df': triad_df, 'fig_triads': fig_triads,
        'twin_df': twin_df, 'fig_families': fig_families, 'fig_twins': fig_twins,
        'meta_share': pd.DataFrame({'deck': CORE_DECKS, 'meta_share': p_pop_w * 100}),
    }


print(f'Computing analyses for {len(ACTIVE_PLATFORMS)} platform(s), adjusted and raw '
      f'(each includes a Swiss simulation; expect a wait)...')
analyses = {}
for _p in VIEWS:
    print(f'  computing {_p!r}...')
    analyses[_p] = compute_platform(_p)
for _p in EMPTY_PLATFORMS:
    print(f'  skipping {_p!r} (no entries)')
print(f'Done. analyses keys: {list(analyses)}')


In [ ]:
# === Sleepers: do any decks outside the core meta actually beat it? ===
# Ranks 26-50 by use rate get judged on one question only -- how do they do
# against the core meta? Each candidate's headline number is its win rate
# against the core decks it has actually played, weighted by how much of the
# field each of those decks represents, so beating three fringe decks doesn't
# read the same as beating the most-played ones.
#
# Three guards keep small samples honest:
#   shrinkage - applied once, to the aggregate, against the deck's total games
#               vs the core. Averaging per-matchup shrunk cells would shrink
#               twice and bury the deck's own evidence. (A raw view's K is 0.)
#   coverage  - the share of the core field the candidate has faced at all.
#   interval  - a Wilson interval from its real game count, carried onto the
#               field-weighted estimate, so a 6-game record can't look elite.
# The break-even line is the core decks' own median field win rate (computed the
# same way), never a hardcoded 50% -- with ties in the mix 50% isn't neutral.

SLEEPER_VERDICT_COLORS = {'Sleeper': FAVORABLE_COLOR, 'Promising': UI.amber, 'No edge': UI.faint}
SLEEPER_VERDICT_ORDER = ['Sleeper', 'Promising', 'No edge']


def _field_wr_vs_core(deck, wr_matrix, n_matrix, core_weights, neutral, k):
    """Core-usage-weighted 3-pt win rate of `deck` against the core meta.

    This is a claim about a deck, not about one matchup, so it is shrunk at that
    level: weight the **raw** matchup rates by core usage, then apply a single
    sample-size adjustment against the deck's *total* games vs the core. Averaging
    already-shrunk cells would shrink twice and throw the deck's own evidence
    away -- a deck with 436 games spread thinly across 25 matchups would collapse
    to the neutral even though its overall record is decisive.

    Returns (wr_percent, coverage, games). Coverage is the share of core meta the
    deck has a recorded matchup against -- weights are renormalised over exactly
    that slice, so the win rate answers "against the part of the field it has
    played".
    """
    num = den = 0.0
    games = 0.0
    for opp in CORE_DECKS:
        w = float(core_weights.get(opp, 0.0))
        if opp == deck or w <= 0:
            continue
        if deck not in wr_matrix.index or opp not in wr_matrix.columns:
            continue
        v, n = wr_matrix.loc[deck, opp], n_matrix.loc[deck, opp]
        if pd.isna(v) or pd.isna(n) or n <= 0:
            continue
        num += w * float(v) * 100.0
        den += w
        games += float(n)
    if den <= 0:
        return np.nan, 0.0, 0.0
    field_raw = num / den
    shrunk = (games * field_raw + k * neutral * 100.0) / (games + k)
    return shrunk, den, games


def compute_sleepers(platform):
    plat_label = PLATFORM_LABELS[platform]
    clean_all = clean_by_platform[platform]['All']
    mats = matrices_by_platform[platform]['All']
    # RAW matchup rates: the deck-level shrink below is applied once, to the
    # aggregate, so the per-matchup values must not already be shrunk.
    wr_m, n_m = mats['raw_wr3'], mats['sample_n']
    _shrink = shrinkage_by_platform[platform]['All']['wr3']
    neutral, shrink_k = float(_shrink['mean']), float(_shrink['k'])

    # Field weights: how much of the core meta each core deck accounts for on
    # THIS platform (the split itself is global, the weighting is per platform).
    core_rows = clean_all[clean_all['deck1'].isin(CORE_DECKS) & clean_all['deck2'].isin(CORE_DECKS)]
    core_games = core_rows.groupby('deck1')['actual_total'].sum().reindex(CORE_DECKS, fill_value=0)
    core_total = float(core_games.sum())
    core_weights = (core_games / core_total).to_dict() if core_total > 0 else {}

    # The bar to clear: what the core decks themselves manage against the core.
    core_field_wr = {d: _field_wr_vs_core(d, wr_m, n_m, core_weights, neutral, shrink_k)[0]
                     for d in CORE_DECKS}
    core_ranked = sorted((v for v in core_field_wr.values() if not pd.isna(v)), reverse=True)
    core_median = float(np.median(core_ranked)) if core_ranked else 50.0

    rows = []
    sleeper_breakdown = {}
    for deck in SLEEPER_DECKS:
        sub = clean_all[(clean_all['deck1'] == deck)
                        & clean_all['deck2'].isin(CORE_DECKS)
                        & (clean_all['actual_total'] > 0)]
        if len(sub) == 0:
            continue
        wins = float(sub['wins'].sum()); losses = float(sub['losses'].sum())
        ties = float(sub['ties'].sum()); games = wins + losses + ties
        if games <= 0:
            continue
        wr_core, coverage, _cov_games = _field_wr_vs_core(deck, wr_m, n_m, core_weights,
                                                          neutral, shrink_k)
        if pd.isna(wr_core):
            continue
        # Aggregate 3-pt record over the games actually played, and the sampling
        # error that record implies. "Effective wins" = (3W+T)/3 makes the Wilson
        # centre the 3-pt rate while the trial count stays the real game count.
        wr3_raw = (3 * wins + ties) / (3 * games) * 100
        _, ci_lo_raw, ci_hi_raw = wilson_ci((3 * wins + ties) / 3.0, games)
        # Carry that error onto the field-weighted estimate: ties move the point
        # estimate, they don't change how much data there is.
        ci_lo = wr_core - (wr3_raw - ci_lo_raw * 100)
        ci_hi = wr_core + (ci_hi_raw * 100 - wr3_raw)

        if ci_lo > core_median:
            verdict = 'Sleeper'
        elif wr_core > core_median:
            verdict = 'Promising'
        else:
            verdict = 'No edge'

        # Per-matchup breakdown, and the favorable/unfavorable decomposition of
        # the headline number. Same arithmetic and the same dict keys as the
        # Consistency tab (cell 5), with one substitution: the line separating a
        # favorable matchup from an unfavorable one is the format's tie-aware
        # break-even, not this deck's own mirror record. A fringe deck's mirror
        # is a handful of games and makes a terrible baseline, and the neutral is
        # already what the headline win rate is shrunk toward, so net impact
        # decomposes that number against the same reference.
        covered = []
        adj_m = mats['smoothed_wr3']  # per-matchup claim -> per-matchup shrinkage
        baseline = neutral * 100.0
        fav_impact = unfav_impact = 0.0
        w_total = 0.0
        for opp in CORE_DECKS:
            if opp == deck or deck not in adj_m.index or opp not in adj_m.columns:
                continue
            v, n = adj_m.loc[deck, opp], n_m.loc[deck, opp]
            if pd.isna(v) or pd.isna(n) or n <= 0:
                continue
            wr = float(v) * 100.0
            raw = wr_m.loc[deck, opp] if deck in wr_m.index and opp in wr_m.columns else np.nan
            # Field weight: how much of the core meta this opponent accounts for.
            # An opponent with no share contributes nothing, exactly as in the
            # Consistency tab -- but it still counts as a matchup played.
            w = float(core_weights.get(opp, 0.0))
            w_total += w
            if wr > baseline:
                fav_impact += w * (wr - baseline)
            elif wr < baseline:
                unfav_impact += w * (baseline - wr)
            covered.append({
                'opponent': opp, 'opp_short': DECK_SHORT_NAMES.get(opp, opp),
                'win_rate': round(wr, 1),
                'raw_wr': None if pd.isna(raw) else round(float(raw) * 100.0, 1),
                'opp_pop': round(w * 100.0, 2), 'games': int(n),
                '_wr': wr, '_w': w,
            })
        if w_total > 0:
            fav_impact /= w_total
            unfav_impact /= w_total
        for mu in covered:
            share = (mu.pop('_w') / w_total) if w_total > 0 else 0.0
            mu['net_impact'] = round(share * (mu.pop('_wr') - baseline), 3)
        covered.sort(key=lambda m: m['net_impact'])
        by_wr = sorted(covered, key=lambda m: -m['win_rate'])

        rows.append({
            'deck': deck, 'short_name': DECK_SHORT_NAMES[deck],
            'use_rate': float(DECK_USE_RATE.get(deck, 0.0)),
            'games': int(games), 'wins': int(wins), 'losses': int(losses), 'ties': int(ties),
            'n_matchups': len(covered),
            'wr_vs_core': wr_core, 'wr3_raw': wr3_raw,
            'ci_lo': ci_lo, 'ci_hi': ci_hi,
            'coverage': coverage * 100.0,
            # Conservative edge: clear the core's median even at the bottom of the
            # interval, scaled by how much of the field the claim rests on.
            'sleeper_score': (ci_lo - core_median) * coverage,
            'would_rank': sum(1 for v in core_ranked if v > wr_core) + 1,
            'verdict': verdict,
            'fav_impact': fav_impact, 'unfav_impact': unfav_impact,
            'net_impact': fav_impact - unfav_impact,
            'best_opp': by_wr[0]['opponent'] if by_wr else '',
            'best_wr': by_wr[0]['win_rate'] if by_wr else np.nan,
            'worst_opp': by_wr[-1]['opponent'] if by_wr else '',
            'worst_wr': by_wr[-1]['win_rate'] if by_wr else np.nan,
        })
        sleeper_breakdown[deck] = covered

    sleeper_df = _frame(rows, ['deck', 'short_name', 'use_rate', 'games', 'wins', 'losses',
                               'ties', 'n_matchups', 'wr_vs_core', 'wr3_raw', 'ci_lo', 'ci_hi',
                               'coverage', 'sleeper_score', 'would_rank', 'verdict',
                               'fav_impact', 'unfav_impact', 'net_impact',
                               'best_opp', 'best_wr', 'worst_opp', 'worst_wr'])
    if len(sleeper_df):
        sleeper_df = sleeper_df.sort_values('sleeper_score', ascending=False).reset_index(drop=True)

    # --- scatter: sample size vs field-weighted win rate ---------------------
    fig_scatter = go.Figure()
    if len(sleeper_df):
        for verdict in SLEEPER_VERDICT_ORDER:
            vd = sleeper_df[sleeper_df['verdict'] == verdict]
            if len(vd) == 0:
                continue
            fig_scatter.add_trace(go.Scatter(
                x=vd['games'], y=vd['wr_vs_core'],
                mode='markers+text', name=verdict,
                text=vd['short_name'], textposition='top center',
                textfont=dict(size=9, color=UI.text2),
                marker=dict(size=(8 + vd['coverage'] / 100 * 14), color=SLEEPER_VERDICT_COLORS[verdict],
                            line=dict(width=2, color='white'), opacity=0.9),
                error_y=dict(type='data', symmetric=False,
                             array=(vd['ci_hi'] - vd['wr_vs_core']).values,
                             arrayminus=(vd['wr_vs_core'] - vd['ci_lo']).values,
                             thickness=1.5, width=3, color=UI.faint),
                hovertemplate=('<b>%{customdata[0]}</b><br>'
                               'WR vs core meta: %{y:.1f}%<br>'
                               '95% interval: %{customdata[1]:.1f}% - %{customdata[2]:.1f}%<br>'
                               'Games vs core: %{x:,} across %{customdata[3]} matchups<br>'
                               'Field coverage: %{customdata[4]:.0f}%<br>'
                               'Use rate: %{customdata[5]:.2f}%<br>'
                               'Would rank #%{customdata[6]} among the core<br>'
                               'Favorable / Unfavorable impact: '
                               '%{customdata[7]:.2f} / %{customdata[8]:.2f}<br>'
                               'Net impact: %{customdata[9]}<br>'
                               '<i>Click for matchup breakdown</i>'
                               '<extra></extra>'),
                # Net impact is pre-formatted: Plotly 2.x rejects '+.2f' in a
                # hovertemplate and silently prints the raw, unrounded value.
                customdata=np.column_stack([
                    vd[['deck', 'ci_lo', 'ci_hi', 'n_matchups', 'coverage',
                        'use_rate', 'would_rank', 'fav_impact', 'unfav_impact']].values,
                    [f'{v:+.2f}' for v in vd['net_impact']]]),
            ))
        fig_scatter.add_hline(y=core_median, line_dash='dash', line_color=UI.faint, line_width=1.5,
                              annotation_text=f'Core median ({core_median:.1f}%)',
                              annotation_font=dict(size=10, color=UI.muted))
    else:
        fig_scatter.add_annotation(text='No decks outside the core meta have recorded games.',
                                   xref='paper', yref='paper', x=0.5, y=0.5,
                                   showarrow=False, font=dict(size=13, color=UI.faint))
    fig_scatter.update_layout(
        title=f'Sleeper Candidates vs the Core Meta - {plat_label}',
        xaxis_title='Games played against core decks',
        yaxis_title='Win Rate vs Core Meta (3-pt, usage-weighted) %',
        height=620, margin=dict(l=80, r=30, t=60, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
    )

    # --- table ---------------------------------------------------------------
    badge_bg = {'Sleeper': UI.fav_bg, 'Promising': UI.warn_bg, 'No edge': UI.wash}
    badge_fg = {'Sleeper': UI.fav_ink, 'Promising': UI.warn, 'No edge': UI.muted}
    n_rows = len(sleeper_df)
    stripe = [[UI.bg, UI.surface][i % 2] for i in range(n_rows)]
    fill = [[badge_bg.get(v, UI.bg) for v in sleeper_df['verdict']]] + [stripe] * 10
    # Net impact is the one column whose sign carries meaning, so it gets the
    # same red/green language as the breakdown bars.
    net_colors = [FAVORABLE_COLOR if v >= 0 else UNFAVORABLE_COLOR
                  for v in sleeper_df['net_impact']]
    font_colors = [[badge_fg.get(v, UI.text) for v in sleeper_df['verdict']]] + \
        [[UI.text] * n_rows] * 9 + [net_colors]
    fig_table = go.Figure(data=[go.Table(
        header=dict(values=['Verdict', 'Deck', 'Use %', 'WR vs Core', '95% interval',
                            'Games', 'Coverage', 'Would rank',
                            'Favorable', 'Unfavorable', 'Net impact'],
                    fill_color=UI.ink, font=dict(color='white', size=12, family=FONT_STACK),
                    align='center', line=dict(width=0), height=32),
        cells=dict(values=[
            [f'<b>{v}</b>' for v in sleeper_df['verdict']],
            [f'<b>{s}</b>' for s in sleeper_df['short_name']],
            sleeper_df['use_rate'].round(2),
            [f'{v:.1f}%' for v in sleeper_df['wr_vs_core']],
            [f'{lo:.1f} - {hi:.1f}' for lo, hi in zip(sleeper_df['ci_lo'], sleeper_df['ci_hi'])],
            [f'{g:,} / {m} MU' for g, m in zip(sleeper_df['games'], sleeper_df['n_matchups'])],
            [f'{c:.0f}%' for c in sleeper_df['coverage']],
            [f'#{r}' for r in sleeper_df['would_rank']],
            [f'{v:.2f}' for v in sleeper_df['fav_impact']],
            [f'{v:.2f}' for v in sleeper_df['unfav_impact']],
            [f'<b>{v:+.2f}</b>' for v in sleeper_df['net_impact']],
        ], fill_color=fill, font=dict(color=font_colors, size=11, family=FONT_STACK),
           align='center', line=dict(width=0.5, color=UI.border), height=30),
        columnwidth=[1.0, 1.7, 0.7, 1.0, 1.1, 1.1, 0.85, 0.85, 0.95, 1.05, 0.95],
    )])
    fig_table.update_layout(
        title=(f'Sleeper Scorecard - {plat_label}<br>'
               f'<span style="font-size:12px;color:{UI.muted}">'
               f'Core median {core_median:.1f}% - a Sleeper clears it even at the bottom of its '
               f'interval. Favorable/Unfavorable split the matchups at the format break-even of '
               f'{neutral * 100:.1f}%, weighted by core meta share.</span>'),
        height=max(340, n_rows * 32 + 140), margin=dict(l=10, r=10, t=90, b=10),
    )

    return {'sleeper_df': sleeper_df, 'fig_sleeper': fig_scatter,
            'fig_sleeper_table': fig_table, 'core_median_wr': core_median,
            'sleeper_breakdown': sleeper_breakdown,
            'sleeper_baseline': neutral * 100.0}


print('Computing sleeper analysis...')
for _p in VIEWS:
    analyses[_p].update(compute_sleepers(_p))
    _sdf = analyses[_p]['sleeper_df']
    _n_sleep = int((_sdf['verdict'] == 'Sleeper').sum()) if len(_sdf) else 0
    print(f"  {_p}: {len(_sdf)} candidates scored, {_n_sleep} clear the core median "
          f"(core median {analyses[_p]['core_median_wr']:.1f}%)")


In [ ]:
# === Mispriced decks: bootstrapped Nash equilibrium ===
# Win rate says a deck is good. It does not say whether the field is playing it
# as much as it deserves -- that is where an edge actually lives. Deck choice is
# a symmetric two-player game, and its Nash equilibrium is the field no deck can
# profitably deviate against. Comparing that field with the one people actually
# register exposes decks the field under- and over-rates.
#
# Two guards, both learned in the experiments notebook:
#   bootstrap - a single linear-program solution is a *vertex*, happy to hang a
#               large weight on a deck whose matchup row is mostly prior. Every
#               figure below is a bootstrap median / interval over NASH_BOOT
#               posterior draws of the matchup matrix, never the raw LP weight.
#   universe  - decks with at least NASH_MIN_GAMES games on this platform, drawn
#               from all TOP_N rather than the core 25: the mispricings worth
#               finding are, almost by definition, decks the field underplays.
#
# "Actual share" is Trainer Hill's share of *players* for the same window and
# sources as the matchups (written by the fetch), not games played -- decks that
# convert play more matches per entry, so games-share over-rates them.
import meta_core as mc
import meta_game as mg

NASH_VERDICT_COLORS = {'Underplayed': FAVORABLE_COLOR, 'Overplayed': UNFAVORABLE_COLOR,
                       'No clear mispricing': UI.faint}
NASH_VERDICT_ORDER = ['Underplayed', 'Overplayed', 'No clear mispricing']
NASH_COLUMNS = ['deck', 'short_name', 'games', 'share_pct', 'nash_point_pct', 'nash_median_pct',
                'q05', 'q95', 'in_support_pct', 'p_under', 'p_over', 'gap_pp', 'edge_pp',
                'edge_q05', 'edge_q95',
                'best_exploit_pct', 'verdict']


def compute_nash(platform, n_boot=NASH_BOOT, seed=42):
    plat_label = PLATFORM_LABELS[platform]
    raw_all = raw_by_platform[platform]['All']
    clean_all = clean_by_platform[platform]['All']
    neutral = float(shrinkage_by_platform[platform]['All']['wr3']['mean'])
    adjust = is_adjusted(platform)     # a raw view's point matrix is already unshrunk
    shares = _th_csv.read_shares(DATA_DIR, base_platform(platform))

    games = clean_all.groupby('deck1')['actual_total'].sum()
    decks = sorted((d for d in analysis_decks
                    if games.get(d, 0) >= NASH_MIN_GAMES and shares.get(d, 0) > 0),
                   key=lambda d: -shares[d])
    empty = {'nash_df': _frame([], NASH_COLUMNS), 'nash_exploit': None}

    def _empty_fig(title, msg):
        f = go.Figure()
        f.add_annotation(text=msg, xref='paper', yref='paper', x=0.5, y=0.5,
                         showarrow=False, font=dict(size=13, color=UI.faint))
        f.update_layout(title=title, height=360, xaxis=dict(visible=False), yaxis=dict(visible=False))
        return f

    if len(decks) < 3:
        msg = (f'Too few decks with {NASH_MIN_GAMES}+ games on this platform '
               f'({len(decks)}) to solve a deck-choice game.')
        return {**empty,
                'fig_nash': _empty_fig(f'Mispriced Decks - {plat_label}', msg),
                'fig_nash_edge': _empty_fig(f'Edge vs the Current Field - {plat_label}', msg),
                'fig_nash_table': _empty_fig(f'Mispricing Scorecard - {plat_label}', msg)}

    n = len(decks)
    s = np.array([shares[d] for d in decks], dtype=float)
    s = s / s.sum()   # the field these decks make up, renormalised over the universe

    W = mc.payoff_matrix(raw_all, decks, neutral)
    point = mg.nash(W)
    edge_point = mg.edge_matrix(W) @ s

    pairs = mc.pair_counts(clean_all, decks)
    rng = np.random.default_rng(seed)
    boot_w = np.empty((n_boot, n))
    boot_edge = np.empty((n_boot, n))
    for b in range(n_boot):
        Wb = mc.bootstrap_payoff(pairs, decks, neutral, rng, adjust=adjust)
        boot_w[b] = mg.nash(Wb)['weights']
        boot_edge[b] = mg.edge_matrix(Wb) @ s

    boot_expl = boot_edge.max(axis=1)
    best_counts = np.bincount(boot_edge.argmax(axis=1), minlength=n)
    q05, q50, q95 = np.percentile(boot_w, [5, 50, 95], axis=0) * 100
    e05, e95 = np.percentile(boot_edge, [5, 95], axis=0) * 100
    share_pct = s * 100

    # How often a draw's equilibrium wants more (less) of each deck than the field
    # plays. A one-sided probability rather than "share outside the 90% range":
    # the equilibrium is sparse, so most decks sit at exactly zero weight in some
    # draws, which pins the 5th percentile to 0 and would make "underplayed"
    # impossible for any deck missing from even 6% of draws.
    p_under = (boot_w * 100 > share_pct).mean(axis=0) * 100
    p_over = (boot_w * 100 < share_pct).mean(axis=0) * 100

    rows = []
    for i, d in enumerate(decks):
        # Underplayed: 90% of draws want more of it than the field plays.
        # Overplayed: 90% of draws want less -- AND it loses to the field it is
        # popular in. Most decks carry no equilibrium weight, so without the
        # second condition every fringe deck would read as "overplayed" by its
        # own sliver of share, which is true and useless.
        if p_under[i] >= NASH_CONFIDENCE:
            verdict = 'Underplayed'
        elif p_over[i] >= NASH_CONFIDENCE and e95[i] < 0:
            verdict = 'Overplayed'
        else:
            verdict = 'No clear mispricing'
        rows.append({
            'deck': d, 'short_name': DECK_SHORT_NAMES.get(d, d), 'games': int(games.get(d, 0)),
            'share_pct': share_pct[i], 'nash_point_pct': point['weights'][i] * 100,
            'nash_median_pct': q50[i], 'q05': q05[i], 'q95': q95[i],
            'in_support_pct': float((boot_w[:, i] > 1e-6).mean() * 100),
            'p_under': p_under[i], 'p_over': p_over[i],
            'gap_pp': q50[i] - share_pct[i],
            'edge_pp': edge_point[i] * 100, 'edge_q05': e05[i], 'edge_q95': e95[i],
            'best_exploit_pct': best_counts[i] / n_boot * 100,
            'verdict': verdict,
        })
    nash_df = _frame(rows, NASH_COLUMNS).sort_values('gap_pp', ascending=False).reset_index(drop=True)

    best = int(np.argmax(edge_point))
    exploit = {
        'value_pp': float(edge_point.max() * 100),
        'q05': float(np.percentile(boot_expl, 5) * 100),
        'q95': float(np.percentile(boot_expl, 95) * 100),
        'best_deck': decks[best], 'best_freq': float(best_counts[best] / n_boot * 100),
        'n_decks': n, 'n_boot': n_boot, 'field_coverage': float(sum(shares[d] for d in decks)),
    }

    # --- scatter: equilibrium weight vs actual share --------------------------
    fig = go.Figure()
    lim = float(max(nash_df['share_pct'].max(), nash_df['q95'].max())) * 1.08
    fig.add_trace(go.Scatter(x=[0, lim], y=[0, lim], mode='lines', hoverinfo='skip',
                             line=dict(color=UI.rule, dash='dash', width=1.5), showlegend=False))
    for verdict in NASH_VERDICT_ORDER:
        vd = nash_df[nash_df['verdict'] == verdict]
        if len(vd) == 0:
            continue
        labelled = (vd['verdict'] != 'No clear mispricing') | (vd['in_support_pct'] >= 50)
        fig.add_trace(go.Scatter(
            x=vd['share_pct'], y=vd['nash_median_pct'], mode='markers+text', name=verdict,
            text=[sn if lab else '' for sn, lab in zip(vd['short_name'], labelled)],
            textposition='top center', textfont=dict(size=9, color=UI.text2),
            marker=dict(size=7 + vd['in_support_pct'] / 100 * 16, color=NASH_VERDICT_COLORS[verdict],
                        line=dict(width=2, color='white'), opacity=0.9),
            error_y=dict(type='data', symmetric=False,
                         array=(vd['q95'] - vd['nash_median_pct']).values,
                         arrayminus=(vd['nash_median_pct'] - vd['q05']).values,
                         thickness=1.2, width=3, color=UI.faint),
            # Signed numbers are pre-formatted: Plotly 2.x rejects '+.2f' in a
            # hovertemplate and silently prints the raw, unrounded value.
            customdata=np.column_stack([
                vd[['deck', 'short_name', 'q05', 'q95', 'in_support_pct']].values,
                [f'{v:+.2f}' for v in vd['gap_pp']], [f'{v:+.2f}' for v in vd['edge_pp']],
                vd[['games', 'p_under']].values]),
            hovertemplate=('<b>%{customdata[1]}</b><br>'
                           'Actual share: %{x:.2f}%<br>'
                           'Equilibrium weight (median): %{y:.2f}%<br>'
                           '90% range: %{customdata[2]:.2f}% - %{customdata[3]:.2f}%<br>'
                           'In equilibrium in %{customdata[4]:.0f}% of draws<br>'
                           'Wants more than actual share in %{customdata[8]:.0f}% of draws<br>'
                           'Gap (median - share): %{customdata[5]}pp<br>'
                           'Edge vs current field: %{customdata[6]}pp<br>'
                           'Games: %{customdata[7]:,}<extra></extra>'),
        ))
    n_under = int((nash_df['verdict'] == 'Underplayed').sum())
    n_over = int((nash_df['verdict'] == 'Overplayed').sum())
    fig.update_layout(
        title=(f'Mispriced Decks - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               f'{n} decks with {NASH_MIN_GAMES}+ games ({exploit["field_coverage"]:.0f}% of players). '
               f'Above the dashed line the equilibrium wants more of a deck than the field plays. '
               f'{n_under} underplayed, {n_over} overplayed; bubble size = how often the deck is in '
               f'the equilibrium at all.</span>'),
        xaxis=dict(title='Actual share of players (%)', range=[0, lim]),
        yaxis=dict(title=f'Equilibrium weight, median of {n_boot} draws (%)', range=[0, lim]),
        height=640, margin=dict(l=80, r=30, t=90, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    )

    # --- bars: each deck's edge against the field as it stands ----------------
    ed = nash_df.sort_values('edge_pp', ascending=True)
    fig_edge = go.Figure(go.Bar(
        y=ed['short_name'], x=ed['edge_pp'], orientation='h',
        marker=dict(color=[NASH_VERDICT_COLORS[v] for v in ed['verdict']], line=dict(width=0)),
        error_x=dict(type='data', symmetric=False, array=(ed['edge_q95'] - ed['edge_pp']).values,
                     arrayminus=(ed['edge_pp'] - ed['edge_q05']).values,
                     thickness=1.2, width=2, color=UI.muted),
        customdata=np.column_stack([
            ed['deck'].values, [f'{v:+.2f}' for v in ed['edge_q05']], [f'{v:+.2f}' for v in ed['edge_q95']],
            ed[['best_exploit_pct', 'share_pct']].values, [f'{v:+.2f}' for v in ed['edge_pp']]]),
        hovertemplate=('<b>%{y}</b><br>Edge vs current field: %{customdata[5]}pp<br>'
                       '90% range: %{customdata[1]} to %{customdata[2]}pp<br>'
                       'Best counter-pick in %{customdata[3]:.0f}% of draws<br>'
                       'Share: %{customdata[4]:.2f}%<extra></extra>'),
    ))
    fig_edge.add_vline(x=0, line_color=UI.faint, line_width=1)
    fig_edge.update_layout(
        title=(f'Edge vs the Current Field - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               f'Field exploitability {exploit["value_pp"]:.2f}pp (90% range {exploit["q05"]:.2f} - '
               f'{exploit["q95"]:.2f}): the best counter-pick, {DECK_SHORT_NAMES.get(exploit["best_deck"], exploit["best_deck"])}, '
               f'beats this field by that much. It is the best counter-pick in '
               f'{exploit["best_freq"]:.0f}% of draws.</span>'),
        xaxis_title='Tie-free edge against the field as registered (pp)', yaxis_title='',
        height=max(520, n * 24 + 140), margin=dict(l=130, r=30, t=90, b=60), showlegend=False,
    )

    # --- table --------------------------------------------------------------
    badge_bg = {'Underplayed': UI.fav_bg, 'Overplayed': UI.unfav_bg, 'No clear mispricing': UI.wash}
    badge_fg = {'Underplayed': UI.fav_ink, 'Overplayed': UI.unfav_ink, 'No clear mispricing': UI.muted}
    n_rows = len(nash_df)
    stripe = [[UI.bg, UI.surface][i % 2] for i in range(n_rows)]
    gap_colors = [FAVORABLE_COLOR if v >= 0 else UNFAVORABLE_COLOR for v in nash_df['gap_pp']]
    edge_colors = [FAVORABLE_COLOR if v >= 0 else UNFAVORABLE_COLOR for v in nash_df['edge_pp']]
    fig_table = go.Figure(data=[go.Table(
        header=dict(values=['Verdict', 'Deck', 'Share %', 'Equilibrium %', '90% range',
                            'In equilibrium', 'Gap (pp)', 'Edge vs field (pp)', 'Games'],
                    fill_color=UI.ink, font=dict(color='white', size=12, family=FONT_STACK),
                    align='center', line=dict(width=0), height=32),
        cells=dict(values=[
            [f'<b>{v}</b>' for v in nash_df['verdict']],
            [f'<b>{sn}</b>' for sn in nash_df['short_name']],
            nash_df['share_pct'].round(2),
            nash_df['nash_median_pct'].round(2),
            [f'{lo:.1f} - {hi:.1f}' for lo, hi in zip(nash_df['q05'], nash_df['q95'])],
            [f'{v:.0f}% of draws' for v in nash_df['in_support_pct']],
            [f'<b>{v:+.2f}</b>' for v in nash_df['gap_pp']],
            [f'{v:+.2f}' for v in nash_df['edge_pp']],
            [f'{g:,}' for g in nash_df['games']],
        ], fill_color=[[badge_bg[v] for v in nash_df['verdict']]] + [stripe] * 8,
           font=dict(color=[[badge_fg[v] for v in nash_df['verdict']]] + [[UI.text] * n_rows] * 5
                     + [gap_colors, edge_colors, [UI.text] * n_rows], size=11, family=FONT_STACK),
           align='center', line=dict(width=0.5, color=UI.border), height=30),
        columnwidth=[1.4, 1.6, 0.8, 1.0, 1.0, 1.1, 0.8, 1.1, 0.8],
    )])
    fig_table.update_layout(
        title=(f'Mispricing Scorecard - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               f'Underplayed: at least {NASH_CONFIDENCE:.0f}% of draws want more of the deck than the field '
               f'plays. Overplayed: at least {NASH_CONFIDENCE:.0f}% want less, and its edge against the '
               f'field is negative across the 90% range. Sorted by gap.</span>'),
        height=max(360, n_rows * 32 + 140), margin=dict(l=10, r=10, t=90, b=10),
    )

    return {'nash_df': nash_df, 'nash_exploit': exploit,
            'fig_nash': fig, 'fig_nash_edge': fig_edge, 'fig_nash_table': fig_table}


print(f'Computing mispriced decks ({NASH_BOOT} bootstrap draws per platform)...')
for _p in VIEWS:
    analyses[_p].update(compute_nash(_p))
    _nd, _ex = analyses[_p]['nash_df'], analyses[_p]['nash_exploit']
    if _ex is None:
        print(f'  {_p}: too few decks to solve')
        continue
    print(f"  {_p}: {_ex['n_decks']} decks, exploitability {_ex['value_pp']:.2f}pp "
          f"[{_ex['q05']:.2f}, {_ex['q95']:.2f}]; "
          f"underplayed {list(_nd.loc[_nd['verdict'] == 'Underplayed', 'short_name'])}, "
          f"overplayed {list(_nd.loc[_nd['verdict'] == 'Overplayed', 'short_name'])}")


In [ ]:
# === Pressure slices: top cut and late rounds ===
# Two questions with the same shape: does a deck win more or less once the stakes
# rise -- in the top cut, and in the last Swiss rounds -- than its matchups say
# it should?
#
# Both slices change who you play as well as how much is at stake, so a deck is
# never compared with its own overall win rate. Its games in the slice are priced,
# opponent by opponent, at that matchup's rate in the complementary slice, and
# only the leftover is reported (scripts/meta_shift.py). A deck that "chokes" in
# the cut because the cut is full of its bad matchups shows no shift here.
#
# The rate is wins over DECIDED games, not the 3-point rate used elsewhere. Late
# Swiss rounds are full of intentional draws into the cut (6.7% of online
# late-round games are ties, against ~1% overall) and top cut treats draws
# differently again, so a tie does not mean the same thing on both sides of
# either comparison.
#
# Late rounds are the selection-bias-free skill measure. The placement tiers
# (Upper 50% / 25%) select players on final standings -- which include the very
# games being measured. Reaching round LATE_ROUND is decided by earlier games,
# and the event set is held fixed (LATE_MIN_PLAYERS+ events on both sides).
import meta_shift as mshift

SHIFT_SIG_COLORS = {'up': FAVORABLE_COLOR, 'down': UNFAVORABLE_COLOR, 'flat': UI.faint}


def _sig(z):
    return 'up' if z >= 1.96 else ('down' if z <= -1.96 else 'flat')


def _shift_figures(sd, plat_label, title, part_label, ref_label, min_games, x_note):
    """Dumbbell (reference -> actual, with the mix expectation) + adjusted-shift bars."""
    shown = sd[sd['enough']].sort_values('adj_delta').copy()
    if shown.empty:
        figs = []
        for t in (title, f'{title}: adjusted shift'):
            f = go.Figure()
            f.add_annotation(text=(f'No deck has {min_games}+ decided games in the {part_label.lower()} '
                                   f'on this platform.'),
                             xref='paper', yref='paper', x=0.5, y=0.5, showarrow=False,
                             font=dict(size=13, color=UI.faint))
            f.update_layout(title=f'{t} - {plat_label}', height=320,
                            xaxis=dict(visible=False), yaxis=dict(visible=False))
            figs.append(f)
        return figs

    shown['short_name'] = shown['deck'].map(lambda d: DECK_SHORT_NAMES.get(d, d))
    shown['sig'] = shown['z'].map(_sig)
    n = len(shown)
    # Signed shifts are pre-formatted: Plotly 2.x rejects '+.1f' in a hovertemplate
    # and silently prints the raw, unrounded value.
    custom = np.column_stack([
        shown[['deck', 'games', 'actual', 'expected', 'ref_wr']].values,
        [f'{v:+.1f}' for v in shown['raw_delta']], [f'{v:+.1f}' for v in shown['adj_delta']],
        shown[['se', 'fallback_share']].values])
    hover = ('<b>%{y}</b><br>'
             f'{part_label}: ' + '%{customdata[2]:.1f}% over %{customdata[1]:,} decided games<br>'
             f'Expected from its {part_label.lower()} opponents: ' + '%{customdata[3]:.1f}%<br>'
             f'{ref_label} overall: ' + '%{customdata[4]:.1f}%<br>'
             'Naive shift: %{customdata[5]}pp<br>'
             'Adjusted shift: %{customdata[6]}pp (se %{customdata[7]:.1f})<br>'
             'Matchups priced at deck average: %{customdata[8]:.0%}<extra></extra>')

    fig = go.Figure()
    for _, r in shown.iterrows():
        fig.add_trace(go.Scatter(x=[r['ref_wr'], r['actual']], y=[r['short_name']] * 2, mode='lines',
                                 line=dict(color=UI.rule, width=3), hoverinfo='skip', showlegend=False))
    fig.add_trace(go.Scatter(x=shown['ref_wr'], y=shown['short_name'], mode='markers', name=ref_label,
                             marker=dict(size=10, color='white', line=dict(width=2, color=UI.muted)),
                             customdata=custom, hovertemplate=hover))
    fig.add_trace(go.Scatter(x=shown['expected'], y=shown['short_name'], mode='markers',
                             name=f'Expected from {part_label.lower()} opponents',
                             marker=dict(symbol='line-ns-open', size=16, color=UI.ink, line=dict(width=2)),
                             customdata=custom, hovertemplate=hover))
    fig.add_trace(go.Scatter(x=shown['actual'], y=shown['short_name'], mode='markers', name=part_label,
                             marker=dict(size=11, color=[SHIFT_SIG_COLORS[s] for s in shown['sig']],
                                         line=dict(width=1.5, color='white')),
                             customdata=custom, hovertemplate=hover))
    fig.update_layout(
        title=(f'{title} - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               f'Hollow: {ref_label.lower()} win rate. Bar tick: what its {part_label.lower()} opponents '
               f'predict. Dot: what it actually did (coloured when the adjusted shift is significant). '
               f'Decks with {min_games}+ decided games.</span>'),
        xaxis_title='Win rate, decided games (%)', yaxis_title='',
        height=max(420, n * 26 + 150), margin=dict(l=150, r=30, t=90, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.0, xanchor='right', x=1),
    )

    fig_adj = go.Figure(go.Bar(
        y=shown['short_name'], x=shown['adj_delta'], orientation='h',
        marker=dict(color=[SHIFT_SIG_COLORS[s] for s in shown['sig']], line=dict(width=0)),
        error_x=dict(type='data', array=(1.96 * shown['se']).values, thickness=1.2, width=2,
                     color=UI.muted),
        customdata=custom, hovertemplate=hover,
    ))
    fig_adj.add_vline(x=0, line_color=UI.faint, line_width=1)
    n_up, n_down = int((shown['sig'] == 'up').sum()), int((shown['sig'] == 'down').sum())
    fig_adj.update_layout(
        title=(f'{title}: adjusted shift - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               f'{x_note} Bars are 95% intervals; {n_up} decks significantly up, {n_down} down, '
               f'of {n}.</span>'),
        xaxis_title='Actual minus opponent-mix expectation (pp)', yaxis_title='',
        height=max(420, n * 26 + 150), margin=dict(l=150, r=30, t=90, b=60), showlegend=False,
    )
    return fig, fig_adj


def _late_crosscheck(late_df, plat):
    """Late-round shift vs the two placement-based skill measures, core decks only."""
    a = analyses[plat]
    late = late_df[late_df['enough']].set_index('deck')['adj_delta']
    panels = [
        ('delta_all_u25', a['skill_df'].set_index('deck')['delta_all_u25'],
         'Top-25% finishers minus All (pp)'),
        ('slope', a['skill_curve_df'].set_index('deck')['slope_pp_per_decade'],
         'Finish Curve slope (pp per decade of placement)'),
    ]
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.1,
                        subplot_titles=[p[2] for p in panels])
    stats_out = {}
    notes = []
    for col, (key, measure, label) in enumerate(panels, start=1):
        both = pd.concat([measure.rename('x'), late.rename('y')], axis=1, join='inner').dropna()
        if len(both) >= 4:
            pr = stats.pearsonr(both['x'], both['y'])
            sr = stats.spearmanr(both['x'], both['y'])
            stats_out[key] = {'n': len(both), 'pearson': float(pr[0]), 'pearson_p': float(pr[1]),
                              'spearman': float(sr[0]), 'spearman_p': float(sr[1])}
            notes.append(f'{"placement delta" if key == "delta_all_u25" else "curve slope"}: '
                         f'r = {pr[0]:+.2f} (p = {pr[1]:.2f}), rho = {sr[0]:+.2f}, n = {len(both)}')
        else:
            stats_out[key] = {'n': len(both)}
            notes.append(f'{key}: too few decks in both ({len(both)})')
        fig.add_trace(go.Scatter(
            x=both['x'], y=both['y'], mode='markers+text',
            text=[DECK_SHORT_NAMES.get(d, d) for d in both.index], textposition='top center',
            textfont=dict(size=9, color=UI.text2),
            marker=dict(size=10, color=[DECK_COLORS.get(d, UI.muted) for d in both.index],
                        line=dict(width=1.5, color='white')),
            customdata=[[d, f'{x:+.2f}', f'{y:+.2f}'] for d, x, y in zip(both.index, both['x'], both['y'])],
            hovertemplate=('<b>%{text}</b><br>' + label + ': %{customdata[1]}<br>'
                           'Late-round shift: %{customdata[2]}pp<extra></extra>'),
            showlegend=False), row=1, col=col)
        fig.add_hline(y=0, line_color=UI.rule, line_width=1, row=1, col=col)
        fig.add_vline(x=0, line_color=UI.rule, line_width=1, row=1, col=col)
        fig.update_xaxes(title_text=label, row=1, col=col)
    fig.update_yaxes(title_text='Late-round adjusted shift (pp)', row=1, col=1)
    fig.update_layout(
        title=(f'Do the finishing-tier measures agree with late rounds? - {PLATFORM_LABELS[plat]}'
               f'<br><span style="font-size:12px;color:{UI.muted}">{"; ".join(notes)}</span>'),
        height=560, margin=dict(l=70, r=30, t=110, b=60),
    )
    return fig, stats_out


def compute_pressure(platform):
    plat_label = PLATFORM_LABELS[platform]
    L = long_raw_by_platform[platform]

    S = mshift.pressure_slices(L)

    # Top cut vs the Swiss games of the same events and window.
    cut_df = mshift.slice_shift(L['LCut'], S['swiss'], analysis_decks, MIN_CUT_GAMES, metric='decided')
    fig_cut, fig_cut_adj = _shift_figures(
        cut_df, plat_label, 'Top Cut vs Swiss', 'Top cut', 'Swiss', MIN_CUT_GAMES,
        'Positive: the deck wins more in the cut than its cut opponents predict from Swiss results.')

    # Late Swiss rounds vs early Swiss rounds, both inside LATE_MIN_PLAYERS+ events.
    late_df = mshift.slice_shift(S['late_swiss'], S['early_swiss'], analysis_decks, MIN_LATE_GAMES,
                                 metric='decided')
    fig_late, fig_late_adj = _shift_figures(
        late_df, plat_label, f'Late Rounds (round {LATE_ROUND}+) vs Early', f'Rounds {LATE_ROUND}+',
        'Early rounds', MIN_LATE_GAMES,
        'Positive: the deck holds up better in the last Swiss rounds than its late opponents predict.')
    fig_late_check, late_check = _late_crosscheck(late_df, platform)

    return {'cut_df': cut_df, 'fig_cut': fig_cut, 'fig_cut_adj': fig_cut_adj,
            'late_df': late_df, 'fig_late': fig_late, 'fig_late_adj': fig_late_adj,
            'fig_late_check': fig_late_check, 'late_check': late_check}


print(f'Computing top-cut and late-round shifts ({LONG_DAYS}-day window)...')
for _p in ACTIVE_PLATFORMS:
    # Decided-game counts, never shrunk, so the raw view shares this result.
    _res = compute_pressure(_p)
    analyses[_p].update(_res)
    analyses[raw_view(_p)].update(_res)
    _c, _l = analyses[_p]['cut_df'], analyses[_p]['late_df']
    _cc, _ll = _c[_c['enough']], _l[_l['enough']]
    _chk = analyses[_p]['late_check'].get('delta_all_u25', {})
    print(f"  {_p}: top cut {len(_cc)} decks ({int((_cc['z'].abs() >= 1.96).sum())} significant); "
          f"late rounds {len(_ll)} decks ({int((_ll['z'].abs() >= 1.96).sum())} significant); "
          f"late vs placement delta r = {_chk.get('pearson', float('nan')):+.2f} (n={_chk.get('n', 0)})")


In [ ]:
# === Conversion: does a deck reach each finishing layer more often than its play predicts? ===
# The decks that fit the meta best should finish high more often than they are
# played -- a deck at 12.3% of play taking 13.9% of Top 8 finishes. Every deck gets
# that ratio, finishes / expected finishes, at each of CONVERSION_LAYERS, from the
# final standings of this window's events (the maths is in scripts/conversion.py):
#
#   * Expected finishes are matched event by event. Each event hands out a Top 8
#     whatever its size, while use rate is weighted by players, so a plain
#     "share of Top 8 / use rate" penalises decks that are popular at big events.
#     For a percent layer the two are the same number.
#   * Ratios are pulled toward 1.0 by a prior fitted per layer, over binomial noise
#     (in Upper 50% half the field gets through, and Poisson would double the noise).
#   * A layer is trusted only where it replicates inside this window: the events are
#     split in half at random, and decks' ratios in one half must predict the other
#     better than shuffling who finished where inside each event does. Layers that
#     fail are greyed, not hidden.
#
# Whether a ratio carries into the NEXT month is a different question, answered once
# in the experiments notebook (section 15); the tab states that result.
import conversion as cv
import th_csv as _th_csv_conv

CONV_SIG_ALPHA = 0.05
# What the month-to-month test (experiments section 15) found. Stated in the tab so
# the ratios are read as what they are.
CONVERSION_FINDING = (
    "Does it last? Checked month against month inside one card pool (experiments notebook, section 15: "
    "three pairs of back-to-back 30-day windows), a deck's ratio persists at every layer -- rank "
    "correlation 0.30-0.54 with the next month, all significant after correcting for testing seven "
    "layers. Most of that is win rate seen through placements: last month's win rate predicts next "
    "month's ratio about as well (0.35-0.55), and only Upper 10% keeps a significant signal of its own "
    "once win rate is controlled for. Read a high ratio as durable evidence that a deck is winning, not "
    "as information the win-rate tabs lack -- and note that it cannot tell a strong deck from strong "
    "players choosing it.")
# A (deck, layer) cell needs this many expected finishes to be shown at all. Below
# it the adjusted ratio is the prior's 1.0 -- no information, which must not read
# as "converts exactly as expected" (a deck absent from every 128+ player event has
# no Top 64 record).
CONV_MIN_CELL_EXP = 1.0
_conv_players = pd.read_csv(STANDINGS_PATH, dtype={'event_id': str, 'source': str, 'deck': str})
# The heat table's rows sort by a clicked column header; the default order is share of play.
# Marks: unsorted column, sorted high to low, sorted low to high.
CONV_SORT_DEFAULT = 'Share of play'
CONV_SORT_MARK = {None: '⇅', True: '▼', False: '▲'}
_CONV_TICKS = [0.25, 0.33, 0.5, 0.67, 0.8, 1, 1.25, 1.5, 2, 3, 4]


def _conv_empty(plat_label, title, height=320):
    f = go.Figure()
    f.add_annotation(text='No final standings for this data source in this window.', xref='paper',
                     yref='paper', x=0.5, y=0.5, showarrow=False, font=dict(size=13, color=UI.faint))
    f.update_layout(title=f'{title} - {plat_label}', height=height,
                    xaxis=dict(visible=False), yaxis=dict(visible=False))
    return f


def _conv_custom(g, meta):
    """Per-point hover payload: slug first, so a click opens the deck's panel."""
    rel = dict(zip(meta['layer'], meta['reliable']))

    def v(x):
        return float(x) if pd.notna(x) else None
    return [[r.deck, v(r.layer_share * 100), v(r.exp_share * 100), v(r.ratio_raw), v(r.lo), v(r.hi),
             v(r.obs), v(r.exp),
             'replicates across halves of this window' if rel[r.layer] else
             'does NOT replicate across halves of this window: mostly noise', r.label]
            for r in g.itertuples()]


_CONV_HOVER = ('%{customdata[1]:.1f}% of finishes vs %{customdata[2]:.1f}% expected from its play<br>'
               'Ratio %{customdata[3]:.2f} raw, adjusted ' + '%{meta_ratio}' +
               ' (90% %{customdata[4]:.2f}-%{customdata[5]:.2f})<br>'
               '%{customdata[6]:.0f} finishes against %{customdata[7]:.1f} expected<br>'
               'Layer: %{customdata[8]}<extra></extra>')


def _grey_columns(fig, meta, note=True):
    for i, r in enumerate(meta.itertuples()):
        if r.reliable:
            continue
        fig.add_shape(type='rect', xref='x', yref='paper', x0=i - 0.5, x1=i + 0.5, y0=0, y1=1,
                      fillcolor=UI.noise, line_width=0, layer='above')
        if note:
            fig.add_annotation(x=r.label, y=1.0, xref='x', yref='paper', yanchor='bottom', showarrow=False,
                               text='noise', font=dict(size=10, color=UI.faint))


def compute_conversion(platform):
    plat_label = PLATFORM_LABELS[platform]
    P = _conv_players[_conv_players['source'].isin(_th_csv_conv.PLATFORM_SOURCES[platform])]
    P = P.reset_index(drop=True)
    if P['deck'].notna().sum() == 0:
        return {'conv_df': pd.DataFrame(), 'conv_meta': pd.DataFrame(), 'conv_events': 0,
                'fig_conv_funnel': _conv_empty(plat_label, 'Conversion funnel'),
                'fig_conv_table': _conv_empty(plat_label, 'Conversion by layer')}

    long, meta = cv.funnel(P, CONVERSION_LAYERS)
    rel = [cv.split_half(P, lay, n_splits=CONV_SPLITS, n_perm=CONV_PERM, seed=0) for lay in meta['layer']]
    meta['rel_rho'] = [r['rho'] for r in rel]
    meta['rel_p'] = [r['p'] for r in rel]
    meta['rel_decks'] = [r['n_decks'] for r in rel]
    meta['reliable'] = meta['rel_p'].fillna(1.0) < CONV_SIG_ALPHA
    long = long.merge(meta[['layer', 'reliable']], on='layer')
    _thin = long['exp'] < CONV_MIN_CELL_EXP
    long.loc[_thin, ['ratio', 'lo', 'hi', 'ratio_raw']] = np.nan
    long.loc[_thin, 'sig'] = 'flat'

    # The dashboard's decks, most-played first, with a real sample at the shallowest layer.
    first = long[long['layer'] == meta['layer'].iloc[0]].set_index('deck')
    shown = [d for d in sorted(CORE_DECKS + SLEEPER_DECKS, key=lambda d: -first['use_share'].get(d, 0))
             if d in first.index and first.at[d, 'exp'] >= cv.MIN_EXP]
    long = long[long['deck'].isin(shown)].reset_index(drop=True)
    # Events that contribute: an event whose decks are all unknown takes no part in any ratio.
    n_events = int(P.loc[P['deck'].notna(), 'event_id'].nunique())
    labels = list(meta['label'])
    ticktext = [f'{r.label}<br><span style="font-size:10px;color:{UI.faint}">top {r.depth * 100:.0f}% '
                f'· {r.events} ev</span>' for r in meta.itertuples()]
    good = [r.label for r in meta.itertuples() if r.reliable]
    rel_note = (f'Replicates across random halves of the {n_events} events: {", ".join(good)}.' if good
                else f'No layer replicates across random halves of the {n_events} events.')

    # --- funnel: one line per deck, shallow to deep -------------------------------
    fig = go.Figure()
    for i, d in enumerate(shown):
        g = long[long['deck'] == d].set_index('layer').reindex(meta['layer']).reset_index()
        name = DECK_SHORT_NAMES.get(d, d)
        fig.add_trace(go.Scatter(
            x=labels, y=g['ratio'], mode='lines+markers', name=name,
            visible=True if i < CONV_SHOW else 'legendonly',
            line=dict(width=2, color=DECK_COLORS.get(d, UI.muted)),
            marker=dict(size=9, color=DECK_COLORS.get(d, UI.muted),
                        symbol=['circle' if s != 'flat' else 'circle-open' for s in g['sig']],
                        line=dict(width=1.5, color=DECK_COLORS.get(d, UI.muted))),
            customdata=_conv_custom(g, meta),
            hovertemplate=(f'<b>{name}</b> - ' + '%{customdata[9]}<br>' +
                           _CONV_HOVER.replace('%{meta_ratio}', '%{y:.2f}'))))
    fig.add_hline(y=1, line=dict(color=UI.faint, dash='dot', width=1))
    _grey_columns(fig, meta)
    fig.update_xaxes(tickvals=labels, ticktext=ticktext, title='Finishing layer, shallow to deep '
                     '(share of the field it holds, events counted)')
    fig.update_yaxes(type='log', tickvals=_CONV_TICKS, ticktext=[f'{t:g}' for t in _CONV_TICKS],
                     title='Finishes / expected (sample-adjusted, log scale)')
    fig.update_layout(
        # Plotly titles never wrap, so every subtitle sentence gets its own line.
        title=(f'Conversion funnel - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               'Above 1: the deck reaches that layer more often than its play predicts. '
               'Filled: the 90% interval excludes 1.<br>'
               f'{rel_note}<br>The {CONV_SHOW} most-played decks are drawn; click the legend for the '
               "rest, or a point for the deck's profile.</span>"),
        height=640, margin=dict(l=80, r=30, t=150, b=90), hovermode='closest')

    # --- table: every shown deck x layer ------------------------------------------
    rows = shown
    Z, T, C = [], [], []
    for d in rows:
        g = long[long['deck'] == d].set_index('layer').reindex(meta['layer']).reset_index()
        Z.append(np.clip(np.log2(g['ratio'].to_numpy(float)), -1, 1).tolist())
        T.append(['&mdash;' if not np.isfinite(r) else (f'<b>{r:.2f}</b>' if s != 'flat' else f'{r:.2f}')
                  for r, s in zip(g['ratio'], g['sig'])])
        # The last field is the sort key for the dashboard's column sort: the adjusted ratio,
        # unclipped (the colour scale stops at 0.5 and 2).
        C.append([cd + [float(r) if np.isfinite(r) else None]
                  for cd, r in zip(_conv_custom(g, meta), g['ratio'].to_numpy(float))])
    ylab = [f'{DECK_SHORT_NAMES.get(d, d)} ({first.at[d, "use_share"] * 100:.1f}%)' for d in rows]
    tick_ratios = [0.5, 0.71, 1, 1.41, 2]
    fig_t = go.Figure(go.Heatmap(
        z=Z, x=labels, y=ylab, text=T, texttemplate='%{text}', textfont=dict(size=11),
        colorscale=HEATMAP_SCALE, zmid=0, zmin=-1, zmax=1, customdata=C, xgap=2, ygap=2,
        colorbar=dict(title='ratio', tickvals=np.log2(tick_ratios).tolist(),
                      ticktext=[f'{t:g}' for t in tick_ratios], len=0.6),
        hovertemplate=('<b>%{y}</b> - %{customdata[9]}<br>' +
                       _CONV_HOVER.replace('%{meta_ratio}', '%{text}'))))
    _grey_columns(fig_t, meta, note=False)
    # Column headers are annotations that capture clicks, so the dashboard can sort the rows
    # by any column (bindRowSorts). 'sort:-1' is the built order, by share of play.
    _head = dict(y=1, yref='paper', yanchor='bottom', showarrow=False, captureevents=True,
                 font=dict(size=12, color=UI.text))
    heads = [dict(_head, x=lab, xref='x', name=f'sort:{i}', text=f'{lab} {CONV_SORT_MARK[None]}')
             for i, lab in enumerate(labels)]
    heads.append(dict(_head, x=0, xref='paper', xanchor='right', name='sort:-1',
                      text=f'{CONV_SORT_DEFAULT} {CONV_SORT_MARK[True]}'))
    fig_t.update_xaxes(side='top', tickvals=labels, ticktext=labels, showticklabels=False)
    fig_t.update_yaxes(autorange='reversed')
    fig_t.update_layout(
        title=(f'Conversion by layer - {plat_label}<br><span style="font-size:12px;color:{UI.muted}">'
               'Sample-adjusted finishes / expected; bold where the 90% interval excludes 1.<br>'
               f'Rows: tracked decks with {cv.MIN_EXP:.0f}+ expected finishes at the shallowest layer, '
               'by share of play (in brackets).<br>Faded columns do not replicate in this window. '
               'Click a column header to sort by it, again to reverse.</span>'),
        annotations=heads,
        meta={'sortRows': {'key': len(C[0][0]) - 1 if C else 10, 'labels': labels,
                           'default': CONV_SORT_DEFAULT, 'marks': {'none': CONV_SORT_MARK[None],
                                                                   'desc': CONV_SORT_MARK[True],
                                                                   'asc': CONV_SORT_MARK[False]}}},
        height=max(420, 26 * len(rows) + 210), margin=dict(l=210, r=30, t=170, b=30))

    return {'conv_df': long, 'conv_meta': meta, 'conv_events': n_events,
            'fig_conv_funnel': fig, 'fig_conv_table': fig_t}


print(f'Computing conversion across {len(CONVERSION_LAYERS)} finishing layers '
      f'({STANDINGS_WINDOW["start_date"]} .. {STANDINGS_WINDOW["end_date"]})...')
for _p in ACTIVE_PLATFORMS:
    # Finishes over expected finishes: no matchup win rate goes in, so the raw view shares it.
    _res = compute_conversion(_p)
    analyses[_p].update(_res)
    analyses[raw_view(_p)].update(_res)
    _m = analyses[_p]['conv_meta']
    if len(_m):
        print(f"  {_p}: {analyses[_p]['conv_events']} events, "
              f"{analyses[_p]['conv_df']['deck'].nunique()} decks; layers that replicate: "
              f"{', '.join(_m.loc[_m['reliable'], 'layer']) or 'none'}")
    else:
        print(f'  {_p}: no standings in this window')


In [ ]:
# Sleeper candidates (decks outside the core meta) for the active PLATFORM.
analyses[PLATFORM]['fig_sleeper'].show()
analyses[PLATFORM]['fig_sleeper_table'].show()
print(f"Sleeper analysis for PLATFORM = {PLATFORM!r}")


In [ ]:
# Win-rate bar chart for the active PLATFORM.
analyses[PLATFORM]['fig_wr'].show()
print(f"Win rate computed for {len(analyses[PLATFORM]['agg_df']) // 3} decks (active platform: {PLATFORM!r}).")


In [ ]:
# Consistency scatter for the active PLATFORM.
analyses[PLATFORM]['fig_consist'].show()
print(f"Consistency analysis: {len(analyses[PLATFORM]['matchup_breakdown']['All'])} decks (platform: {PLATFORM!r}).")


In [ ]:
# Skill-shift matchup heatmap and bar chart for the active PLATFORM.
analyses[PLATFORM]['fig_skill_mu_heatmap'].show()
analyses[PLATFORM]['fig_skill_mu_top'].show()
print(f"Skill-sensitive matchups: {len(analyses[PLATFORM]['skill_mu_df'])} pairs (platform: {PLATFORM!r}).")


In [ ]:
# Main matchup heatmap and most-lopsided matchups for the active PLATFORM.
analyses[PLATFORM]['fig_heatmap'].show()
analyses[PLATFORM]['fig_lopsided'].show()
print(f"Matchup heatmap rendered for platform {PLATFORM!r} ({len(analyses[PLATFORM]['lop_df'])} lopsided pairs shown).")


In [ ]:
# Swiss simulation results for the active PLATFORM.
analyses[PLATFORM]['fig_swiss'].show()
print(f"Swiss simulation rendered for platform {PLATFORM!r} ({N_SIMS:,} runs).")


In [ ]:
# Skill ceiling: rank-bump and scatter for the active PLATFORM.
analyses[PLATFORM]['fig_bump'].show()
analyses[PLATFORM]['fig_skill'].show()
print(f"Skill ceiling rendered for platform {PLATFORM!r}.")


In [ ]:
# === Dashboard: multi-platform HTML widget ===
import os
import json as jsonlib

# === Overview tier rating ====================================================
# A composite of two things, each measured against a line taken from this
# platform's own data rather than a round number:
#   win rate   vs the format's tie-aware break-even (~49%, not 50%: ties put an
#              even matchup below 50)
#   cut rate   vs the simulated field's own average cut rate
# The weights and letter cutoffs are a scoring scale, not a claim about the game.
#
# It used to add 0.3 x the "skill delta" (Upper 25% win rate minus All). The
# experiments notebook's section 14 audit removed it: across back-to-back monthly
# windows of one card pool, where plain win rate replicates (rho +0.37, +0.60),
# the delta does not (+0.28, +0.10, neither significant), and sampling noise is
# 60-100% of its spread between decks. A number that does not persist from one
# month to the next cannot rank decks.
TIER_CUTOFFS = (('S', 12), ('A', 5), ('B', -2), ('C', -8))


def _tier_baselines(plat):
    """(break-even win rate %, field-average cut rate %) for one platform.

    The cut baseline is weighted by simulated appearances, so it is the average
    cut rate of the field the Swiss simulation actually played.
    """
    a = analyses[plat]
    sw = a['swiss_df']
    n = sw['n_appearances'].to_numpy(float)
    field_cut = float(np.average(sw['p_cut'], weights=n)) if n.sum() > 0 else np.nan
    return float(a['neutral_wr']), field_cut


def _tier_score(wr, p_cut, neutral, field_cut):
    """Composite score; a missing input contributes nothing rather than NaN."""
    score = 0.0
    if np.isfinite(wr):
        score += (wr - neutral) * 2
    if np.isfinite(p_cut) and np.isfinite(field_cut):
        score += (p_cut - field_cut) * 0.5
    return score


def _tier_from_score(score):
    for tier, cutoff in TIER_CUTOFFS:
        if score >= cutoff:
            return tier
    return 'D'


def _overview_frame(plat):
    """One row per deck on this platform: everything the Overview shows."""
    a = analyses[plat]
    overview = a['skill_df'].merge(a['meta_share'], on='deck', how='left')
    overview = overview.merge(a['swiss_df'][['deck', 'mean_mp', 'p_cut']], on='deck', how='left')
    overview = overview.merge(
        a['consist_df'][a['consist_df']['tier'] == 'All'][
            ['deck', 'matchup_std', 'fav_impact', 'unfav_impact', 'net_impact']
        ], on='deck', how='left'
    )
    # Drop decks that don't appear at all on this platform.
    overview = overview[overview['meta_share'].fillna(0) > 0].copy()
    neutral, field_cut = _tier_baselines(plat)
    overview['tier_score'] = [_tier_score(w, c, neutral, field_cut)
                              for w, c in zip(overview['wr_All'], overview['p_cut'])]
    overview['tier_rating'] = overview['tier_score'].map(_tier_from_score)
    # Finishing-tier win rates are shown only on a real sample: in-person Upper 25%
    # once put a deck at 0% -- a -48pp "delta" -- on two games.
    games = a['agg_df'].pivot_table(index='deck', columns='tier', values='total_games', aggfunc='sum')
    for tier in ('Upper50', 'Upper25'):
        n = overview['deck'].map(games[tier] if tier in games else {}).fillna(0)
        overview.loc[n < MIN_SAMPLE, f'wr_{tier}'] = np.nan
    overview.loc[overview['wr_Upper25'].isna(), 'delta_all_u25'] = np.nan
    # Computed alongside the role bootstrap (same draws); see _role_stability.
    roles = (roles_data_per.get(plat) or {}).get('deck')
    if roles is not None and 'tier_stability' in roles.columns:
        overview = overview.merge(roles[['deck', 'tier_stability']], on='deck', how='left')
    else:
        overview['tier_stability'] = np.nan
    return overview.sort_values('wr_All', ascending=False).reset_index(drop=True)


TIER_BADGE = {'S': (UI.gold_bg, UI.gold_ink), 'A': (UI.fav_bg, UI.fav_ink), 'B': (UI.accent_wash2, UI.accent),
              'C': (UI.warn_bg, UI.warn), 'D': (UI.unfav_bg, UI.unfav_ink)}


def _build_overview(plat):
    """The Overview as an HTML table: Plotly tables fire no click events, and a
    row click is the most direct way into a deck's detail panel."""
    ov = _overview_frame(plat)
    neutral, field_cut = _tier_baselines(plat)

    def num(v, fmt):
        return format(v, fmt) if pd.notna(v) and np.isfinite(v) else '&mdash;'

    rows = []
    for _, r in ov.iterrows():
        bg, fg = TIER_BADGE.get(r['tier_rating'], (UI.wash, UI.text))
        held = r['tier_stability']
        held_cls = ' ov-shaky' if pd.notna(held) and held < 50 else ''
        delta = r['delta_all_u25']
        delta_cls = '' if pd.isna(delta) else (' delta-pos' if delta >= 0 else ' delta-neg')
        rows.append(
            f'<tr class="ov-row" data-deck="{r["deck"]}" tabindex="0" title="Open deck profile">'
            f'<td><span class="tier-badge" style="background:{bg};color:{fg}">{r["tier_rating"]}</span></td>'
            f'<td class="num{held_cls}">{num(held, ".0f")}%</td>'
            f'<td class="opp-name" style="border-left:3px solid {DECK_COLORS.get(r["deck"], UI.rule)}">'
            f'{DECK_SHORT_NAMES.get(r["deck"], r["deck"])}</td>'
            f'<td class="num">{num(r["meta_share"], ".1f")}</td>'
            f'<td class="num">{num(r["wr_All"], ".1f")}</td>'
            f'<td class="num">{num(r["wr_Upper50"], ".1f")}</td>'
            f'<td class="num">{num(r["wr_Upper25"], ".1f")}</td>'
            f'<td class="num">{num(r["fav_impact"], ".2f")}</td>'
            f'<td class="num">{num(r["unfav_impact"], ".2f")}</td>'
            f'<td class="num">{num(r["mean_mp"], ".1f")}</td>'
            f'<td class="num">{num(r["p_cut"], ".1f")}</td>'
            f'<td class="num{delta_cls}">{num(delta, "+.1f")}</td>'
            f'</tr>')
    head = ''.join(f'<th class="{c}">{h}</th>' for h, c in (
        ('Tier', ''), ('Tier held', 'num'), ('Deck', ''), ('Meta %', 'num'), ('WR All%', 'num'),
        ('WR top 50%', 'num'), ('WR top 25%', 'num'), ('Fav Impact', 'num'), ('Unfav Impact', 'num'),
        ('Swiss MP', 'num'), ('Cut %', 'num'), ('Top 25% &minus; All', 'num')))
    return (
        f'<div class="ov-title">Meta Overview - {PLATFORM_LABELS[plat]}</div>'
        f'<div class="ov-sub">Tier score: win rate vs the {neutral:.1f}% break-even and cut rate vs the '
        f'field\'s {field_cut:.1f}%. "Tier held" = share of {ROLE_BOOT} resamples keeping the same tier '
        f'(under 50% in amber); an upper bound, since cut % is held fixed. The top-50% / top-25% columns '
        f'are win rates among players who <i>finished</i> there -- selected on having won, so every deck '
        f'looks better, and the difference between decks does not persist month to month (it is not a '
        f'skill measure). Shown with {MIN_SAMPLE}+ games. Click a deck for its full profile.</div>'
        f'<div class="compare-table-wrap"><table class="compare-table ov-table">'
        f'<thead><tr>{head}</tr></thead><tbody>{"".join(rows)}</tbody></table></div>')


# === Meta Roles tab: classify decks into roles using two taxonomies =========
# "Meta Position" is descriptive (Pillar, Counter, Sleeper, ...); "Tournament
# Strategy" is prescriptive (Best-in-Class, Safe Pick, ...). Both derive from
# this platform's own quantiles -- no hardcoded deck names or magic numbers.
META_ROLE_ORDER = ['Pillar', 'Rising', 'Counter', 'Sleeper', 'Polarizing', 'Fallen',
                   'Niche', 'Generalist']
META_ROLE_COLORS = {
    'Pillar':     UI.accent, 'Counter':    UI.unfav, 'Sleeper':    UI.teal,
    'Rising':     UI.cyan,
    'Polarizing': UI.orange, 'Fallen':     UI.warn, 'Niche':      UI.violet,
    'Generalist': UI.muted,
}
META_ROLE_DEFS = {
    'Pillar':     'High meta share AND high win rate. These decks define the format and warp deck-building decisions.',
    'Counter':    'Out-performs against Pillars even at lower popularity. Anti-meta tech aimed at the format leaders.',
    'Sleeper':    'Strong win rate at low popularity. Under-represented relative to their performance.',
    'Polarizing': 'Wide spread of favorable vs unfavorable matchups. Boom-or-bust depending on what you face.',
    'Rising':     'Gaining share fast on a below-median win rate. New or still being solved rather than in decline -- and you will face more of it, not less.',
    'Fallen':     'High popularity, weak win rate, and share flat or shrinking. Overplayed relative to results, with the field already moving off it.',
    'Niche':      'Low usage with moderate win rate. Found a specific corner of the meta.',
    'Generalist': 'Even matchup spread, moderate WR and usage. Few extreme matchups in either direction.',
}
TOURN_ROLE_ORDER = ['Best-in-Class', 'Safe Pick', 'Counter Pick', 'Field Pick', 'Avoid']
TOURN_ROLE_COLORS = {
    'Best-in-Class': UI.gold, 'Safe Pick': UI.emerald,
    'Counter Pick': UI.accent, 'Field Pick': UI.accent2, 'Avoid': UI.unfav,
}
# "Counter Pick" replaced "High-Ceiling Counter", and Avoid lost its "Polarizing with a
# negative skill delta" clause: both leaned on the placement skill delta, which the
# section 14 audit found does not persist between months. What is left describes
# what the matchup data does support -- whether a deck's value depends on the field.
TOURN_ROLE_DEFS = {
    'Best-in-Class':  'The single top-performing Pillar. Default pick if you have to choose one deck cold.',
    'Safe Pick':      'Reliably converts in Swiss (high cut rate, low matchup variance). Lock for a long event.',
    'Counter Pick':   'Its value depends on the field: it beats the format leaders, has a lopsided matchup spread, or wins well at low popularity. Pick it when you expect the decks it beats.',
    'Field Pick':     'Plays best against the broad field of less-popular decks. Good for early Swiss rounds.',
    'Avoid':          'Popular, below-median win rate, and the field is not moving onto it. Pick something else.',
}


def _family_mu_variance(fam_matrix, usage_map, families):
    """Spread of a family's matchup win rates, weighted by opponent family usage.

    The same definition the per-deck consistency tab uses, one level up.
    """
    out = []
    for f in families:
        if f not in fam_matrix.index:
            out.append(np.nan)
            continue
        row = fam_matrix.loc[f].dropna()
        if len(row) == 0:
            out.append(np.nan)
            continue
        opp_w = np.array([usage_map.get(o, 0.0) for o in row.index], dtype=float)
        vals = row.values * 100.0
        if opp_w.sum() == 0:
            out.append(float(np.std(vals)))
            continue
        mean = float(np.average(vals, weights=opp_w))
        out.append(float(np.sqrt(np.average((vals - mean) ** 2, weights=opp_w))))
    return out


def _build_family_wr_matrix(plat, raw=None):
    """Aggregate the per-deck All-tier matchup data to family level (raw wins
    over raw total games, then smoothed). Within-family matches are skipped
    (they become "mirror" at family level).

    `raw` overrides the real frame so the stability bootstrap can build the same
    matrix from a resampled draw instead of a reimplementation of this roll-up.
    """
    raw = raw_by_platform[plat]['All'] if raw is None else raw
    families = sorted(set(archetype_by_deck.values()))
    fam_wins = {}
    fam_total = {}
    for _, row in raw.iterrows():
        if row['is_mirror'] or row['is_other']:
            continue
        # Core-only, to match what the rest of the standard tabs analyse.
        if row['deck1'] not in CORE_DECKS or row['deck2'] not in CORE_DECKS:
            continue
        fa = archetype_by_deck.get(row['deck1'])
        fb = archetype_by_deck.get(row['deck2'])
        if fa is None or fb is None or fa == fb:
            continue
        key = (fa, fb)
        fam_wins[key] = fam_wins.get(key, 0) + int(row['wins'])
        fam_total[key] = fam_total.get(key, 0) + int(row['actual_total'])
    matrix = pd.DataFrame(np.nan, index=families, columns=families)
    fam_rates = []
    for (fa, fb), w in fam_wins.items():
        g = fam_total[(fa, fb)]
        if g > 0:
            fam_rates.append(((fa, fb), w / g, g))
    if fam_rates:
        # Same sample-size shrinkage as the per-deck matchups, one level up, so a
        # two-game family pairing can't read as a blowout (none in a raw view).
        neutral, k = _eb_shrinkage(np.array([r for _, r, _ in fam_rates]),
                                   np.array([n for _, _, n in fam_rates]), is_adjusted(plat))
        for (fa, fb), rate, n in fam_rates:
            matrix.loc[fa, fb] = (n * rate + k * neutral) / (n + k)
    return matrix


def _family_color(fam):
    """Pick a deterministic color for a family. Prefer the color of any member
    deck (typically the parent variant); fall back to a slate gray."""
    members = [d for d, f in archetype_by_deck.items() if f == fam]
    for m in members:
        if m in DECK_COLORS:
            return DECK_COLORS[m]
    return UI.muted


def _velocity_for(plat, units, unit_of_deck):
    """Share velocity per unit, fitted straight from the cached daily trend.

    `units` may be decks or archetype families. The daily counts are re-keyed
    onto the unit and summed *before* the slope is fitted, rather than fitting
    per deck and averaging afterwards: share is additive, but a bootstrap
    interval is not, and averaging intervals would understate a family's
    movement.

    Direction comes from the interval rather than the point estimate -- a deck
    whose 90% range spans zero is drifting no faster than sampling noise. A
    platform with too few periods (in-person majors are a handful of discrete
    weekends) returns 'Unknown' for everything, and the classifier then behaves
    exactly as it did before there was a trend at all.
    """
    units = list(units)
    empty = pd.DataFrame({'deck': units, 'pp_per_wk': np.nan, 'vel_q05': np.nan,
                          'vel_q95': np.nan, 'trend': 'Unknown', 'trend_weeks': 0})
    days = trend_by_platform.get(plat) or []
    if not days:
        return empty

    remapped = []
    for day in days:
        agg = {}
        for entry in day.get('decks', []):
            u = unit_of_deck(entry['deck_id'])
            if u is None:
                continue
            agg[u] = agg.get(u, 0) + entry.get('count', 0)
        remapped.append({'date': day['date'],
                         'decks': [{'deck_id': u, 'count': n} for u, n in agg.items()]})

    sh, ct, _lab = md.weekly_series(remapped, units, days_per_bucket=7)
    if len(sh) < MIN_TREND_PERIODS:
        return empty

    v = md.velocity(sh, ct, n_boot=VELOCITY_BOOT, rng=np.random.default_rng(17))
    q05, q95 = v['pp_q05'], v['pp_q95']
    return pd.DataFrame({
        'deck': units,
        'pp_per_wk': v['pp_per_period'],
        'vel_q05': q05, 'vel_q95': q95,
        'trend': np.where(q05 > 0, 'Rising', np.where(q95 < 0, 'Falling', 'Flat')),
        'trend_weeks': v['periods_present'],
    })


def _build_role_inputs(plat, aggregation):
    """Build the per-unit base dataframe + matchup matrix + display_name map."""
    a = analyses[plat]
    if aggregation == 'deck':
        base = a['skill_df'].merge(a['meta_share'], on='deck', how='left')
        base = base.merge(a['swiss_df'][['deck', 'p_cut']], on='deck', how='left')
        consist_all = a['consist_df'][a['consist_df']['tier'] == 'All'][['deck', 'matchup_std']]
        base = base.merge(consist_all, on='deck', how='left')
        base = base[base['meta_share'].fillna(0) > 0].copy()
        base = base.rename(columns={'matchup_std': 'mu_variance', 'wr_All': 'wr', 'meta_share': 'usage'})
        wr_matrix = matrices_by_platform[plat]['All']['smoothed_wr']
        base['display_name'] = base['deck'].map(lambda d: DECK_SHORT_NAMES.get(d, d))
        base['color'] = base['deck'].map(lambda d: DECK_COLORS.get(d, UI.muted))
        _core = set(base['deck'])
        base = base.merge(_velocity_for(plat, base['deck'],
                                        lambda d: d if d in _core else None),
                          on='deck', how='left')
        return base.reset_index(drop=True), wr_matrix, 'deck'
    # aggregation == 'family'
    deck_base, _, _ = _build_role_inputs(plat, 'deck')
    if 'archetype_by_deck' not in globals():
        # Defensive: archetype map cell hasn't run; treat every deck as its own family.
        deck_base['family'] = deck_base['deck']
        name_map = {}
    else:
        deck_base['family'] = deck_base['deck'].map(archetype_by_deck).fillna(deck_base['deck'])
        name_map = _archetype_name_by_slug
    # All-tier total_games per member for weighted averages
    agg_all = a['agg_df'][a['agg_df']['tier'] == 'All'].set_index('deck')['total_games']
    deck_base['_w'] = deck_base['deck'].map(agg_all).fillna(1.0).astype(float)
    grouped = deck_base.groupby('family', sort=False)
    def _wavg(col, df):
        v = df[col].values.astype(float)
        w = df['_w'].values.astype(float)
        m = ~np.isnan(v) & (w > 0)
        if not m.any():
            return np.nan
        return float(np.average(v[m], weights=w[m]))
    rows = []
    for fam, g in grouped:
        rows.append({
            'deck': fam,  # keep column name 'deck' for downstream compatibility
            'usage': float(g['usage'].sum()),
            'wr': _wavg('wr', g),
            'p_cut': _wavg('p_cut', g),
        })
    fam_df = pd.DataFrame(rows)
    # Fit family velocity from the trend directly rather than aggregating the
    # per-deck slopes -- see _velocity_for.
    _fam_of = dict(zip(deck_base['deck'], deck_base['family']))
    fam_df = fam_df.merge(_velocity_for(plat, fam_df['deck'], _fam_of.get),
                          on='deck', how='left')
    fam_wr_matrix = _build_family_wr_matrix(plat)
    fam_df['mu_variance'] = _family_mu_variance(
        fam_wr_matrix, dict(zip(fam_df['deck'], fam_df['usage'])), fam_df['deck'])
    fam_df['display_name'] = fam_df['deck'].map(lambda f: name_map.get(f, f.replace('-', ' ').title()))
    fam_df['color'] = fam_df['deck'].map(_family_color)
    return fam_df.reset_index(drop=True), fam_wr_matrix, 'family'


def _compute_meta_roles(plat, aggregation='deck'):
    base, wr_matrix, unit_kind = _build_role_inputs(plat, aggregation)
    if base is None or len(base) < 5:
        return None
    out = _classify_roles(base, wr_matrix, aggregation)
    # merge() drops DataFrame.attrs, and the rules panel reads the cutoffs from
    # there, so carry them across explicitly.
    cutoffs = dict(out.attrs.get('cutoffs', {}))
    stab = _role_stability(plat, base, out, aggregation)
    if stab is not None:
        out = out.merge(stab, on='deck', how='left')
        out['rationale'] = [
            f'{r}; role held in {s:.0f}% of {ROLE_BOOT} resamples'
            if pd.notna(s) else r
            for r, s in zip(out['rationale'], out['meta_stability'])]
        cutoffs['role_boot'] = ROLE_BOOT
    out.attrs['cutoffs'] = cutoffs
    return out


def _classify_roles(base, wr_matrix, aggregation='deck'):
    """Assign both taxonomies to an already-built input frame.

    Split out of `_compute_meta_roles` so the stability bootstrap can push
    resampled inputs through exactly the same rules, rather than through a
    reimplementation of them that could drift.
    """
    base = base.copy()
    base['wr'] = base['wr'].fillna(base['wr'].median() if base['wr'].notna().any() else 0.0)
    wr_mid = base['wr'].median()
    mu_med = base['mu_variance'].median()
    base['mu_variance'] = base['mu_variance'].fillna(mu_med if not pd.isna(mu_med) else 0.0)
    base['p_cut'] = base['p_cut'].fillna(0.0)

    usage_hi = base['usage'].quantile(0.66)
    usage_lo = base['usage'].quantile(0.33)
    sleeper_wr = base['wr'].quantile(0.75)
    var_hi = base['mu_variance'].quantile(0.75)

    base['meta_role'] = ''
    base.loc[(base['usage'] >= usage_hi) & (base['wr'] >= wr_mid) & (base['meta_role'] == ''), 'meta_role'] = 'Pillar'
    # Split what used to be a single "Fallen" bucket by actual direction of
    # travel. Before there was a trend input, a deck the field was piling into
    # and one the field was abandoning got the same label and the same "Avoid"
    # advice, purely because both sat above the usage line with a below-median
    # win rate. They are opposite situations: a riser is new or unsolved, and it
    # is what you will face MORE of next event.
    base['trend'] = base['trend'].fillna('Unknown')
    _rising = base['trend'].eq('Rising')
    base.loc[(base['usage'] >= usage_hi) & (base['wr'] < wr_mid) & _rising
             & (base['meta_role'] == ''), 'meta_role'] = 'Rising'
    base.loc[(base['usage'] >= usage_hi) & (base['wr'] < wr_mid) & (base['meta_role'] == ''), 'meta_role'] = 'Fallen'
    base.loc[(base['usage'] <= usage_lo) & (base['wr'] >= sleeper_wr) & (base['meta_role'] == ''), 'meta_role'] = 'Sleeper'

    pillars = base.loc[base['meta_role'] == 'Pillar', 'deck'].tolist()
    wr_vs_pillars = {}
    for unit in base['deck']:
        if not pillars or unit not in wr_matrix.index:
            wr_vs_pillars[unit] = np.nan
            continue
        vals = []
        for p in pillars:
            if p == unit or p not in wr_matrix.columns:
                continue
            v = wr_matrix.loc[unit, p]
            if not pd.isna(v):
                vals.append(float(v) * 100.0)
        wr_vs_pillars[unit] = float(np.mean(vals)) if vals else np.nan
    base['wr_vs_pillars'] = base['deck'].map(wr_vs_pillars)

    # Counter (refined): track per-Pillar WR, not just the average. With many
    # stylistically different Pillars in the meta, an average WR collapses
    # asymmetric counters into mediocrity -- a deck that beats Dragapult 60%
    # but loses to Mega Lucario 40% averages ~50% and would otherwise miss
    # the Counter bar. Require beating >=2 distinct Pillars at >=median+5pp,
    # weighted by Pillar popularity (so beating the most-played Pillars matters
    # more for ranking and downstream tournament-strategy decisions).
    counter_beat_threshold = wr_mid + 5.0
    pillar_set = set(pillars)
    pillar_usage = {p: float(base.loc[base['deck'] == p, 'usage'].iloc[0])
                    for p in pillars if (base['deck'] == p).any()}
    pillar_display = {p: base.loc[base['deck'] == p, 'display_name'].iloc[0]
                      for p in pillars if (base['deck'] == p).any()}
    beats_by_deck = []
    for unit in base['deck']:
        if unit in pillar_set or unit not in wr_matrix.index:
            beats_by_deck.append([])
            continue
        rec = []
        for p in pillars:
            if p == unit or p not in wr_matrix.columns:
                continue
            v = wr_matrix.loc[unit, p]
            if pd.isna(v):
                continue
            wr_pct = float(v) * 100.0
            if wr_pct >= counter_beat_threshold:
                rec.append({'pillar': p, 'wr': wr_pct})
        beats_by_deck.append(rec)
    base['pillars_beaten'] = beats_by_deck
    base['pillars_beaten_count'] = base['pillars_beaten'].map(len)
    def _adv_score(beats):
        return sum((b['wr'] - wr_mid) * pillar_usage.get(b['pillar'], 0.0) for b in beats)
    base['counter_score'] = base['pillars_beaten'].map(_adv_score)
    counter_mask = (base['pillars_beaten_count'] >= 2) & (base['meta_role'] == '')
    base.loc[counter_mask, 'meta_role'] = 'Counter'
    base.loc[(base['mu_variance'] >= var_hi) & (base['meta_role'] == ''), 'meta_role'] = 'Polarizing'
    base.loc[(base['usage'] <= usage_lo) & (base['wr'] >= wr_mid - 3.0) & (base['meta_role'] == ''), 'meta_role'] = 'Niche'
    base.loc[(base['mu_variance'] < mu_med) & (base['meta_role'] == ''), 'meta_role'] = 'Generalist'

    unlabeled = (base['meta_role'] == '')
    if unlabeled.any() and (~unlabeled).any():
        wvp_fill = base['wr_vs_pillars'].median()
        if pd.isna(wvp_fill):
            wvp_fill = base['wr'].median()
        base['_wvp_fill'] = base['wr_vs_pillars'].fillna(wvp_fill)
        feat_cols = ['usage', 'wr', 'mu_variance', '_wvp_fill']
        X = base[feat_cols].values.astype(float)
        mu = X.mean(axis=0)
        sigma = X.std(axis=0)
        sigma = np.where(sigma == 0, 1.0, sigma)
        Xz = (X - mu) / sigma
        unlabeled_idx = np.where(unlabeled.values)[0]
        roles_present = base.loc[~unlabeled, 'meta_role'].unique()
        centroids = {r: Xz[(base['meta_role'] == r).values].mean(axis=0) for r in roles_present}
        role_col_pos = base.columns.get_loc('meta_role')
        for idx in unlabeled_idx:
            dists = {r: float(np.linalg.norm(Xz[idx] - c)) for r, c in centroids.items()}
            base.iat[idx, role_col_pos] = min(dists, key=dists.get)
        base = base.drop(columns=['_wvp_fill'])
    base.loc[base['meta_role'] == '', 'meta_role'] = 'Generalist'

    base['tournament_role'] = ''
    pillars_df = base[base['meta_role'] == 'Pillar'].sort_values('wr', ascending=False)
    if len(pillars_df):
        base.loc[base['deck'] == pillars_df.iloc[0]['deck'], 'tournament_role'] = 'Best-in-Class'
    p_cut_med = base['p_cut'].median()
    safe = ((base['meta_role'] == 'Pillar')
            | ((base['meta_role'] == 'Generalist') & (base['p_cut'] >= p_cut_med))) & (base['tournament_role'] == '')
    base.loc[safe, 'tournament_role'] = 'Safe Pick'
    # Trajectory now enters through Fallen, which no longer contains risers, so
    # Avoid inherits the fix without a clause of its own. An earlier attempt added
    # "falling AND below-median WR" directly here; that is a conjunction of two
    # conditions each true of about half the field, and it labelled 9 of 25 decks
    # Avoid purely by construction.
    avoid = (base['meta_role'] == 'Fallen') & (base['tournament_role'] == '')
    base.loc[avoid, 'tournament_role'] = 'Avoid'
    counter_pick = base['meta_role'].isin(['Counter', 'Polarizing', 'Sleeper']) & (base['tournament_role'] == '')
    base.loc[counter_pick, 'tournament_role'] = 'Counter Pick'
    base.loc[base['tournament_role'] == '', 'tournament_role'] = 'Field Pick'

    # Stash cutoffs on the dataframe so the rules panel can read them.
    base.attrs['cutoffs'] = {
        'usage_hi': float(usage_hi), 'usage_lo': float(usage_lo),
        'wr_mid': float(wr_mid), 'sleeper_wr': float(sleeper_wr),
        'var_hi': float(var_hi), 'mu_med': float(mu_med),
        'counter_beat_threshold': float(counter_beat_threshold),
        'counter_min_pillars_beaten': 2,
        'p_cut_med': float(p_cut_med),
        'trend_weeks': int(base['trend_weeks'].max()) if 'trend_weeks' in base else 0,
        'n_rising': int((base['trend'] == 'Rising').sum()),
        'n_falling': int((base['trend'] == 'Falling').sum()),
        'aggregation': aggregation,
    }

    def _rationale(row):
        bits = [f"{row['usage']:.1f}% usage", f"{row['wr']:.1f}% 3pt-WR", f"σ={row['mu_variance']:.1f}"]
        pb = row.get('pillars_beaten') or []
        if len(pb) > 0:
            ranked = sorted(pb, key=lambda b: -b['wr'])
            shown = ranked[:3]
            beats_str = ', '.join(
                f"{pillar_display.get(b['pillar'], b['pillar'])} ({b['wr']:.0f}%)"
                for b in shown
            )
            if len(ranked) > 3:
                beats_str += f" +{len(ranked) - 3} more"
            bits.append(f"beats {beats_str}")
        tr = row.get('trend')
        if tr in ('Rising', 'Falling'):
            bits.append(f"{tr.lower()} {row['pp_per_wk']:+.2f}pp/wk")
        return ', '.join(bits)
    base['rationale'] = base.apply(_rationale, axis=1)
    return base.reset_index(drop=True)


def _build_roles_scatter(plat, taxonomy, df):
    role_col = 'meta_role' if taxonomy == 'meta' else 'tournament_role'
    palette = META_ROLE_COLORS if taxonomy == 'meta' else TOURN_ROLE_COLORS
    order = META_ROLE_ORDER if taxonomy == 'meta' else TOURN_ROLE_ORDER
    plat_label = PLATFORM_LABELS[plat]
    tax_label = 'Meta Position' if taxonomy == 'meta' else 'Tournament Strategy'
    fig = go.Figure()
    usage_median = df['usage'].median()
    max_usage = max(float(df['usage'].max()), 1.0)
    for role in order:
        sub = df[df[role_col] == role]
        if len(sub) == 0:
            continue
        marker_size = 8 + 22 * np.sqrt(np.clip(sub['usage'].values / max_usage, 0, 1))
        fig.add_trace(go.Scatter(
            x=sub['usage'], y=sub['wr'],
            mode='markers+text',
            text=sub['display_name'].values,
            textposition='top center', textfont=dict(size=9, color=UI.text2),
            marker=dict(size=marker_size, color=palette.get(role, UI.faint),
                        line=dict(width=2, color='white'), opacity=0.9),
            name=role,
            customdata=np.stack([sub['display_name'].values, sub['rationale'].values], axis=-1),
            hovertemplate=('<b>%{customdata[0]}</b><br>'
                           f'Role: {role}<br>'
                           'Usage: %{x:.2f}%<br>'
                           '3pt WR: %{y:.1f}%<br>'
                           '%{customdata[1]}<extra></extra>'),
        ))
    fig.add_hline(y=df['wr'].median(), line_dash='dot', line_color=UI.rule, line_width=1.5)
    fig.add_vline(x=usage_median, line_dash='dot', line_color=UI.rule, line_width=1.5)
    fig.update_layout(
        title=f'{tax_label} - {plat_label}',
        xaxis_title='Meta Share (%)', yaxis_title='3-Point Win Rate (%)',
        height=560, margin=dict(l=70, r=30, t=60, b=60),
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5,
                    bgcolor='rgba(255,255,255,0)', bordercolor='rgba(0,0,0,0)'),
    )
    return fig


def _build_roles_cards_html(plat, taxonomy, df):
    role_col = 'meta_role' if taxonomy == 'meta' else 'tournament_role'
    palette = META_ROLE_COLORS if taxonomy == 'meta' else TOURN_ROLE_COLORS
    defs = META_ROLE_DEFS if taxonomy == 'meta' else TOURN_ROLE_DEFS
    order = META_ROLE_ORDER if taxonomy == 'meta' else TOURN_ROLE_ORDER
    aggregation = df.attrs.get('cutoffs', {}).get('aggregation', 'deck')
    # In family mode, list each family's member decks under the family row.
    members_by_family = {}
    if aggregation == 'family' and 'archetype_by_deck' in globals():
        for d, f in archetype_by_deck.items():
            members_by_family.setdefault(f, []).append(d)
    parts = ['<div class="roles-cards-grid">']
    for role in order:
        sub = df[df[role_col] == role].sort_values('wr', ascending=False)
        if len(sub) == 0:
            continue
        color = palette.get(role, UI.faint)
        row_chunks = []
        for _, r in sub.iterrows():
            members_html = ''
            if aggregation == 'family':
                ms = sorted(members_by_family.get(r['deck'], []))
                if len(ms) > 1:
                    members_html = (
                        '<span class="role-deck-members">'
                        + ', '.join(DECK_SHORT_NAMES.get(m, m) for m in ms)
                        + '</span>'
                    )
            deck_attr = '' if aggregation == 'family' else f' data-deck="{r["deck"]}"'
            row_chunks.append(
                f'<div class="role-deck-row"{deck_attr}>'
                f'<span class="role-deck-name" style="border-left-color:{r["color"]}">'
                f'<b>{r["display_name"]}</b></span>'
                f'<span class="role-deck-rationale">{r["rationale"]}</span>'
                f'{members_html}'
                f'</div>'
            )
        unit_word = 'family' if aggregation == 'family' else 'deck'
        plural = 'families' if aggregation == 'family' else 'decks'
        count_label = f'{len(sub)} {plural if len(sub) != 1 else unit_word}'
        parts.append(
            f'<section class="role-card" style="border-top-color:{color}">'
            f'<header class="role-card-header">'
            f'<span class="role-badge" style="background:{color}">{role}</span>'
            f'<span class="role-count">{count_label}</span>'
            f'</header>'
            f'<p class="role-def">{defs.get(role, "")}</p>'
            f'<div class="role-deck-list">{"".join(row_chunks)}</div>'
            f'</section>'
        )
    parts.append('</div>')
    return ''.join(parts)


def _resampled_role_inputs(plat, base0, rng):
    """One posterior draw of the inputs that depend on a finite record.

    Each unordered matchup's (win, loss, tie) split is drawn from a Dirichlet
    posterior with a uniform prior -- Dir(w+1, l+1, t+1) -- and reflected into
    the other direction so the pair stays a mirror image. The same draw then
    feeds every input it should: aggregate win rate, the popularity-weighted
    matchup spread, and the matrix the Counter rule reads. (It no longer draws the
    Upper 25% slice: no rule reads the skill delta since the section 14 audit.)

    Usage is resampled too, as a multinomial over games played.

    `p_cut` is held fixed. It comes from a 10,000-run Swiss simulation per
    platform and re-running that per draw is not affordable, so the stability
    reported here is an UPPER bound: the true assignment is at least this shaky.
    """
    decks = list(base0['deck'])
    idx = {d: i for i, d in enumerate(decks)}
    n = len(decks)
    pop = popularity_by_platform[plat].reindex(decks).fillna(0.0).to_numpy(float)

    out = {}
    for tier in ('All',):
        df = raw_by_platform[plat][tier]
        df = df[df['deck1'].isin(idx) & df['deck2'].isin(idx)
                & ~df['is_mirror'] & ~df['is_other']]
        a = df['deck1'].map(idx).to_numpy(int)
        b = df['deck2'].map(idx).to_numpy(int)
        keep = a < b
        a, b = a[keep], b[keep]
        w = df['wins'].to_numpy(float)[keep]
        l = df['losses'].to_numpy(float)[keep]
        t = df['ties'].to_numpy(float)[keep]

        p = rng.gamma(np.stack([w + 1.0, l + 1.0, t + 1.0], axis=1))
        p = p / p.sum(axis=1, keepdims=True)
        games = w + l + t
        dw, dl, dt = p[:, 0] * games, p[:, 1] * games, p[:, 2] * games

        # Simple WR matrix (what the Counter rule and mu_variance read), shrunk
        # with this slice's own estimated strength rather than a re-estimate per
        # draw, which would add noise the data does not have.
        sh = shrinkage_by_platform[plat][tier]['win_prob']
        neutral, k = float(sh['mean']), float(sh['k'])
        M = np.full((n, n), np.nan)
        with np.errstate(invalid='ignore', divide='ignore'):
            rate = np.where(games > 0, dw / np.where(games > 0, games, 1), np.nan)
        shrunk = (games * np.nan_to_num(rate) + k * neutral) / (games + k)
        M[a, b] = shrunk
        M[b, a] = 1.0 - shrunk

        # Aggregate 3-point win rate per deck, from the same draw.
        agg_w = np.zeros(n); agg_l = np.zeros(n); agg_t = np.zeros(n)
        np.add.at(agg_w, a, dw); np.add.at(agg_l, a, dl); np.add.at(agg_t, a, dt)
        np.add.at(agg_w, b, dl); np.add.at(agg_l, b, dw); np.add.at(agg_t, b, dt)
        sides = agg_w + agg_l + agg_t
        wr3 = np.where(sides > 0, (3 * agg_w + agg_t) / (3 * np.where(sides > 0, sides, 1)) * 100,
                       np.nan)
        if tier == 'All':
            # Both directions, because the family roll-up reads asymmetric rows.
            names = np.array(decks)
            raw_res = pd.DataFrame({
                'deck1': np.concatenate([names[a], names[b]]),
                'deck2': np.concatenate([names[b], names[a]]),
                'wins': np.concatenate([dw, dl]),
                'actual_total': np.concatenate([games, games]),
                'is_mirror': False, 'is_other': False,
            })
        out[tier] = (M, wr3)

    M_all, wr_all = out['All']

    # Popularity-weighted spread of a deck's matchup win rates -- the same
    # definition the consistency tab uses, recomputed on the drawn matrix.
    mu_var = np.full(n, np.nan)
    for i in range(n):
        row = M_all[i] * 100.0
        ok = np.isfinite(row) & (pop > 0)
        ok[i] = False
        if ok.sum() >= 2:
            wts = pop[ok] / pop[ok].sum()
            mean = np.average(row[ok], weights=wts)
            mu_var[i] = float(np.sqrt(np.average((row[ok] - mean) ** 2, weights=wts)))

    total_pop = pop.sum()
    usage = (rng.multinomial(int(total_pop), pop / total_pop) / total_pop * 100
             if total_pop > 0 else pop)

    b = base0.copy()
    b['wr'] = wr_all
    b['usage'] = usage
    b['mu_variance'] = mu_var
    return b, pd.DataFrame(M_all, index=decks, columns=decks), raw_res


def _family_from_draw(plat, deck_res, raw_res, fam0, weights):
    """Roll one resampled deck-level draw up to families.

    Only the columns the draw actually moves are overwritten on a copy of the
    real family frame, so display names, colours and the fixed inputs (p_cut,
    share trend) stay exactly as the tab shows them.
    """
    fam_of = deck_res['deck'].map(archetype_by_deck).fillna(deck_res['deck'])
    g = deck_res.assign(_family=fam_of, _w=weights)

    def _wavg(col, sub_df):
        v = sub_df[col].to_numpy(dtype=float)
        w = sub_df['_w'].to_numpy(dtype=float)
        m = np.isfinite(v) & (w > 0)
        return float(np.average(v[m], weights=w[m])) if m.any() else np.nan

    agg = {fam: {'usage': float(sub_df['usage'].sum()),
                 'wr': _wavg('wr', sub_df)}
           for fam, sub_df in g.groupby('_family', sort=False)}

    out = fam0.copy()
    for col in ('usage', 'wr'):
        out[col] = [agg.get(f, {}).get(col, np.nan) for f in out['deck']]
    M = _build_family_wr_matrix(plat, raw_res)
    out['mu_variance'] = _family_mu_variance(
        M, dict(zip(out['deck'], out['usage'])), out['deck'])
    return out, M


def _role_stability(plat, base0, point, aggregation='deck', n_boot=ROLE_BOOT, seed=29):
    """How often each unit keeps the role it was assigned, over `n_boot` draws.

    Takes the already-classified point estimate rather than recomputing it, so
    the baseline being compared against is exactly the one the tab displays.

    Note what is NOT resampled: `p_cut` (see above) and the share trend, which
    comes from a separate 13-week series and carries its own interval already.

    The deck view also measures how often each deck keeps its Overview tier. The
    tier score reads the input a draw moves (win rate) plus `p_cut`, so the same
    draws answer it at no extra cost -- and with the same
    upper-bound caveat. Those draws are first re-centred on the displayed values:
    the Dir(w+1, l+1, t+1) prior drags every drawn win rate toward 44% (by 0.2pp
    on average and several pp on thin slices), and against fixed letter cutoffs
    that bias would read as instability for any deck just above a line.
    """
    point = point.set_index('deck')
    rng = np.random.default_rng(seed)
    track_tier = aggregation == 'deck'
    if track_tier:
        wr_draws = []
    if aggregation == 'family':
        # Families are classified against quantiles over ~22 families, not over
        # the 25 decks, so their thresholds -- and therefore their stability --
        # are genuinely different numbers, not a restatement of the deck view.
        deck0, _, _ = _build_role_inputs(plat, 'deck')
        if deck0 is None or len(deck0) < 5:
            return None
        agg_all = analyses[plat]['agg_df']
        agg_all = agg_all[agg_all['tier'] == 'All'].set_index('deck')['total_games']
        weights = deck0['deck'].map(agg_all).fillna(1.0).astype(float).to_numpy()
    meta_hits = {d: 0 for d in point.index}
    tourn_hits = {d: 0 for d in point.index}
    for _ in range(n_boot):
        if aggregation == 'family':
            deck_res, _, raw_res = _resampled_role_inputs(plat, deck0, rng)
            b, M = _family_from_draw(plat, deck_res, raw_res, base0, weights)
        else:
            b, M, _ = _resampled_role_inputs(plat, base0, rng)
            if track_tier:
                wr_draws.append(b['wr'].to_numpy(float))
        drawn = _classify_roles(b, M, aggregation).set_index('deck')
        for d in point.index:
            if d in drawn.index:
                meta_hits[d] += int(drawn.at[d, 'meta_role'] == point.at[d, 'meta_role'])
                tourn_hits[d] += int(drawn.at[d, 'tournament_role'] == point.at[d, 'tournament_role'])
    out = pd.DataFrame({
        'deck': list(point.index),
        'meta_stability': [meta_hits[d] / n_boot * 100 for d in point.index],
        'tourn_stability': [tourn_hits[d] / n_boot * 100 for d in point.index],
    })
    if track_tier:
        stab = _tier_stability(plat, base0, np.array(wr_draws))
        out['tier_stability'] = out['deck'].map(stab)
    return out


def _tier_stability(plat, base0, wr_draws):
    """{deck: % of draws keeping the Overview tier}, from (n_boot x decks) draws.

    Each deck's draws are shifted so their mean is the displayed value: what is
    being asked is how far the record's uncertainty spreads around the number the
    table shows, not how far a prior moves it.
    """
    neutral, field_cut = _tier_baselines(plat)
    wr0 = base0['wr'].to_numpy(float)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore', RuntimeWarning)   # all-NaN columns stay NaN
        wr_c = wr_draws - np.nanmean(wr_draws, axis=0) + wr0
    p_cut = base0['p_cut'].to_numpy(float)
    out = {}
    for j, deck in enumerate(base0['deck']):
        tier0 = _tier_from_score(_tier_score(wr0[j], p_cut[j], neutral, field_cut))
        held = sum(_tier_from_score(_tier_score(wr_c[i, j], p_cut[j], neutral, field_cut)) == tier0
                   for i in range(len(wr_c)))
        out[deck] = held / len(wr_c) * 100
    return out


def _build_rules_panel_html(plat, df):
    """Emit a collapsed <details> block that lists every threshold + current
    computed cutoff so the user can audit role assignments."""
    if df is None or len(df) == 0 or 'cutoffs' not in df.attrs:
        return ''
    c = df.attrs['cutoffs']
    plat_label = PLATFORM_LABELS.get(plat, plat)
    agg_label = 'archetype family' if c['aggregation'] == 'family' else 'deck'
    rows = [
        ('Pillar',     'usage ≥ P66 AND 3pt-WR ≥ median',
         f"usage ≥ {c['usage_hi']:.2f}%, WR ≥ {c['wr_mid']:.2f}%"),
        ('Rising',     'usage ≥ P66 AND 3pt-WR < median AND share trend rising',
         f"usage ≥ {c['usage_hi']:.2f}%, WR < {c['wr_mid']:.2f}%, "
         f"velocity interval above 0 ({c['n_rising']} decks)"),
        ('Fallen',     'usage ≥ P66 AND 3pt-WR < median AND not rising',
         f"usage ≥ {c['usage_hi']:.2f}%, WR < {c['wr_mid']:.2f}%"),
        ('Sleeper',    'usage ≤ P33 AND 3pt-WR ≥ P75',
         f"usage ≤ {c['usage_lo']:.2f}%, WR ≥ {c['sleeper_wr']:.2f}%"),
        ('Counter',    f"Beats ≥{c['counter_min_pillars_beaten']} Pillars (per-Pillar WR ≥ median + 5pp); ranked by usage-weighted advantage",
         f"≥{c['counter_min_pillars_beaten']} Pillars beaten at WR ≥ {c['counter_beat_threshold']:.2f}%"),
        ('Polarizing', 'matchup σ ≥ P75',
         f"σ ≥ {c['var_hi']:.2f}"),
        ('Niche',      'usage ≤ P33 AND 3pt-WR ≥ median − 3pp',
         f"usage ≤ {c['usage_lo']:.2f}%, WR ≥ {c['wr_mid']-3:.2f}%"),
        ('Generalist', 'matchup σ < median (catch-all for low-variance leftovers)',
         f"σ < {c['mu_med']:.2f}"),
    ]
    tourn_rows = [
        ('Best-in-Class',        'highest-WR Pillar', '—'),
        ('Safe Pick',            'Pillar OR (Generalist with p_cut ≥ median p_cut)',
         f"p_cut ≥ {c['p_cut_med']:.2f}%"),
        ('Avoid',                'Fallen', '—'),
        ('Counter Pick',         'Counter, Polarizing or Sleeper (not already placed above)', '—'),
        ('Field Pick',           'everything else', '—'),
    ]
    def _badge(role, palette):
        col = palette.get(role, UI.faint)
        return f'<span class="role-badge" style="background:{col}">{role}</span>'
    meta_html = ''.join(
        f'<tr><td>{_badge(role, META_ROLE_COLORS)}</td><td>{rule}</td><td><code>{cutoff}</code></td></tr>'
        for role, rule, cutoff in rows
    )
    tourn_html = ''.join(
        f'<tr><td>{_badge(role, TOURN_ROLE_COLORS)}</td><td>{rule}</td><td><code>{cutoff}</code></td></tr>'
        for role, rule, cutoff in tourn_rows
    )
    return (
        '<details class="roles-rules">'
        '<summary>How are roles assigned? <span class="rules-hint">(click to expand)</span></summary>'
        '<div class="rules-body">'
        f'<p class="rules-intro">Thresholds derive from this platform\'s own distribution -- no hardcoded values. '
        f'Current cutoffs for <b>{plat_label}</b> ({agg_label} view):</p>'
        '<table class="rules-table"><thead><tr><th>Role</th><th>Rule</th><th>Current cutoff</th></tr></thead>'
        f'<tbody>{meta_html}</tbody></table>'
        '<p class="rules-section">Tournament strategy mapping:</p>'
        '<table class="rules-table"><thead><tr><th>Role</th><th>Rule</th><th>Current cutoff</th></tr></thead>'
        f'<tbody>{tourn_html}</tbody></table>'
        '<p class="rules-note">P33 / P66 / P75 refer to the 33rd / 66th / 75th percentiles of this platform\'s own values. '
        'Ties are folded into 3-pt WR (W=3, T=1, L=0) so the midpoint correctly accounts for draws.</p>'
        '</div></details>'
    )


# Roles run first: their bootstrap also yields the Overview's tier stability.
# Everything from here on is built per view (each platform, adjusted and raw),
# except the ledger snapshot, which records the default adjusted views only.
roles_data_per = {}
roles_scatter_html = {}
roles_cards_html = {}
roles_rules_html = {}
for _plat in VIEWS:
    roles_data_per[_plat] = {}
    roles_scatter_html[_plat] = {}
    roles_cards_html[_plat] = {}
    roles_rules_html[_plat] = {}
    for _agg in ('deck', 'family'):
        _df = _compute_meta_roles(_plat, _agg)
        roles_data_per[_plat][_agg] = _df
        roles_scatter_html[_plat][_agg] = {}
        roles_cards_html[_plat][_agg] = {}
        if _df is None or len(_df) == 0:
            roles_rules_html[_plat][_agg] = ''
            for _tax in ('meta', 'tournament'):
                roles_scatter_html[_plat][_agg][_tax] = '<p class="h2h-no-data">Insufficient data for this platform.</p>'
                roles_cards_html[_plat][_agg][_tax] = ''
            continue
        roles_rules_html[_plat][_agg] = _build_rules_panel_html(_plat, _df)
        for _tax in ('meta', 'tournament'):
            _fig = _build_roles_scatter(_plat, _tax, _df)
            if _agg == 'deck':      # a family is not one deck, so only the deck view links through
                _fig.update_layout(meta={'deckClick': True})
            roles_scatter_html[_plat][_agg][_tax] = _fig.to_html(
                full_html=False, include_plotlyjs=False,
                div_id=f'roles-scatter-{_plat}-{_agg}-{_tax}'
            )
            roles_cards_html[_plat][_agg][_tax] = _build_roles_cards_html(_plat, _tax, _df)


# Figures whose marks are single decks: a click opens that deck's detail panel.
# (The Consistency and Sleepers scatters keep their own inline breakdown, which
# links on to the panel; heatmaps and matchup bars show pairs, not one deck.)
DECK_CLICK_FIGS = {'winrate', 'swiss', 'bump', 'skill', 'skill_curve', 'skill_slope',
                   'nash', 'nash_edge', 'cut', 'cut_adj', 'late', 'late_adj', 'late_check',
                   'conv_funnel', 'conv_table'}

fig_keys = ['overview', 'winrate', 'heatmap', 'lopsided', 'swiss', 'bump', 'skill',
            'skill_mu_heatmap', 'skill_mu_top', 'sleeper', 'sleeper_table',
            'structure', 'triads', 'families', 'twins',
            'skill_curve', 'skill_slope', 'nash', 'nash_edge', 'nash_table',
            'cut', 'cut_adj', 'late', 'late_adj', 'late_check', 'conv_funnel', 'conv_table']
# Only platforms with entries are rendered; the rest appear as disabled options
# in the Data Source selector (see PLATFORM_SELECTOR_HTML below).
# A raw view's figure that comes out identical to its adjusted twin (the tab reads
# no matchup win rate) reuses the twin's div, so its tab block can be written once
# for both views. VIEWS lists each adjusted view before its raw one.
chart_divs = {plat: {} for plat in VIEWS}
_adjusted_json = {}
for plat in VIEWS:
    a = analyses[plat]
    figs = {
        'overview': _build_overview(plat),
        'winrate': a['fig_wr'],
        'heatmap': a['fig_heatmap'],
        'lopsided': a['fig_lopsided'],
        'swiss': a['fig_swiss'],
        'bump': a['fig_bump'],
        'skill': a['fig_skill'],
        'skill_mu_heatmap': a['fig_skill_mu_heatmap'],
        'skill_mu_top': a['fig_skill_mu_top'],
        'sleeper': a['fig_sleeper'],
        'sleeper_table': a['fig_sleeper_table'],
        'skill_curve': a['fig_skill_curve'],
        'skill_slope': a['fig_skill_slope'],
        'structure': a['fig_structure'],
        'triads': a['fig_triads'],
        'families': a['fig_families'],
        'twins': a['fig_twins'],
        'nash': a['fig_nash'],
        'nash_edge': a['fig_nash_edge'],
        'nash_table': a['fig_nash_table'],
        'cut': a['fig_cut'],
        'cut_adj': a['fig_cut_adj'],
        'late': a['fig_late'],
        'late_adj': a['fig_late_adj'],
        'late_check': a['fig_late_check'],
        'conv_funnel': a['fig_conv_funnel'],
        'conv_table': a['fig_conv_table'],
    }
    for key, fig in figs.items():
        if isinstance(fig, str):          # already HTML (the Overview table)
            chart_divs[plat][key] = fig
            continue
        if key in DECK_CLICK_FIGS:
            # Merged, not replaced: a figure can carry other opt-ins (the Conversion table's sortRows).
            fig.update_layout(meta={**(fig.layout.meta or {}), 'deckClick': True})
        # The sleeper scatter keeps a div per view: its click handler finds it by view id.
        if key != 'sleeper':
            if is_adjusted(plat):
                _adjusted_json[plat, key] = fig.to_json()
            elif fig.to_json() == _adjusted_json[base_platform(plat), key]:
                chart_divs[plat][key] = chart_divs[base_platform(plat)][key]
                continue
        # The sleeper scatter needs a predictable id so the click handler can
        # find it; div_id=None is to_html's default for everything else.
        chart_divs[plat][key] = fig.to_html(
            full_html=False, include_plotlyjs=False,
            div_id=(f'sleeper-scatter-{plat}' if key == 'sleeper' else None))

consist_scatter_html = {}
for plat in VIEWS:
    consist_scatter_html[plat] = analyses[plat]['fig_consist'].to_html(
        full_html=False, include_plotlyjs=False, div_id=f'consist-scatter-{plat}'
    )

consist_scatter_data_per = {plat: analyses[plat]['consist_scatter_data'] for plat in VIEWS}
matchup_breakdown_per = {plat: analyses[plat]['matchup_breakdown'] for plat in VIEWS}
sleeper_breakdown_per = {plat: analyses[plat]['sleeper_breakdown'] for plat in VIEWS}


# === Deck detail panel: every per-deck number on one card ====================
# Each tab answers one question about every deck; this collects one deck's
# answers. Built after every analysis so it can only restate what the tabs
# show -- nothing here is computed afresh. Anything a deck has no number for
# (a sleeper has no tier; a thin deck has no top-cut record) is null, and the
# panel says why rather than showing a zero.
def _js(v, nd=2):
    """JSON-safe scalar: NaN/inf -> None, numpy -> python, floats rounded."""
    if v is None:
        return None
    if isinstance(v, (bool, np.bool_)):
        return bool(v)
    if isinstance(v, (int, np.integer)):
        return int(v)
    if isinstance(v, (float, np.floating)):
        return round(float(v), nd) if np.isfinite(v) else None
    return v


def _row(df, deck, key='deck'):
    if df is None or len(df) == 0 or key not in df.columns:
        return None
    hit = df[df[key] == deck]
    return None if hit.empty else hit.iloc[0]


def _build_deck_detail(plat):
    a = analyses[plat]
    ov = _overview_frame(plat)
    roles = (roles_data_per.get(plat) or {}).get('deck')
    shares = _th_csv.read_shares(DATA_DIR, base_platform(plat))
    agg_all = a['agg_df'][a['agg_df']['tier'] == 'All']
    consist_all = a['consist_df'][a['consist_df']['tier'] == 'All']
    core_idx = {i: d for i, d in enumerate(CORE_DECKS)}
    on_platform = set(a['meta_share'].loc[a['meta_share']['meta_share'] > 0, 'deck'])
    on_platform |= set(a['sleeper_df']['deck'])
    out = {}
    for deck in [d for d in CORE_DECKS + SLEEPER_DECKS if d in on_platform]:
        fam = archetype_by_deck.get(deck)
        rec = {'deck': deck, 'short': DECK_SHORT_NAMES.get(deck, deck),
               'name': deck.replace('-', ' ').title(), 'color': DECK_COLORS.get(deck, UI.muted),
               'family': archetype_display_name.get(fam) if fam and fam != deck else None,
               'core': deck in CORE_DECKS, 'use_rate': _js(DECK_USE_RATE.get(deck)),
               'share': _js(shares.get(deck))}

        r = _row(ov, deck)
        if r is not None:
            ag = _row(agg_all, deck)
            lo = hi = None
            if ag is not None and ag['total_games'] > 0:
                lo, hi = msk.wilson(ag['wins'], ag['ties'], ag['total_games'])
                lo, hi = lo * 100, hi * 100
            rec['overview'] = {'tier': r['tier_rating'], 'score': _js(r['tier_score'], 1),
                               'held': _js(r['tier_stability'], 0), 'wr': _js(r['wr_All'], 1),
                               'wr_lo': _js(lo, 1), 'wr_hi': _js(hi, 1),
                               'games': _js(ag['total_games']) if ag is not None else None,
                               'wr_u50': _js(r['wr_Upper50'], 1), 'wr_u25': _js(r['wr_Upper25'], 1),
                               'mean_mp': _js(r['mean_mp'], 1), 'p_cut': _js(r['p_cut'], 1),
                               'skill_delta': _js(r['delta_all_u25'], 1)}
        c = _row(consist_all, deck)
        if c is not None:
            rec['consistency'] = {'fav': _js(c['fav_impact']), 'unfav': _js(c['unfav_impact']),
                                  'net': _js(c['net_impact']), 'spread': _js(c['matchup_std'], 1),
                                  'n_fav': _js(c['n_fav']), 'n_unfav': _js(c['n_unfav'])}
        rr = _row(roles, deck)
        if rr is not None:
            rec['roles'] = {'meta': rr['meta_role'], 'meta_held': _js(rr.get('meta_stability'), 0),
                            'tournament': rr['tournament_role'],
                            'tourn_held': _js(rr.get('tourn_stability'), 0),
                            'trend': _js(rr.get('trend')), 'pp_per_wk': _js(rr.get('pp_per_wk'))}
        sc = _row(a['skill_curve_df'], deck)
        if sc is not None:
            rec['skill_curve'] = {'slope': _js(sc['slope_pp_per_decade']), 'se': _js(sc['se']),
                                  'games': _js(sc['games'])}
        for key, frame in (('late', a['late_df']), ('cut', a['cut_df'])):
            s = _row(frame, deck)
            rec[key] = None if s is None else {
                'games': _js(s['games']), 'enough': _js(s['enough']), 'actual': _js(s['actual'], 1),
                'expected': _js(s['expected'], 1), 'ref': _js(s['ref_wr'], 1),
                'adj': _js(s['adj_delta'], 1), 'se': _js(s['se'], 1)}
        n = _row(a['nash_df'], deck)
        if n is not None:
            rec['nash'] = {'share': _js(n['share_pct']), 'median': _js(n['nash_median_pct']),
                           'q05': _js(n['q05']), 'q95': _js(n['q95']),
                           'support': _js(n['in_support_pct'], 0), 'p_under': _js(n['p_under'], 0),
                           'edge': _js(n['edge_pp']), 'verdict': n['verdict']}
        twins = []
        tw = a['twin_df']
        if len(tw):
            for _, t in tw.iterrows():
                d1, d2 = core_idx.get(int(t['deck_idx'])), core_idx.get(int(t['twin_idx']))
                if deck in (d1, d2):
                    other = d2 if deck == d1 else d1
                    twins.append({'deck': other, 'short': DECK_SHORT_NAMES.get(other, other),
                                  'r': _js(t['r']), 'mutual': bool(t['mutual'])})
        rec['twins'] = twins
        sl = _row(a['sleeper_df'], deck)
        if sl is not None:
            rec['sleeper'] = {'verdict': sl['verdict'], 'wr': _js(sl['wr_vs_core'], 1),
                              'lo': _js(sl['ci_lo'], 1), 'hi': _js(sl['ci_hi'], 1),
                              'coverage': _js(sl['coverage'], 0), 'would_rank': _js(sl['would_rank']),
                              'games': _js(sl['games'])}
        cv_rows = a['conv_df'][a['conv_df']['deck'] == deck] if len(a['conv_df']) else a['conv_df']
        if len(cv_rows):
            rec['conversion'] = [{'layer': r['layer'], 'label': r['label'], 'ratio': _js(r['ratio']),
                                  'lo': _js(r['lo']), 'hi': _js(r['hi']),
                                  'share': _js(r['layer_share'] * 100, 1), 'exp_share': _js(r['exp_share'] * 100, 1),
                                  'sig': r['sig'], 'reliable': bool(r['reliable'])}
                                 for _, r in cv_rows.sort_values('order').iterrows()]
        out[deck] = rec
    return out


deck_detail_per = {plat: _build_deck_detail(plat) for plat in VIEWS}
short_to_slug = {v: k for k, v in DECK_SHORT_NAMES.items()}


# === Prediction ledger: record this build, score the builds that have matured ====
# The snapshot is the deck-detail payload itself -- literally what this dashboard
# shows -- so what gets scored later can never drift from what was claimed.
import ledger as _ledger

_manifest = jsonlib.loads(FETCH_MANIFEST.read_text(encoding='utf-8'))
_ledger.write_snapshot(
    _manifest['window']['end_date'], _manifest['window'], _manifest.get('fetched_on'), LEDGER_RULES_VERSION,
    {plat: {'neutral_wr': float(analyses[plat]['neutral_wr']), 'core': list(CORE_DECKS),
            'decks': deck_detail_per[plat]} for plat in ACTIVE_PLATFORMS},
    ledger_dir=LEDGER_DIR)
# Outcomes available on disk (a historical build covering the next window) need no
# network; the fetch cell measures the rest for live builds.
for _s in _ledger.load_snapshots(LEDGER_DIR):
    if not (LEDGER_DIR / 'outcomes' / f"{_s['end']}.json").exists():
        _ledger.build_outcome(_s, LEDGER_DIR)
_ledger_trend = LEDGER_DIR / 'meta-trend.json'
LEDGER_ERAS = (_ledger.card_pool_eras(jsonlib.loads(_ledger_trend.read_text(encoding='utf-8'))['daily_trend'])
               if _ledger_trend.exists() else [])
ledger_scores = _ledger.score_all(LEDGER_ERAS, ledger_dir=LEDGER_DIR)
(LEDGER_DIR / 'scores.json').write_text(jsonlib.dumps(ledger_scores, indent=1), encoding='utf-8')

VERDICT_BADGE = {'supported': (UI.fav_bg, UI.fav_ink), 'contradicted': (UI.unfav_bg, UI.unfav_ink),
                 'inconclusive': (UI.wash, UI.text2), 'not enough data': (UI.warn_bg, UI.warn),
                 'pending': (UI.accent_wash2, UI.accent)}


def _build_track_record_html(plat):
    # The ledger scores the adjusted build a platform shows by default, in either view.
    rows = ledger_scores['platforms'].get(base_platform(plat))
    scored = [s for s in ledger_scores['snapshots'] if s['status'] == 'scored']
    head = (f'<div class="ov-title">Track Record - {PLATFORM_LABELS[plat]}</div>'
            f'<div class="ov-sub">Scored {ledger_scores["as_of"]}: {len(scored)} earlier build(s), each '
            f'compared with the window that followed it. Verdicts come from permutation tests '
            f'(p &lt; 0.025 in one direction); intervals are 90% bootstrap ranges.</div>')
    if not rows:
        body = ('<p class="h2h-no-data">Nothing to score yet on this data source: no earlier build has a '
                'finished window after it. Every build is recorded, so this fills in as time passes.</p>')
    else:
        trs = []
        for r in rows:
            bg, fg = VERDICT_BADGE.get(r['verdict'], (UI.wash, UI.text2))
            if r['stat'] is None:
                result = '&mdash;'
            elif r['unit'] == 'rank correlation':
                result = f"{r['stat']:+.2f} <span class=\"dd-na\">[{r['lo']:+.2f}, {r['hi']:+.2f}]</span>"
            else:
                result = (f"{r['stat']:+.1f} {r['unit'].replace('pp of ', 'pp ')} "
                          f"<span class=\"dd-na\">[{r['lo']:+.1f}, {r['hi']:+.1f}]</span>")
            decks = (f"{r['n']}" if r['kind'] != 'group'
                     else f"{r['n']} vs {r.get('n_rest', 0)}")
            note = f" <span class=\"dd-na\">({r['note']})</span>" if r.get('note') else ''
            trs.append(
                f"<tr><td class=\"opp-name\">{r['claim']}{note}</td><td class=\"num\">{r['pairs']}</td>"
                f"<td class=\"num\">{decks if r['kind'] != 'pending' else '&mdash;'}</td>"
                f"<td class=\"num\">{result}</td>"
                f"<td class=\"num\">{'&mdash;' if r.get('p') is None else format(r['p'], '.3f')}</td>"
                f"<td><span class=\"tier-badge\" style=\"background:{bg};color:{fg}\">{r['verdict']}</span></td></tr>")
        body = ('<div class="compare-table-wrap"><table class="compare-table">'
                '<thead><tr><th>Claim</th><th class="num">Builds</th><th class="num">Decks</th>'
                '<th class="num">Result [90%]</th><th class="num">p</th><th>Verdict</th></tr></thead>'
                f'<tbody>{"".join(trs)}</tbody></table></div>')
    snaps = ''.join(
        f"<tr><td>{s['window']['start_date']} &ndash; {s['end']}</td>"
        f"<td>{s['outcome_window']['start_date']} &ndash; {s['outcome_window']['end_date']}</td>"
        f"<td>{s['status']}</td><td class=\"dd-na\">{s['rules_version']}</td></tr>"
        for s in ledger_scores['snapshots'])
    eras = ', '.join(e['start'] for e in LEDGER_ERAS) or 'none detected'
    return (head + body +
            '<div class="compare-table-title">Recorded builds</div>'
            '<div class="compare-table-wrap"><table class="compare-table"><thead><tr><th>Build window</th>'
            '<th>Scored against</th><th>Status</th><th>Rules</th></tr></thead>'
            f'<tbody>{snaps}</tbody></table></div>'
            f'<p class="ov-sub" style="margin-top:10px">Set releases found in the trend (no score crosses one): '
            f'{eras}.</p>')


for plat in VIEWS:
    chart_divs[plat]['track'] = _build_track_record_html(plat)

# === Compare Decks tab: per-platform data payload ===
def _build_compare_data():
    out = {}
    for plat in VIEWS:
        a = analyses[plat]
        p_clean = clean_by_platform[plat]
        # Head-to-head at the All tier (one tier is enough — the card cites All;
        # skill-shift comparison reads matchup_breakdown which already has tiers).
        h2h_all = {}
        df_all = p_clean['All']
        for d1 in CORE_DECKS:
            sub = df_all[(df_all['deck1'] == d1) & df_all['deck2'].isin(CORE_DECKS)]
            row_dict = {}
            for _, r in sub.iterrows():
                n = int(r['actual_total'])
                if n == 0:
                    continue
                w, l, t = int(r['wins']), int(r['losses']), int(r['ties'])
                wr = float(r['win_rate'])
                _, ci_lo, ci_hi = wilson_ci(w, n)
                row_dict[r['deck2']] = {
                    'wr': round(wr, 1), 'wins': w, 'losses': l, 'ties': t, 'n': n,
                    'ci_lo': round(ci_lo * 100, 1), 'ci_hi': round(ci_hi * 100, 1),
                }
            h2h_all[d1] = row_dict
        # Per-deck profile across the three tiers
        profile = {}
        consist_all = a['consist_df'][a['consist_df']['tier'] == 'All']
        consist_map = {row['deck']: row for _, row in consist_all.iterrows()}
        for deck in CORE_DECKS:
            tier_data = {}
            for tier in ['All', 'Upper50', 'Upper25']:
                sub = a['agg_df'][(a['agg_df']['deck'] == deck) & (a['agg_df']['tier'] == tier)]
                if len(sub) == 0:
                    continue
                r = sub.iloc[0]
                tier_data[tier] = {
                    'wr': round(float(r['wr_3pt']), 1),
                    'simple_wr': round(float(r['simple_wr']), 1),
                    'ci_lo': round(float(r['ci_lo']), 1),
                    'ci_hi': round(float(r['ci_hi']), 1),
                    'games': int(round(r['total_games'])),   # tier counts can be fractional
                }
            if not tier_data:
                continue
            cr = consist_map.get(deck)
            def _safe(v):
                return round(float(v), 2) if cr is not None and not pd.isna(v) else None
            profile[deck] = {
                'short': DECK_SHORT_NAMES[deck],
                'color': DECK_COLORS[deck],
                'tiers': tier_data,
                'fav_impact': _safe(cr['fav_impact']) if cr is not None else None,
                'unfav_impact': _safe(cr['unfav_impact']) if cr is not None else None,
            }
        decks_in = sorted(
            profile.keys(),
            key=lambda d: -profile[d]['tiers'].get('All', {}).get('games', 0),
        )
        out[plat] = {'h2h': h2h_all, 'profile': profile, 'decks': decks_in}
    return out

compare_data_per = _build_compare_data()


# === Meta Simulator: per-platform payload ====================================
# Everything the browser needs to answer "if the field looked like THIS, how
# would each deck do?": the tie-aware smoothed matchup matrix per finishing tier
# plus the real use rates, which is what the Default button restores.
SIM_TIERS = ['All', 'Upper50', 'Upper25']


def _build_simulator_data():
    out = {}
    for plat in VIEWS:
        # Games played tells us which decks appear on this platform; the field
        # composition itself is use rate (games played is not use rate -- decks
        # that convert into the cut play more matches per entry).
        pop = popularity_by_platform[plat]
        decks = [d for d in ranked_decks if float(pop.get(d, 0)) > 0]
        if not decks:
            continue
        use = DECK_USE_RATE.reindex(decks).fillna(0.0)
        use_total = float(use.sum())
        wr_per_tier, games_per_tier = {}, {}
        for tier in SIM_TIERS:
            mats = matrices_by_platform[plat][tier]
            wr_m, n_m = mats['smoothed_wr3'], mats['sample_n']
            mirror = mirror_by_platform[plat][tier]
            mirror_wr, mirror_n = mirror['baseline'], mirror['games']
            wr_rows, n_rows = {}, {}
            for d in decks:
                row_wr, row_n = {}, {}
                for opp in decks:
                    if opp == d:
                        # A mirror is a real matchup in a simulated field, and ties
                        # push it off 50%, so use the deck's own mirror record --
                        # sample-adjusted in both views (break-even lines are shared).
                        if d in mirror_wr:
                            row_wr[opp] = round(mirror_wr[d], 2)
                            row_n[opp] = mirror_n[d]
                        continue
                    v, n = wr_m.loc[d, opp], n_m.loc[d, opp]
                    if pd.isna(v) or pd.isna(n) or n <= 0:
                        continue
                    row_wr[opp] = round(float(v) * 100, 2)
                    row_n[opp] = int(round(n))
                wr_rows[d], n_rows[d] = row_wr, row_n
            wr_per_tier[tier], games_per_tier[tier] = wr_rows, n_rows
        out[plat] = {
            'decks': decks,
            'names': {d: DECK_SHORT_NAMES.get(d, d) for d in decks},
            'colors': {d: DECK_COLORS.get(d, UI.muted) for d in decks},
            'core': [d for d in decks if d in CORE_DECKS],
            'usage': {d: (round(float(use[d]) / use_total * 100, 2) if use_total > 0 else 0.0)
                      for d in decks},
            'wr': wr_per_tier,
            'games': games_per_tier,
            'minSample': MIN_SAMPLE,
        }
    return out


simulator_data_per = _build_simulator_data()

SIMULATOR_HTML = '''
<div class="sim-controls">
  <div class="sim-control-group">
    <span class="sim-label">Input mode</span>
    <div class="sim-mode-toggle">
      <button class="simmode-btn active" data-mode="percentage" onclick="setSimMode('percentage')">Percentage</button>
      <button class="simmode-btn" data-mode="weighted" onclick="setSimMode('weighted')">Weighted</button>
    </div>
  </div>
  <div class="sim-control-group">
    <label class="sim-label" for="sim-tier-select">Win Rate Source</label>
    <select id="sim-tier-select" class="consist-select">
      <option value="All" selected>All Players</option>
      <option value="Upper50">Top 50%</option>
      <option value="Upper25">Top 25%</option>
    </select>
  </div>
  <label class="sim-toggle-label">
    <input type="checkbox" id="sim-only-field"><span>Only decks in the field</span>
  </label>
  <div class="sim-control-group sim-buttons">
    <button class="sim-btn sim-btn-clear" onclick="simClear()">Clear</button>
    <button class="sim-btn sim-btn-default" onclick="simDefault()">Default</button>
  </div>
</div>
<div class="sim-status" id="sim-status"></div>
<div class="sim-layout">
  <div class="sim-panel">
    <div class="sim-panel-title">Field composition</div>
    <div id="sim-inputs" class="sim-inputs"></div>
  </div>
  <div class="sim-panel">
    <div class="sim-panel-title">Expected performance against that field</div>
    <div id="sim-results"></div>
  </div>
</div>
'''


CONSIST_SELECTOR_HTML = (
    '<div class="consist-controls">\n'
    '  <label for="consist-tier-select" class="consist-label">Win Rate Source</label>\n'
    '  <select id="consist-tier-select" class="consist-select">\n'
    '    <option value="All" selected>All Players</option>\n'
    '    <option value="Upper50">Top 50%</option>\n'
    '    <option value="Upper25">Top 25%</option>\n'
    '    <option value="Weighted">Weighted toward top finishers</option>\n'
    '  </select>\n'
    '</div>\n'
)
CONSIST_DETAIL_HTML = (
    '<div id="consist-detail-panel">\n'
    '  <div class="detail-head"><h3 id="consist-detail-title"></h3>'
    '<button type="button" class="deck-link-btn" id="consist-detail-open" data-deck="">'
    'Open deck profile &rarr;</button></div>\n'
    '  <div id="consist-detail-chart"></div>\n'
    '</div>\n'
)
# Same panel, same styling, for the Sleepers tab. One per document (not per
# platform) so the ids stay unique -- applyPlatform hides it on a switch.
SLEEPER_DETAIL_HTML = (
    '<div id="sleeper-detail-panel">\n'
    '  <div class="detail-head"><h3 id="sleeper-detail-title"></h3>'
    '<button type="button" class="deck-link-btn" id="sleeper-detail-open" data-deck="">'
    'Open deck profile &rarr;</button></div>\n'
    '  <div id="sleeper-detail-chart"></div>\n'
    '</div>\n'
)

COMPARE_HTML = '''
<div class="compare-controls">
  <label class="compare-label" for="compare-deck-a">Deck A</label>
  <select id="compare-deck-a" class="compare-select"></select>
  <span class="compare-vs">vs</span>
  <label class="compare-label" for="compare-deck-b">Deck B</label>
  <select id="compare-deck-b" class="compare-select"></select>
  <span class="compare-toggle-wrap">
    <label class="compare-toggle-label">
      <input type="checkbox" id="compare-diff-toggle">
      <span>Show difference (A &minus; B)</span>
    </label>
    <label class="compare-toggle-label">
      <input type="checkbox" id="compare-skill-toggle">
      <span>Show top-finisher shift (Top 25% &minus; All)</span>
    </label>
  </span>
</div>
<div class="compare-h2h-card">
  <div class="compare-h2h-title" id="compare-h2h-title"></div>
  <div class="compare-h2h-body" id="compare-h2h-body"></div>
</div>
<div class="compare-profile-row">
  <div class="profile-card" id="compare-profile-a"></div>
  <div class="profile-card" id="compare-profile-b"></div>
</div>
<div id="compare-divergence-chart"></div>
<div id="compare-skilltier-chart" style="margin-top: 18px;"></div>
'''


# Meta Roles tab: nested toggles -- Aggregation (Deck/Family) wraps Taxonomy
# (Meta Position/Tournament Strategy) wraps the per-platform content blocks
# that the existing applyPlatform() handler flips by data-platform attribute.
_roles_agg_toggle = (
    '<div class="roles-agg-toggle">'
    '<button class="aggtab-btn active" data-agg="deck" onclick="switchRolesAggregation(\'deck\')">By Deck</button>'
    '<button class="aggtab-btn" data-agg="family" onclick="switchRolesAggregation(\'family\')">By Family</button>'
    '</div>'
)
_roles_tax_toggle = (
    '<div class="roles-taxonomy-toggle">'
    '<button class="subtab-btn active" data-tax="meta" onclick="switchRolesTaxonomy(\'meta\')">Meta Position</button>'
    '<button class="subtab-btn" data-tax="tournament" onclick="switchRolesTaxonomy(\'tournament\')">Tournament Strategy</button>'
    '</div>'
)
_roles_agg_blocks = ''
for _agg in ('deck', 'family'):
    _agg_display = 'block' if _agg == 'deck' else 'none'
    _tax_blocks = ''
    for _tax in ('meta', 'tournament'):
        _tax_display = 'block' if _tax == 'meta' else 'none'
        _platform_blocks = ''
        for _plat in VIEWS:
            _plat_display = 'block' if _plat == DEFAULT_PLATFORM else 'none'
            _scatter = roles_scatter_html[_plat][_agg][_tax]
            _cards = roles_cards_html[_plat][_agg][_tax]
            _rules = roles_rules_html[_plat][_agg] if _tax == 'meta' else ''
            # Rules panel sits inside the meta-tax container only, to avoid duplication.
            _platform_blocks += (
                f'<div class="platform-content" data-platform="{_plat}" '
                f'style="display:{_plat_display}">{_rules}{_scatter}'
                f'<div class="roles-cards-wrap">{_cards}</div></div>'
            )
        _tax_blocks += (
            f'<div class="roles-taxonomy-content" data-tax="{_tax}" '
            f'style="display:{_tax_display}">{_platform_blocks}</div>'
        )
    _roles_agg_blocks += (
        f'<div class="roles-aggregation-content" data-agg="{_agg}" '
        f'style="display:{_agg_display}">{_roles_tax_toggle}{_tax_blocks}</div>'
    )
ROLES_HTML = _roles_agg_toggle + _roles_agg_blocks


# Every finishing-tier view carries the section 14 audit result with it.
FINISH_TIER_NOTE = ('A game counts in Top 25% when both players finished in the top quarter of their '
                    'event\'s final standings (their division, at a major), ranked as on the Conversion tab: '
                    'players who dropped rank below every finisher, by match points. '
                    'Top-tier win rates describe players who FINISHED there, who were selected on having '
                    'won, so every deck looks better in them. Checked month against month inside one card '
                    'pool, the difference between decks does not persist (experiments notebook, section 14), '
                    'so this is description, not a measure of which decks reward skill -- and no tier or role '
                    'uses it.')

tab_config = [
    ('Overview', ['overview'], 'Meta summary table with tier ratings based on composite performance score. '
     'Each input is measured against a line from this platform\'s own data -- the tie-aware break-even '
     'and the field\'s average cut rate -- rather than a flat 50% and 20%. "Tier held" is how often the '
     'deck keeps its tier when its matchup records are resampled: a tier held in under half of draws '
     'is a coin flip between two letters, not a rating.'),
    ('Win Rate', ['winrate'], 'Aggregate win rates for all players and among top-50% / top-25% finishers, with Wilson '
     'confidence intervals. ' + FINISH_TIER_NOTE),
    ('Matchups', ['heatmap', 'lopsided'], 'Matchup heatmap and most lopsided matchups.'),
    ('Consistency', ['__consist_custom__'], 'Favorable vs unfavorable matchup impact, weighted by opponent popularity. '
     'The line between the two is each deck\'s own mirror rate rather than a flat 50%: ties put an even matchup below 50%, and decks differ in how often they draw. '
     'A thin mirror record is pulled toward the format prior. Click a deck dot for detailed breakdown.'),
    ('Swiss Sim', ['swiss'], f'Simulated Swiss tournament results ({N_SIMS:,} runs, {SWISS_ROUNDS} rounds, {CUT_THRESHOLD} MP cut).'),
    ('Conversion', ['conv_funnel', 'conv_table'],
     'Does a deck reach each finishing layer more often than its play predicts? Each ratio is the '
     'deck\'s finishes at that layer over the finishes its players would expect if deck choice did '
     'not matter, event by event, from the final standings of every event in this window. For a '
     'percent layer that is simply its share of the layer over its share of play. For a Top N layer, '
     'matching event by event matters: every event hands out a Top 8 whatever its size, so a plain '
     '"share of Top 8 over use rate" penalises decks that are popular at big events. Top N only counts '
     'events of at least 2N players, and layers are ordered by how deep they actually are -- Top 64 in '
     '128+ player events can hold more of the field than Upper 25%. Ratios are pulled toward 1.0 in '
     'proportion to how thin the record is, and a layer is greyed when its ratios in one random half '
     'of the events fail to predict the other half. ' + CONVERSION_FINDING),
    ('Finishing Tiers', ['bump', 'skill'], 'How each deck\'s win rate and rank change from all players to top-25% '
     'finishers. ' + FINISH_TIER_NOTE),
    ('Matchups by Finish', ['skill_mu_heatmap', 'skill_mu_top'], 'How individual matchup win rates shift from all '
     'players to top-25% finishers. ' + FINISH_TIER_NOTE),
    ('Finish Curve', ['skill_curve', 'skill_slope'],
     'Win rate as a continuous function of finishing position, using five disjoint placement '
     'bands (0-5, 5-10, 10-25, 25-50, 50-100) rather than the two overlapping slices the Finishing '
     'Tiers tab compares. Bands are built by differencing nested top-N% cuts, so each deck-game '
     'is counted in exactly one band and the uncertainty is honest. The slope rests on 3-5 points, so '
     'weigh it against the per-band intervals and standard errors. ' + FINISH_TIER_NOTE),
    ('Late Rounds', ['late', 'late_adj', 'late_check'],
     f'A skill measure that does not select on results. The finishing-tier tabs compare players by '
     f'final standings, and those standings include the very games being measured, so a deck '
     f'looks better in Upper 25% partly because winning is how you get there. Here the split is '
     f'Swiss round {LATE_ROUND}+ against earlier rounds, inside the same events '
     f'({LATE_MIN_PLAYERS}+ players, so every event reaches round {LATE_ROUND}), over {LONG_DAYS} days. '
     'Each deck\'s late games are priced at its early-round matchup rates against the opponents it '
     'actually met, so facing tougher decks late is not read as a drop. Win rates are over decided '
     'games: final rounds are full of intentional draws into the cut. Two caveats: online players '
     'with poor records drop before the late rounds, and Swiss pairs late opponents by record, so '
     'this measures whether a deck holds up among players still in contention -- not deck '
     'strength. The last chart checks whether the finishing-tier measures agree.'),
    ('Meta Roles', ['__roles_custom__'], 'Decks grouped by role in the format. "Meta Position" describes what each deck IS; "Tournament Strategy" suggests how to USE it. Thresholds derive from this platform\'s own data.'),
    ('Compare Decks', ['__compare_custom__'], 'Side-by-side comparison: head-to-head record, deck profiles, '
     'common-opponent divergence, and win rate across finishing tiers (a description of top finishers, '
     'not a skill measure).'),
    ('Sleepers', ['sleeper', 'sleeper_table'], 'Decks outside the core meta by use rate, scored purely on their matchups against it. '
     'Win rates are usage-weighted and tie-aware (3-pt); the bar to clear is the core decks\' own median, and coverage shows how much of the field the record actually rests on. '
     'Favorable and unfavorable impact split those matchups at the format break-even, weighted by core meta share; click a bubble for the full breakdown.'),
    ('Mispriced Decks', ['nash', 'nash_edge', 'nash_table'],
     'Deck choice is a game: its Nash equilibrium is the field no deck can profitably deviate '
     'against. Where the equilibrium wants more of a deck than players actually register, the '
     'field is underrating it -- that, not raw win rate, is where a deck-choice edge lives. '
     f'A single equilibrium solution is a corner of a linear program and will hang a big weight on '
     f'a deck whose record is mostly guesswork, so every weight shown is a median over {NASH_BOOT} '
     f'resampled matchup matrices, with its 90% range and how often the deck is in the equilibrium '
     f'at all. A verdict needs {NASH_CONFIDENCE:.0f}% of draws on the same side of actual share. Decks with '
     f'{NASH_MIN_GAMES}+ games on the platform, from all {TOP_N} tracked rather than the core 25; '
     'share is Trainer Hill\'s share of players for the same window.'),
    ('Top Cut', ['cut', 'cut_adj'],
     f'Does a deck win more or less once the cut starts? Over {LONG_DAYS} days, each deck\'s top-cut '
     'games are compared with the Swiss games of the same events -- but not with its Swiss win rate. '
     'A cut bracket is full of the decks that made it there, so a deck can "drop" in the cut simply '
     'by meeting its bad matchups more often. Its cut games are priced opponent by opponent at the '
     'Swiss matchup rates, and only the leftover counts. Win rates are over decided games, since cut '
     f'and Swiss treat draws differently. Online events cut in phase 2; majors play a second Swiss day '
     f'first and cut in phase 3, which is why in-person has almost no cut games. Decks with '
     f'{MIN_CUT_GAMES}+ decided cut games.'),
    ('Format Shape', ['structure', 'triads'],
     'Splits the matchup matrix into the closest power ranking plus the rock-paper-scissors '
     'no ranking can express. The cyclic share is what makes deck choice a matchup bet rather '
     'than a strength contest; it has no external reference point, so read it across tiers '
     'rather than as an absolute. The triangles are the specific loops driving it.'),
    ('Deck Families', ['families', 'twins'],
     'Groups decks by their matchup profile - what they beat and lose to - rather than by '
     'their cards. Two decks with correlated profiles are near-interchangeable answers to the '
     'format, so the choice between them is preference rather than strategy. No name or card '
     'information is used, which is why recovering known variants of one archetype is a check '
     'on the method rather than a finding.'),
    ('Meta Simulator', ['__simulator_custom__'], 'Build a hypothetical field and see how every deck would do against it. '
     'Percentage mode treats your numbers as shares of the field; weighted mode normalises whatever relative weights you enter.'),
    ('Track Record', ['track'],
     'Does this dashboard\'s advice hold up? Every build records the verdicts it shows. Once the '
     f'{FETCH_DAYS}-day window after a build has finished (plus {_ledger.LAG_DAYS} days for late-reported '
     'events), each claim is checked against what actually happened: do higher tiers win more next '
     'window, do underplayed decks outperform, do Rising decks keep gaining? The claims and tests '
     'were fixed before any outcome was looked at. A build is never scored across a set release, '
     'overlapping daily rebuilds count once, and the win rate\'s own persistence is scored as a '
     'positive control. "Inconclusive" means the evidence so far cannot tell; it is not a pass.'),
]

# Every per-matchup win rate below is sample-adjusted; say so where it matters.
_shrink_ref = analyses[DEFAULT_PLATFORM]
_shrink_note = (f"Win rates are sample-adjusted: each matchup is pulled toward the format's "
                f"tie-aware break-even of {_shrink_ref['neutral_wr']:.1f}% (not 50%) with a weight "
                f"of {_shrink_ref['shrink_k']:.0f} games, estimated from how much of the spread "
                f"between matchups is real rather than sampling noise. A thin record therefore "
                f"can't masquerade as a blowout -- and because the target is the break-even line "
                f"itself, the adjustment can never turn a losing matchup into a winning one.")
# What the "Raw win rates" checkbox shows instead; the page swaps the two notes.
_raw_note = ("Raw win rates: every matchup is exactly its record, with no sample-size adjustment, so a "
             "thin record can read as a blowout. Everything built on the matchups (simulations, tiers, "
             "roles) is recomputed from the raw rates. Each deck's break-even line, from its mirror "
             "record, keeps its adjustment.")
_wr_note = (f'<span class="wr-note-adj">{_shrink_note}</span>'
            f'<span class="wr-note-raw">{_raw_note}</span>')
_SHRINK_TABS = {'Matchups', 'Consistency', 'Swiss Sim', 'Matchups by Finish', 'Sleepers',
                'Meta Simulator', 'Format Shape', 'Deck Families',
                'Finish Curve', 'Mispriced Decks'}
tab_config = [(label, keys, (desc + ' ' + _wr_note) if label in _SHRINK_TABS else desc)
              for label, keys, desc in tab_config]

# Navigation: five groups, each with its own row of tabs. Tab content keeps the
# tab_config order (ids tab0..), so only the buttons are regrouped; links use the
# label's slug (#win-rate), which does not move when tabs are reordered.
NAV_GROUPS = [
    ('Meta', ['Overview', 'Win Rate', 'Meta Roles', 'Format Shape', 'Deck Families', 'Track Record']),
    ('Matchups', ['Matchups', 'Consistency', 'Compare Decks', 'Matchups by Finish']),
    ('Tournament', ['Swiss Sim', 'Meta Simulator', 'Conversion', 'Top Cut']),
    ('Finishes', ['Finishing Tiers', 'Finish Curve', 'Late Rounds']),
    ('Edges', ['Sleepers', 'Mispriced Decks']),
]
_tab_index = {label: i for i, (label, _, _) in enumerate(tab_config)}
_grouped = [label for _, labels in NAV_GROUPS for label in labels]
if sorted(_grouped) != sorted(_tab_index) or len(set(_grouped)) != len(_grouped):
    raise ValueError('NAV_GROUPS must list every tab exactly once: '
                     f'missing {sorted(set(_tab_index) - set(_grouped))}, '
                     f'unknown {sorted(set(_grouped) - set(_tab_index))}')
_slug = lambda s: re.sub(r'[^a-z0-9]+', '-', s.lower()).strip('-')
_first_tab = _tab_index[NAV_GROUPS[0][1][0]]

_nav_groups_html, _nav_rows_html = '', ''
for g, (group, labels) in enumerate(NAV_GROUPS):
    on = ' active' if g == 0 else ''
    _nav_groups_html += (f'<button type="button" class="nav-group{on}" data-group="{_slug(group)}" '
                         f'onclick="openGroup(\'{_slug(group)}\')">{group}</button>')
    _btns = ''.join(
        f'<button type="button" class="tab-btn{" active" if _tab_index[l] == _first_tab else ""}" '
        f'data-tab="tab{_tab_index[l]}" data-slug="{_slug(l)}" data-group="{_slug(group)}" '
        f'onclick="openTab(event, \'tab{_tab_index[l]}\')">{l}</button>' for l in labels)
    _nav_rows_html += f'<div class="nav-tabs{on}" data-group="{_slug(group)}">{_btns}</div>\n'
NAV_HTML = (f'<nav class="nav" aria-label="Dashboard sections"><div class="nav-groups">{_nav_groups_html}</div>\n'
            f'{_nav_rows_html}</nav>\n')

tab_content_html = ''
for i, (label, keys, desc) in enumerate(tab_config):
    tab_display = 'block' if i == _first_tab else 'none'
    if '__compare_custom__' in keys:
        # Compare-Decks body is a single shared block; data swaps via JS on platform change.
        body = COMPARE_HTML
    elif '__simulator_custom__' in keys:
        # Same shared-block pattern: one simulator, repopulated per platform.
        body = SIMULATOR_HTML
    elif '__roles_custom__' in keys:
        body = ROLES_HTML
    else:
        per_platform_blocks = ''
        for plat in ACTIVE_PLATFORMS:
            views = [plat, raw_view(plat)]
            if '__consist_custom__' in keys:
                charts_html = {v: consist_scatter_html[v] for v in views}  # only the scatter is per-platform
            else:
                charts_html = {v: ''.join(chart_divs[v][k] for k in keys) for v in views}
            # A tab that reads no matchup win rate is written once, for both views
            # (data-platform is a space-separated list of the views it serves).
            groups = [views] if charts_html[views[0]] == charts_html[views[1]] else [[v] for v in views]
            for group in groups:
                plat_display = 'block' if DEFAULT_PLATFORM in group else 'none'
                per_platform_blocks += (
                    f'<div class="platform-content" data-platform="{" ".join(group)}" '
                    f'style="display:{plat_display}">{charts_html[group[0]]}</div>\n'
                )
        # Consistency tab wraps the platform blocks with a single shared
        # tier-selector + detail panel so IDs are unique across the document.
        if '__consist_custom__' in keys:
            body = CONSIST_SELECTOR_HTML + per_platform_blocks + CONSIST_DETAIL_HTML
        elif 'sleeper' in keys:
            body = per_platform_blocks + SLEEPER_DETAIL_HTML
        else:
            body = per_platform_blocks
    tab_content_html += (
        f'<div id="tab{i}" class="tab-content" style="display:{tab_display}">'
        f'<p class="tab-desc">{desc}</p>{body}</div>\n'
    )

# Options come from the data: a platform with no entries stays listed but
# disabled, so its absence reads as "no data" rather than a missing feature.
_platform_options = ''
for _p in PLATFORMS:
    _p_label = PLATFORM_LABELS.get(_p, _p)
    if _p in ACTIVE_PLATFORMS:
        _selected = ' selected' if _p == DEFAULT_PLATFORM else ''
        _platform_options += f'    <option value="{_p}"{_selected}>{_p_label}</option>\n'
    else:
        _platform_options += f'    <option value="{_p}" disabled>{_p_label} (no data)</option>\n'

PLATFORM_SELECTOR_HTML = (
    '<div class="platform-selector">\n'
    '  <label for="platform-select" class="platform-label">Data Source</label>\n'
    '  <select id="platform-select" class="platform-select">\n'
    f'{_platform_options}'
    '  </select>\n'
    '  <label class="wr-raw-toggle" title="Show every matchup at its raw record, with no sample-size '
    'adjustment. Simulations, tiers and roles are recomputed from the raw rates.">\n'
    '    <input type="checkbox" id="raw-wr-toggle"><span>Raw win rates</span>\n'
    '  </label>\n'
    '  <span class="deck-picker">\n'
    '    <label for="deck-detail-select" class="platform-label">Deck profile</label>\n'
    '    <select id="deck-detail-select" class="platform-select"></select>\n'
    '  </span>\n'
    '</div>\n'
)

# One drawer for the whole document; its content is rendered from
# deckDetailPer[currentPlatform] whenever it opens or the platform changes.
DECK_DRAWER_HTML = (
    '<div id="deck-drawer-backdrop"></div>\n'
    '<aside id="deck-drawer" aria-hidden="true" aria-labelledby="deck-drawer-title">\n'
    '  <div class="dd-head">\n'
    '    <div><div id="deck-drawer-title"></div><div id="deck-drawer-sub"></div></div>\n'
    '    <button id="deck-drawer-close" type="button" aria-label="Close deck profile">&times;</button>\n'
    '  </div>\n'
    '  <div id="deck-drawer-body"></div>\n'
    '  <div class="dd-section-title" id="deck-drawer-chart-title"></div>\n'
    '  <div id="deck-drawer-chart"></div>\n'
    '</aside>\n'
)

n_decks_total = len(analysis_decks)
_deck_note = (f'{len(CORE_DECKS)} core decks analysed'
              + (f' + {len(SLEEPER_DECKS)} sleeper candidates' if SLEEPER_DECKS else ''))
_empty_note = ('' if not EMPTY_PLATFORMS else ' &middot; No entries for ' +
               ', '.join(PLATFORM_LABELS.get(p, p) for p in EMPTY_PLATFORMS))
mb_json = jsonlib.dumps(matchup_breakdown_per)
sb_json = jsonlib.dumps(sleeper_breakdown_per)
csd_json = jsonlib.dumps(consist_scatter_data_per)
cd_json = jsonlib.dumps(compare_data_per)

# Build the HTML in pieces to avoid f-string brace headaches with CSS/JS.
HEAD_HTML = (
    '<!DOCTYPE html>\n<html lang="en">\n<head>\n'
    '<meta charset="utf-8">\n'
    '<meta name="viewport" content="width=device-width, initial-scale=1">\n'
    '<title>Pokemon TCG Meta Analyzer</title>\n'
    '<link rel="preconnect" href="https://fonts.googleapis.com">\n'
    '<link rel="preconnect" href="https://fonts.gstatic.com" crossorigin>\n'
    '<link href="https://fonts.googleapis.com/css2?family=Barlow:wght@400;500;600;700'
    '&family=Barlow+Condensed:wght@600;700&display=swap" rel="stylesheet">\n'
    '<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>\n'
)

CSS_HTML = """<style>
  *, *::before, *::after { box-sizing: border-box; }
  html { -webkit-text-size-adjust: 100%; }
  body { font-family: var(--font); margin: 0; padding: 0; overflow-x: hidden;
         background: var(--bg); color: var(--text); line-height: 1.5; }
  .container { max-width: 1400px; margin: 0 auto; padding: 20px 32px 32px; }
  /* --- Tournament shell: navy band, yellow stripe, condensed caps --------- */
  .site-header { background: linear-gradient(120deg, var(--navy), var(--navy2)); color: #fff;
                 border-bottom: 4px solid var(--stripe); }
  .site-header-inner { max-width: 1400px; margin: 0 auto; padding: 20px 32px 16px; }
  .header-top { display: flex; flex-wrap: wrap; align-items: flex-end; justify-content: space-between;
                gap: 10px 24px; margin-bottom: 14px; }
  .brand h1 { font-family: var(--font-display); font-weight: 700; font-size: 32px; line-height: 1.05;
              text-transform: uppercase; letter-spacing: 0.04em; margin: 0; color: #fff; }
  .brand h1 .mark { color: var(--stripe); }
  .brand .subtitle { color: var(--on-navy); font-size: 13px; margin: 4px 0 0; }
  .header-meta { text-align: right; font-size: 12px; color: var(--on-navy); }
  .header-meta .through { display: block; font-family: var(--font-display); font-weight: 600; font-size: 16px;
                          text-transform: uppercase; letter-spacing: 0.06em; color: var(--stripe); }
  .header-meta a { color: #fff; }
  .phone-link { display: inline-block; margin-top: 6px; padding: 4px 10px; color: #fff; text-decoration: none;
                border: 1px solid rgba(255,255,255,0.35); border-radius: var(--radius);
                font-family: var(--font-display); font-weight: 600; text-transform: uppercase; letter-spacing: 0.05em; }
  .phone-link:hover { background: rgba(255,255,255,0.12); }
  .platform-selector { display: flex; flex-wrap: wrap; align-items: center; gap: 10px 14px; }
  .platform-label { font-family: var(--font-display); font-size: 14px; font-weight: 600; color: var(--on-navy);
                    text-transform: uppercase; letter-spacing: 0.06em; }
  .tab-content .platform-label, .consist-controls .platform-label { color: var(--text); }
  .site-header .platform-select { background-color: #fff; border-color: transparent; }
  .site-footer { max-width: 1400px; margin: 0 auto; padding: 0 32px 28px; font-size: 12px; color: var(--muted); }
  .site-footer a { color: var(--accent); }
  /* --- Grouped navigation -------------------------------------------------- */
  .nav { margin: 0 0 18px; }
  .nav-groups { display: flex; flex-wrap: wrap; gap: 4px 0; }
  .nav-group { font-family: var(--font-display); font-weight: 700; font-size: 17px; letter-spacing: 0.05em;
               text-transform: uppercase; border: 0; margin: 0 -2px 0 0; padding: 9px 24px; cursor: pointer;
               background: var(--border); color: var(--text);
               clip-path: polygon(10px 0, 100% 0, calc(100% - 10px) 100%, 0 100%); }
  .nav-group:hover { background: var(--accent-wash2); color: var(--ink); }
  .nav-group.active { background: var(--accent); color: #fff; }
  .nav-group:focus-visible, .tab-btn:focus-visible { outline: 2px solid var(--stripe); outline-offset: 2px; }
  .nav-tabs { display: none; gap: 2px; padding-top: 6px; border-bottom: 2px solid var(--border);
              overflow-x: auto; scrollbar-width: thin; }
  .nav-tabs.active { display: flex; }
  .platform-content { overflow-x: auto; }
  .platform-select, .consist-select {
    font-family: var(--font); font-size: 13px; font-weight: 500;
    color: var(--ink); background: var(--bg); border: 1px solid var(--rule);
    border-radius: var(--radius); padding: 8px 32px 8px 14px; cursor: pointer;
    appearance: none; -webkit-appearance: none;
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='12' viewBox='0 0 12 12'%3E%3Cpath fill='%2364748b' d='M2 4l4 4 4-4'/%3E%3C/svg%3E");
    background-repeat: no-repeat; background-position: right 12px center;
    transition: border-color 0.2s, box-shadow 0.2s;
  }
  .platform-select:hover, .consist-select:hover { border-color: var(--accent); }
  .platform-select:focus, .consist-select:focus { outline: none; border-color: var(--accent);
    box-shadow: 0 0 0 3px var(--accent-ring); }
  .wr-raw-toggle { display: flex; align-items: center; gap: 6px; cursor: pointer; user-select: none;
    font-family: var(--font-display); font-size: 14px; font-weight: 600; color: #fff;
    text-transform: uppercase; letter-spacing: 0.06em; }
  .wr-raw-toggle input { width: 15px; height: 15px; margin: 0; accent-color: var(--stripe); cursor: pointer; }
  body.raw-wr .wr-raw-toggle { color: var(--stripe); }
  .wr-note-raw { display: none; }
  body.raw-wr .wr-note-adj { display: none; }
  body.raw-wr .wr-note-raw { display: inline; }
  .tab-btn { font-family: var(--font-display); font-weight: 600; font-size: 15px; letter-spacing: 0.04em;
             text-transform: uppercase; padding: 8px 14px; margin-bottom: -2px; border: 0;
             border-bottom: 3px solid transparent; background: transparent; color: var(--muted);
             cursor: pointer; white-space: nowrap; }
  .tab-btn:hover { color: var(--ink); }
  .tab-btn.active { color: var(--accent); border-bottom-color: var(--accent); }
  .tab-content { background: var(--surface); padding: 24px 28px; border-radius: var(--radius);
                 border: 1px solid var(--border); border-top: 3px solid var(--navy); }
  .tab-desc { color: var(--muted); font-size: 13px; margin: 0 0 20px;
              padding: 10px 16px; border-left: 4px solid var(--stripe);
              background: var(--bg); border-radius: var(--radius); line-height: 1.6; }
  @media (max-width: 720px) {
    .container { padding: 12px 16px 24px; }
    .site-header-inner { padding: 14px 16px 12px; }
    .site-footer { padding: 0 16px 24px; }
    .brand h1 { font-size: 24px; }
    .header-meta { text-align: left; }
    .tab-content { padding: 14px 12px; }
    .nav-group { font-size: 15px; padding: 8px 16px; }
    .deck-picker { margin-left: 0; }
    .platform-select { max-width: calc(100vw - 32px); }
    /* Figures are laid out for a desktop; below this width they would crush their
       labels, so each keeps a readable width and scrolls inside its own box. */
    .platform-content .plotly-graph-div { min-width: 680px; }
  }
  .consist-controls { display: flex; align-items: center; gap: 12px; margin-bottom: 16px; }
  .consist-label { font-size: 13px; font-weight: 500; color: var(--text2); }
  #consist-detail-panel, #sleeper-detail-panel { margin-top: 24px; display: none; background: var(--bg);
    border: 1px solid var(--border); border-top: 3px solid var(--accent);
    border-radius: var(--radius); padding: 20px 24px; }
  #consist-detail-title, #sleeper-detail-title { margin: 0 0 12px; color: var(--ink);
    font-size: 16px; font-weight: 600; }
  .compare-controls { display: flex; align-items: center; gap: 12px; margin-bottom: 16px; flex-wrap: wrap; }
  .compare-label { font-size: 13px; font-weight: 500; color: var(--text2); }
  .compare-select {
    font-family: var(--font); font-size: 13px; font-weight: 500;
    color: var(--ink); background: var(--bg); border: 1px solid var(--rule);
    border-radius: var(--radius); padding: 8px 32px 8px 14px; cursor: pointer;
    appearance: none; -webkit-appearance: none;
    background-image: url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='12' height='12' viewBox='0 0 12 12'%3E%3Cpath fill='%2364748b' d='M2 4l4 4 4-4'/%3E%3C/svg%3E");
    background-repeat: no-repeat; background-position: right 12px center;
    min-width: 200px; transition: border-color 0.2s, box-shadow 0.2s;
  }
  .compare-select:hover { border-color: var(--accent); }
  .compare-select:focus { outline: none; border-color: var(--accent); box-shadow: 0 0 0 3px var(--accent-ring); }
  .compare-vs { font-weight: 600; color: var(--faint); padding: 0 4px; }
  .compare-toggle-wrap { margin-left: auto; display: flex; gap: 14px; flex-wrap: wrap; }
  .compare-toggle-label { display: flex; align-items: center; gap: 6px; font-size: 13px; color: var(--text2); cursor: pointer; user-select: none; }
  .compare-h2h-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius);
    padding: 18px 22px; margin-bottom: 16px; box-shadow: none; }
  .compare-h2h-title { font-size: 14px; font-weight: 600; color: var(--ink); margin-bottom: 12px;
    text-transform: uppercase; letter-spacing: 0.5px; }
  .compare-h2h-body { display: flex; align-items: center; gap: 18px; flex-wrap: wrap; }
  .h2h-pill { display: inline-flex; align-items: baseline; gap: 8px; padding: 10px 16px;
    border-radius: var(--radius); background: var(--bg); border: 1px solid var(--border); }
  .h2h-pill .big { font-size: 28px; font-weight: 600; }
  .h2h-pill .lbl { font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.5px; }
  .h2h-record { font-size: 13px; color: var(--text2); }
  .h2h-no-data { color: var(--faint); font-style: italic; font-size: 13px; }
  .compare-profile-row { display: grid; grid-template-columns: 1fr 1fr; gap: 16px; margin-bottom: 16px; }
  .profile-card { background: var(--surface); border: 1px solid var(--border); border-radius: var(--radius);
    padding: 18px 22px; box-shadow: none; border-top: 4px solid var(--accent); }
  .profile-card-name { font-size: 16px; font-weight: 600; color: var(--ink); margin-bottom: 12px; }
  .profile-stats { display: grid; grid-template-columns: max-content 1fr; gap: 6px 16px; font-size: 13px; }
  .profile-stats .lbl { color: var(--muted); }
  .profile-stats .val { color: var(--ink); font-weight: 500; }
  @media (max-width: 720px) { .compare-profile-row { grid-template-columns: 1fr; } }
  .compare-table-title { font-size: 14px; font-weight: 500; color: var(--muted);
    margin: 16px 0 8px; text-align: center; }
  .compare-table-wrap { overflow-x: auto; border: 1px solid var(--border);
    border-radius: var(--radius); background: white; }
  .compare-table { width: 100%; border-collapse: collapse; font-size: 13px; }
  .compare-table thead th { background: var(--ink); color: white;
    padding: 10px 14px; text-align: left; font-weight: 600; font-size: 11px;
    text-transform: uppercase; letter-spacing: 0.5px; white-space: nowrap; }
  .compare-table thead th.num { text-align: right; }
  .compare-table tbody td { padding: 8px 14px; border-bottom: 1px solid var(--wash); white-space: nowrap; }
  .compare-table tbody tr:last-child td { border-bottom: none; }
  .compare-table tbody tr:nth-child(even) td { background: var(--bg); }
  .compare-table tbody tr:hover td { background: var(--accent-wash); }
  .compare-table .num { text-align: right; font-variant-numeric: tabular-nums; }
  .compare-table .opp-name { font-weight: 500; color: var(--ink); }
  .compare-table .deck-a { border-left: 3px solid var(--deck-a, var(--accent)); }
  .compare-table .deck-b { border-left: 3px solid var(--deck-b, var(--unfav)); }
  .compare-table .delta-pos { color: var(--fav); font-weight: 500; }
  .compare-table .delta-neg { color: var(--unfav); font-weight: 500; }
  .compare-table .delta-neutral { color: var(--muted); }
  .compare-table .no-data { color: var(--rule); font-style: italic; }
  .js-plotly-plot .plotly .modebar { right: 8px !important; }
  /* Meta Roles tab */
  .roles-taxonomy-toggle { display: inline-flex; gap: 4px; padding: 4px;
    background: var(--wash); border-radius: var(--radius); margin-bottom: 18px; }
  .subtab-btn { padding: 8px 18px; border: none; background: transparent; cursor: pointer;
    font-size: 13px; font-weight: 500; font-family: var(--font); color: var(--muted);
    border-radius: var(--radius); transition: all 0.2s ease; }
  .subtab-btn:hover { background: var(--border); color: var(--text2); }
  .subtab-btn.active { background: white; color: var(--accent); box-shadow: none; }
  .roles-cards-wrap { margin-top: 18px; }
  .roles-cards-grid { display: grid; grid-template-columns: repeat(auto-fill, minmax(320px, 1fr)); gap: 14px; }
  .role-card { background: var(--surface); border: 1px solid var(--border); border-top: 4px solid var(--rule);
    border-radius: var(--radius); padding: 16px 18px; box-shadow: none; }
  .role-card-header { display: flex; align-items: center; justify-content: space-between;
    gap: 10px; margin-bottom: 8px; }
  .role-badge { display: inline-block; padding: 4px 10px; border-radius: var(--radius);
    color: white; font-size: 11px; font-weight: 600; text-transform: uppercase; letter-spacing: 0.5px; }
  .role-count { font-size: 11px; color: var(--faint); text-transform: uppercase; letter-spacing: 0.5px; }
  .role-def { font-size: 12px; color: var(--muted); margin: 0 0 12px; line-height: 1.5; }
  .role-deck-list { display: flex; flex-direction: column; gap: 4px; }
  .role-deck-row { display: flex; flex-direction: column; gap: 2px; padding: 6px 0; border-bottom: 1px solid var(--wash); }
  .role-deck-row:last-child { border-bottom: none; }
  .role-deck-name { font-size: 13px; color: var(--ink); padding-left: 8px; border-left: 3px solid var(--rule); }
  .role-deck-rationale { font-size: 11px; color: var(--faint); padding-left: 11px; font-variant-numeric: tabular-nums; }
  .role-deck-members { font-size: 11px; color: var(--muted); padding-left: 11px; font-style: italic; }
  .roles-agg-toggle { display: inline-flex; gap: 4px; padding: 4px;
    background: var(--accent-wash2); border-radius: var(--radius); margin-bottom: 14px; }
  .aggtab-btn { padding: 8px 18px; border: none; background: transparent; cursor: pointer;
    font-size: 13px; font-weight: 600; font-family: var(--font); color: var(--text2);
    border-radius: var(--radius); transition: all 0.2s ease; text-transform: uppercase; letter-spacing: 0.4px; }
  .aggtab-btn:hover { background: var(--accent-rule); color: var(--ink); }
  .aggtab-btn.active { background: white; color: var(--accent); box-shadow: none; }
  details.roles-rules { margin-bottom: 18px; border: 1px solid var(--border); border-left: 3px solid var(--accent2);
    border-radius: var(--radius); padding: 10px 16px; background: var(--bg); }
  details.roles-rules summary { cursor: pointer; font-size: 13px; font-weight: 600; color: var(--ink);
    list-style: none; }
  details.roles-rules summary::-webkit-details-marker { display: none; }
  details.roles-rules summary::before { content: '▶'; display: inline-block; margin-right: 8px;
    color: var(--accent2); transition: transform 0.2s ease; }
  details.roles-rules[open] summary::before { transform: rotate(90deg); }
  details.roles-rules .rules-hint { font-weight: 400; color: var(--faint); font-size: 12px; margin-left: 6px; }
  details.roles-rules .rules-body { padding-top: 10px; }
  details.roles-rules .rules-intro,
  details.roles-rules .rules-section,
  details.roles-rules .rules-note { font-size: 12px; color: var(--text2); margin: 8px 0; line-height: 1.5; }
  details.roles-rules .rules-section { font-weight: 600; color: var(--ink); margin-top: 14px; }
  details.roles-rules .rules-note { color: var(--faint); font-style: italic; }
  details.roles-rules table.rules-table { width: 100%; border-collapse: collapse; font-size: 12px;
    margin: 6px 0; background: white; border: 1px solid var(--border); border-radius: var(--radius); overflow: hidden; }
  details.roles-rules table.rules-table th { background: var(--ink); color: white; padding: 6px 10px;
    text-align: left; font-weight: 600; font-size: 11px; text-transform: uppercase; letter-spacing: 0.4px; }
  details.roles-rules table.rules-table td { padding: 6px 10px; border-bottom: 1px solid var(--wash); vertical-align: middle; }
  details.roles-rules table.rules-table tr:last-child td { border-bottom: none; }
  details.roles-rules table.rules-table code { font-family: 'Consolas', monospace;
    background: var(--accent-wash); color: var(--accent); padding: 1px 6px; border-radius: var(--radius); font-size: 11px; }

  /* --- Meta Simulator ------------------------------------------------- */
  .sim-controls { display: flex; flex-wrap: wrap; align-items: center; gap: 18px;
    margin-bottom: 12px; padding: 12px 16px; background: var(--bg);
    border: 1px solid var(--border); border-radius: var(--radius); }
  .sim-control-group { display: flex; align-items: center; gap: 10px; }
  .sim-label { font-size: 13px; font-weight: 600; color: var(--text);
    text-transform: uppercase; letter-spacing: 0.4px; }
  .sim-mode-toggle { display: inline-flex; border: 1px solid var(--rule);
    border-radius: var(--radius); overflow: hidden; background: white; }
  .simmode-btn { border: none; background: white; color: var(--muted); cursor: pointer;
    padding: 6px 14px; font-size: 13px; font-weight: 500; font-family: inherit; }
  .simmode-btn.active { background: var(--accent); color: white; }
  .sim-buttons { margin-left: auto; }
  .sim-btn { border: 1px solid var(--rule); background: white; color: var(--text); cursor: pointer;
    padding: 7px 16px; font-size: 13px; font-weight: 600; font-family: inherit;
    border-radius: var(--radius); transition: all 0.15s ease; }
  .sim-btn:hover { border-color: var(--accent); color: var(--accent); }
  .sim-btn-default { background: var(--accent); border-color: var(--accent); color: white; }
  .sim-btn-default:hover { background: var(--accent-dark); color: white; }
  .sim-toggle-label { display: inline-flex; align-items: center; gap: 6px;
    font-size: 13px; color: var(--text2); cursor: pointer; }
  .sim-status { font-size: 13px; color: var(--muted); margin: 0 0 12px 2px; }
  .sim-status.warn { color: var(--warn); font-weight: 600; }
  .sim-layout { display: grid; grid-template-columns: minmax(260px, 340px) 1fr; gap: 20px;
    align-items: start; }
  @media (max-width: 900px) { .sim-layout { grid-template-columns: 1fr; } }
  .sim-panel { background: white; border: 1px solid var(--border); border-radius: var(--radius); padding: 14px 16px; }
  .sim-panel-title { font-size: 13px; font-weight: 600; color: var(--ink); margin-bottom: 10px;
    text-transform: uppercase; letter-spacing: 0.4px; }
  .sim-inputs { max-height: 620px; overflow-y: auto; padding-right: 6px; }
  .sim-row { display: grid; grid-template-columns: 10px 1fr 76px 52px; align-items: center;
    gap: 8px; padding: 3px 0; border-bottom: 1px solid var(--wash); }
  .sim-swatch { width: 10px; height: 10px; border-radius: var(--radius); }
  .sim-deck { font-size: 12px; color: var(--text); white-space: nowrap; overflow: hidden;
    text-overflow: ellipsis; }
  .sim-badge { display: inline-block; margin-left: 5px; padding: 0 5px; border-radius: var(--radius);
    background: var(--violet-bg); color: var(--violet-ink); font-size: 9px; font-weight: 700; vertical-align: middle; }
  .sim-input { width: 100%; padding: 3px 6px; font-size: 12px; font-family: inherit;
    border: 1px solid var(--rule); border-radius: var(--radius); text-align: right; color: var(--ink); }
  .sim-input:focus { outline: none; border-color: var(--accent); box-shadow: 0 0 0 2px var(--accent-ring); }
  .sim-use { font-size: 11px; color: var(--faint); text-align: right; }
  table.sim-table td.sim-rank { color: var(--faint); font-variant-numeric: tabular-nums; }
  table.sim-table td.sim-deck-cell { font-weight: 600; border-left: 3px solid transparent; padding-left: 8px; }
  table.sim-table td.sim-wr { font-weight: 700; font-variant-numeric: tabular-nums; }
  .sim-thin { display: block; font-size: 10px; color: var(--warn); font-weight: 600; }
  table.sim-table tr.sim-out-of-field td.sim-deck-cell { font-weight: 500; color: var(--muted); }

  /* --- Overview table + deck links --------------------------------------- */
  .ov-title { font-size: 18px; font-weight: 600; color: var(--ink); margin: 0 0 4px; }
  .ov-sub { font-size: 12px; color: var(--muted); margin: 0 0 12px; line-height: 1.5; }
  .ov-table tbody tr.ov-row { cursor: pointer; }
  .ov-table tbody tr.ov-row:focus-visible td { outline: 2px solid var(--accent); outline-offset: -2px; }
  .ov-table td.ov-shaky { color: var(--warn); font-weight: 600; }
  .tier-badge { display: inline-block; min-width: 26px; text-align: center; padding: 2px 8px;
    border-radius: var(--radius); font-weight: 700; font-size: 12px; }
  .deck-picker { margin-left: auto; display: flex; align-items: center; gap: 12px; }
  .detail-head { display: flex; align-items: center; justify-content: space-between; gap: 12px; }
  .deck-link-btn { border: 1px solid var(--accent-rule); background: var(--accent-wash); color: var(--accent); cursor: pointer;
    font: 600 12px var(--font); padding: 6px 12px; border-radius: var(--radius); white-space: nowrap; }
  .deck-link-btn:hover { background: var(--accent-wash2); }
  .role-deck-row[data-deck] { cursor: pointer; }
  .role-deck-row[data-deck]:hover .role-deck-name { color: var(--accent); }

  /* --- Deck detail drawer ------------------------------------------------ */
  #deck-drawer-backdrop { position: fixed; inset: 0; background: var(--scrim);
    opacity: 0; pointer-events: none; transition: opacity 0.2s ease; z-index: 999; }
  #deck-drawer-backdrop.open { opacity: 1; pointer-events: auto; }
  #deck-drawer { position: fixed; top: 0; right: 0; height: 100vh; width: min(620px, 100vw);
    background: var(--surface); box-shadow: -8px 0 32px rgba(17, 26, 47, 0.22); z-index: 1000;
    transform: translateX(105%); transition: transform 0.25s ease; overflow-y: auto;
    padding: 0 24px 32px; border-left: 4px solid var(--accent); }
  #deck-drawer.open { transform: translateX(0); }
  .dd-head { position: sticky; top: 0; background: var(--surface); z-index: 1; display: flex;
    align-items: flex-start; justify-content: space-between; gap: 12px;
    padding: 20px 0 12px; border-bottom: 1px solid var(--border); margin-bottom: 14px; }
  #deck-drawer-title { font-size: 20px; font-weight: 600; color: var(--ink); }
  #deck-drawer-sub { font-size: 12px; color: var(--muted); margin-top: 2px; }
  #deck-drawer-close { border: none; background: var(--wash); color: var(--text2); font-size: 22px;
    line-height: 1; width: 34px; height: 34px; border-radius: var(--radius); cursor: pointer; flex: none; }
  #deck-drawer-close:hover { background: var(--border); color: var(--ink); }
  .dd-kpis { display: grid; grid-template-columns: repeat(3, 1fr); gap: 10px; margin-bottom: 14px; }
  .dd-kpi { background: var(--bg); border: 1px solid var(--border); border-radius: var(--radius); padding: 10px 12px; }
  .dd-kpi .v { font-size: 22px; font-weight: 600; color: var(--ink); font-variant-numeric: tabular-nums; }
  .dd-kpi .l { font-size: 11px; color: var(--muted); text-transform: uppercase; letter-spacing: 0.4px; }
  .dd-kpi .s { font-size: 11px; color: var(--faint); }
  .dd-section { border: 1px solid var(--border); border-radius: var(--radius); padding: 12px 14px; margin-bottom: 10px; }
  .dd-section-title { font-size: 12px; font-weight: 600; color: var(--accent); text-transform: uppercase;
    letter-spacing: 0.5px; margin: 0 0 8px; }
  .dd-grid { display: grid; grid-template-columns: max-content 1fr; gap: 5px 16px; font-size: 13px; }
  .dd-grid .lbl { color: var(--muted); }
  .dd-grid .val { color: var(--ink); font-weight: 500; font-variant-numeric: tabular-nums; }
  .dd-na { color: var(--faint); font-style: italic; font-weight: 400; }
  .dd-pos { color: var(--fav); } .dd-neg { color: var(--unfav); } .dd-warn { color: var(--warn); }
  .dd-pill { display: inline-block; padding: 1px 8px; border-radius: var(--radius); font-size: 11px;
    font-weight: 600; color: white; }
  #deck-drawer-chart-title { margin-top: 14px; }
  @media (max-width: 560px) { .dd-kpis { grid-template-columns: repeat(2, 1fr); } }
</style>
</head>
"""

_win_end = pd.Timestamp(STANDINGS_WINDOW['end_date'])
_win_start = pd.Timestamp(STANDINGS_WINDOW['start_date'])
_fmt_day = lambda t: f'{t.day} {t:%b %Y}'
BODY_HTML = (
    '<body>\n<header class="site-header"><div class="site-header-inner">\n'
    '  <div class="header-top">\n'
    '    <div class="brand">\n'
    '      <h1>Pokemon TCG <span class="mark">Meta Analyzer</span></h1>\n'
    f'      <p class="subtitle">{_deck_note}{_empty_note}</p>\n'
    '    </div>\n'
    '    <div class="header-meta">\n'
    f'      <span class="through">Data through {_fmt_day(_win_end)}</span>\n'
    f'      {FETCH_DAYS}-day window from {_fmt_day(_win_start)} &middot; data: '
    '<a href="https://www.trainerhill.com/" rel="noopener">Trainer Hill</a>, '
    '<a href="https://limitlesstcg.com/" rel="noopener">Limitless</a><br>\n'
    '      <a class="phone-link" href="matchups.html">Phone matchups &rarr;</a>\n'
    '    </div>\n'
    '  </div>\n'
    f'  {PLATFORM_SELECTOR_HTML}'
    '</div></header>\n'
    '<div class="container">\n'
    f'  {NAV_HTML}'
    f'  {tab_content_html}\n'
    '</div>\n'
    '<footer class="site-footer">Matchup, share and trend data from '
    '<a href="https://www.trainerhill.com/" rel="noopener">Trainer Hill</a>; event standings and pairings from '
    '<a href="https://limitlesstcg.com/" rel="noopener">Limitless</a>. '
    'Not affiliated with either, or with The Pok&eacute;mon Company.</footer>\n'
    f'{DECK_DRAWER_HTML}'
)

JS_HTML = """<script>
var matchupBreakdownPer = __MB__JSON__;
var sleeperBreakdownPer = __SB__JSON__;
var consistScatterDataPer = __CSD__JSON__;
var UI = __UI__JSON__;   // the palette from cell 0, so drawn charts match the page
var currentPlatform = '__DEFAULT_PLATFORM__';
var currentConsistTier = 'All';

// currentPlatform holds a VIEW: the Data Source, plus RAW_SUFFIX while "Raw win
// rates" is ticked. Every *Per payload and every platform block is keyed by view.
var RAW_SUFFIX = '__RAW_SUFFIX__';
function isRawView(v) { return v.slice(-RAW_SUFFIX.length) === RAW_SUFFIX; }
function basePlatform(v) { return isRawView(v) ? v.slice(0, -RAW_SUFFIX.length) : v; }
function currentView() {
  var raw = document.getElementById('raw-wr-toggle');
  return document.getElementById('platform-select').value + (raw && raw.checked ? RAW_SUFFIX : '');
}

var tierLabels = {'All': 'All Players', 'Upper50': 'Top 50%', 'Upper25': 'Top 25%', 'Weighted': 'Weighted toward top finishers'};
var FAV_COLOR = '__FAV__';
var UNFAV_COLOR = '__UNFAV__';

var compareDataPer = __CMP__JSON__;

function populateCompareDropdowns(plat) {
  var data = compareDataPer[plat];
  if (!data) return;
  var sa = document.getElementById('compare-deck-a');
  var sb = document.getElementById('compare-deck-b');
  if (!sa || !sb) return;
  var prevA = sa.value, prevB = sb.value;
  var optsHtml = data.decks.map(function(d) {
    var p = data.profile[d];
    return '<option value="' + d + '">' + (p ? p.short : d) + '</option>';
  }).join('');
  sa.innerHTML = optsHtml;
  sb.innerHTML = optsHtml;
  if (prevA && data.profile[prevA]) sa.value = prevA;
  else sa.value = data.decks[0] || '';
  var defaultB = data.decks.find(function(d) { return d !== sa.value; }) || data.decks[0] || '';
  if (prevB && data.profile[prevB] && prevB !== sa.value) sb.value = prevB;
  else sb.value = defaultB;
}

function renderH2HCard(data, a, b) {
  var titleEl = document.getElementById('compare-h2h-title');
  var bodyEl = document.getElementById('compare-h2h-body');
  if (!titleEl || !bodyEl) return;
  var pa = data.profile[a], pb = data.profile[b];
  if (!pa || !pb) {
    titleEl.textContent = 'Head-to-Head';
    bodyEl.innerHTML = '<span class="h2h-no-data">Pick two decks.</span>';
    return;
  }
  titleEl.textContent = 'Head-to-Head: ' + pa.short + ' vs ' + pb.short + ' (All Players)';
  if (a === b) {
    bodyEl.innerHTML = '<span class="h2h-no-data">Pick two different decks.</span>';
    return;
  }
  var h = data.h2h[a] && data.h2h[a][b];
  if (!h) {
    bodyEl.innerHTML = '<span class="h2h-no-data">No recorded matchup at the All tier on this platform.</span>';
    return;
  }
  var color = h.wr >= 50 ? FAV_COLOR : UNFAV_COLOR;
  bodyEl.innerHTML =
    '<div class="h2h-pill"><span class="big" style="color:' + color + '">' + h.wr.toFixed(1) + '%</span>' +
    '<span class="lbl">' + pa.short + ' WR</span></div>' +
    '<div class="h2h-record"><b>' + h.wins + 'W &middot; ' + h.losses + 'L &middot; ' + h.ties + 'T</b> &nbsp;(' + h.n + ' games)</div>' +
    '<div class="h2h-record">95% CI: ' + h.ci_lo.toFixed(1) + '% &ndash; ' + h.ci_hi.toFixed(1) + '%</div>';
}

function renderProfileCard(data, elId, deck) {
  var el = document.getElementById(elId);
  if (!el) return;
  var p = data.profile[deck];
  if (!p) { el.innerHTML = '<span class="h2h-no-data">No data for ' + deck + '</span>'; return; }
  el.style.borderTopColor = p.color;
  function fmt(t) {
    var x = p.tiers[t];
    if (!x) return '<span style="color:var(--faint)">&mdash;</span>';
    return x.wr.toFixed(1) + '% <span style="color:var(--faint);font-size:11px">(n=' + x.games.toLocaleString() + ')</span>';
  }
  var skillDelta = (p.tiers.All && p.tiers.Upper25) ? (p.tiers.Upper25.wr - p.tiers.All.wr) : null;
  var html = '<div class="profile-card-name" style="color:' + p.color + '">' + p.short + '</div>' +
    '<div class="profile-stats">' +
    '<span class="lbl">All players</span><span class="val">' + fmt('All') + '</span>' +
    '<span class="lbl">Top 50%</span><span class="val">' + fmt('Upper50') + '</span>' +
    '<span class="lbl">Top 25%</span><span class="val">' + fmt('Upper25') + '</span>' +
    '<span class="lbl">Top 25% &minus; All</span><span class="val">' +
      (skillDelta == null ? '&mdash;'
        : '<span style="color:' + (skillDelta >= 0 ? FAV_COLOR : UNFAV_COLOR) + '">' +
          (skillDelta >= 0 ? '+' : '') + skillDelta.toFixed(1) + ' pp</span>') +
      '</span>' +
    (p.fav_impact != null ? '<span class="lbl">Favorable impact</span><span class="val">' + p.fav_impact.toFixed(2) + '</span>' : '') +
    (p.unfav_impact != null ? '<span class="lbl">Unfavorable impact</span><span class="val">' + p.unfav_impact.toFixed(2) + '</span>' : '') +
    '</div>';
  el.innerHTML = html;
}

function renderDivergenceChart(a, b, skill, diff) {
  var el = document.getElementById('compare-divergence-chart');
  if (!el) return;
  var mbPlat = matchupBreakdownPer[currentPlatform] || {};
  var allMb = mbPlat.All;
  var u25Mb = mbPlat.Upper25;
  if (!allMb) { Plotly.purge(el); el.innerHTML = '<div class="h2h-no-data">No matchup data for this platform.</div>'; return; }
  function vec(deck, mb) {
    var m = {}; ((mb && mb[deck]) || []).forEach(function(r) { m[r.opponent] = r; }); return m;
  }
  var aData = {}, bData = {}, xLabel, unit;
  if (skill) {
    if (!u25Mb) {
      Plotly.purge(el);
      el.innerHTML = '<div class="h2h-no-data">Top 25% data not available for this platform.</div>';
      return;
    }
    var aAll = vec(a, allMb), aU25 = vec(a, u25Mb);
    var bAll = vec(b, allMb), bU25 = vec(b, u25Mb);
    Object.keys(aAll).forEach(function(o) {
      if (aU25[o]) aData[o] = { wr: aU25[o].win_rate - aAll[o].win_rate, opp_short: aAll[o].opp_short, opp_pop: aAll[o].opp_pop, n_all: aAll[o].games, n_u25: aU25[o].games };
    });
    Object.keys(bAll).forEach(function(o) {
      if (bU25[o]) bData[o] = { wr: bU25[o].win_rate - bAll[o].win_rate, opp_short: bAll[o].opp_short, opp_pop: bAll[o].opp_pop, n_all: bAll[o].games, n_u25: bU25[o].games };
    });
    xLabel = 'WR Shift: Top 25% − All (pp)';
    unit = 'pp';
  } else {
    var aLst = allMb[a] || [], bLst = allMb[b] || [];
    aLst.forEach(function(r) { aData[r.opponent] = { wr: r.win_rate, opp_short: r.opp_short, opp_pop: r.opp_pop, games: r.games }; });
    bLst.forEach(function(r) { bData[r.opponent] = { wr: r.win_rate, opp_short: r.opp_short, opp_pop: r.opp_pop, games: r.games }; });
    xLabel = 'Win Rate vs Opponent (%)';
    unit = '%';
  }
  var pa = compareDataPer[currentPlatform].profile[a];
  var pb = compareDataPer[currentPlatform].profile[b];
  var paName = pa ? pa.short : a;
  var pbName = pb ? pb.short : b;
  var paColor = pa ? pa.color : UI.accent;
  var pbColor = pb ? pb.color : UI.unfav;

  if (diff) {
    // Single-bar diff mode: only opponents present for BOTH decks
    var pairs = [];
    Object.keys(aData).forEach(function(o) {
      if (o === a || o === b) return;
      if (bData[o]) pairs.push({
        opp: o,
        label: aData[o].opp_short || bData[o].opp_short || o,
        d: aData[o].wr - bData[o].wr,
        a_wr: aData[o].wr, b_wr: bData[o].wr,
        pop: aData[o].opp_pop != null ? aData[o].opp_pop
            : (bData[o].opp_pop != null ? bData[o].opp_pop : 0),
      });
    });
    if (pairs.length === 0) {
      Plotly.purge(el);
      el.innerHTML = '<div class="h2h-no-data">No common opponents to compare.</div>';
      return;
    }
    pairs.sort(function(p1, p2) { return p1.pop - p2.pop; });  // ascending by use rate: most-popular at TOP of chart
    var diffLabels = pairs.map(function(p) { return p.label; });
    var diffVals   = pairs.map(function(p) { return p.d; });
    var diffColors = pairs.map(function(p) { return p.d >= 0 ? paColor : pbColor; });
    var diffHovers = pairs.map(function(p) {
      var sign = p.d >= 0 ? '+' : '';
      return '<b>vs ' + p.label + '</b><br>' +
             paName + ': ' + p.a_wr.toFixed(1) + ' ' + unit + '<br>' +
             pbName + ': ' + p.b_wr.toFixed(1) + ' ' + unit + '<br>' +
             'Difference: ' + sign + p.d.toFixed(1) + ' ' + unit;
    });
    var diffTrace = {
      type: 'bar', orientation: 'h',
      y: diffLabels, x: diffVals,
      marker: { color: diffColors, line: { width: 0 } },
      text: diffHovers, hoverinfo: 'text',
    };
    var diffXTitle = (skill ? 'Top-finisher shift gap (pp): ' : 'Win-rate gap (' + unit + '): ')
                   + '← ' + pbName + ' favored  │  ' + paName + ' favored →';
    var diffLayout = {
      title: { text: skill
          ? ('Top-finisher shift gap: ' + paName + ' − ' + pbName)
          : ('Win-rate gap: ' + paName + ' − ' + pbName),
        font: { size: 14 } },
      height: Math.max(420, pairs.length * 26 + 140),
      margin: { l: 150, r: 40, t: 70, b: 60 },
      xaxis: { title: { text: diffXTitle }, gridcolor: UI.grid,
               tickfont: { size: 11, color: UI.muted },
               zeroline: true, zerolinewidth: 2, zerolinecolor: UI.rule },
      yaxis: { title: '', automargin: true, gridcolor: UI.grid,
               tickfont: { size: 11, color: UI.muted } },
      showlegend: false,
      paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: UI.surface,
      font: { family: UI.font, color: UI.text, size: 12 },
      hoverlabel: { bgcolor: 'white', bordercolor: UI.border, font: { size: 12, color: UI.ink } },
    };
    Plotly.newPlot(el, [diffTrace], diffLayout, { responsive: true });
    return;
  }

  // Side-by-side mode: render an HTML table (replaces former parallel-bar chart).
  Plotly.purge(el);  // clear any prior Plotly chart so the table replaces it cleanly
  var oppSet = {};
  Object.keys(aData).forEach(function(o) { if (o !== a && o !== b) oppSet[o] = true; });
  Object.keys(bData).forEach(function(o) { if (o !== a && o !== b) oppSet[o] = true; });
  var opps = Object.keys(oppSet);
  if (opps.length === 0) {
    el.innerHTML = '<div class="h2h-no-data">No common-opponent data to compare.</div>';
    return;
  }
  var popOf = function(o) {
    return aData[o] && aData[o].opp_pop != null ? aData[o].opp_pop
         : (bData[o] && bData[o].opp_pop != null ? bData[o].opp_pop : 0);
  };
  opps.sort(function(o1, o2) { return popOf(o2) - popOf(o1); });  // descending by use rate

  var headerSuffix = skill ? ' shift' : ' WR';
  var cellUnit = skill ? ' pp' : '%';
  var titleText = skill
    ? ('Common-opponent top-finisher shift: ' + paName + ' vs ' + pbName)
    : ('Common-opponent win rates: ' + paName + ' vs ' + pbName);
  var fmtCell = function(d, deckCls) {
    if (!d || d.wr == null) return '<td class="num no-data ' + deckCls + '">&mdash;</td>';
    var n = d.games != null ? d.games : (d.n_all != null ? d.n_all : null);
    var nStr = n != null ? ' <span style="color:var(--faint);font-size:11px">(' + n + ')</span>' : '';
    return '<td class="num ' + deckCls + '">' + d.wr.toFixed(1) + cellUnit + nStr + '</td>';
  };
  var fmtDelta = function(av, bv) {
    if (av == null || bv == null) return '<td class="num delta-neutral">&mdash;</td>';
    var d = av - bv;
    var sign = d >= 0 ? '+' : '';
    var cls = Math.abs(d) < 1 ? 'delta-neutral' : (d > 0 ? 'delta-pos' : 'delta-neg');
    return '<td class="num ' + cls + '">' + sign + d.toFixed(1) + ' pp</td>';
  };

  var headerHtml = '<thead><tr>'
    + '<th>Opponent</th>'
    + '<th class="num deck-a">' + paName + headerSuffix + '</th>'
    + '<th class="num deck-b">' + pbName + headerSuffix + '</th>'
    + '<th class="num">&Delta; (' + paName + ' &minus; ' + pbName + ')</th>'
    + '<th class="num">Use rate</th>'
    + '</tr></thead>';
  var rowsHtml = opps.map(function(o) {
    var ad = aData[o], bd = bData[o];
    var label = (ad && ad.opp_short) || (bd && bd.opp_short) || o;
    var av = ad ? ad.wr : null, bv = bd ? bd.wr : null;
    return '<tr>'
      + '<td class="opp-name">' + label + '</td>'
      + fmtCell(ad, 'deck-a') + fmtCell(bd, 'deck-b') + fmtDelta(av, bv)
      + '<td class="num">' + popOf(o).toFixed(1) + '%</td>'
      + '</tr>';
  }).join('');
  el.innerHTML =
    '<div class="compare-table-title">' + titleText + '</div>'
    + '<div class="compare-table-wrap" style="--deck-a:' + paColor + ';--deck-b:' + pbColor + ';">'
    + '<table class="compare-table">' + headerHtml + '<tbody>' + rowsHtml + '</tbody></table></div>';
}

function renderSkillTierChart(data, a, b) {
  var el = document.getElementById('compare-skilltier-chart');
  if (!el) return;
  var pa = data.profile[a], pb = data.profile[b];
  if (!pa || !pb) { Plotly.purge(el); el.innerHTML = ''; return; }
  var tiers = ['All', 'Upper50', 'Upper25'];
  var labels = ['All Players', 'Top 50%', 'Top 25%'];
  function build(p) {
    return {
      name: p.short, type: 'scatter', mode: 'lines+markers',
      x: labels, y: tiers.map(function(t) { return p.tiers[t] ? p.tiers[t].wr : null; }),
      line: { color: p.color, width: 3 }, marker: { size: 12, color: p.color, line: { width: 2, color: 'white' } },
      error_y: {
        type: 'data', symmetric: false,
        array: tiers.map(function(t) {
          var x = p.tiers[t]; return x ? Math.max(0, x.ci_hi - x.simple_wr) : null;
        }),
        arrayminus: tiers.map(function(t) {
          var x = p.tiers[t]; return x ? Math.max(0, x.simple_wr - x.ci_lo) : null;
        }),
        thickness: 1.5, color: UI.faint,
      },
      hovertemplate: '<b>' + p.short + '</b><br>%{x}: %{y:.1f}%<extra></extra>',
    };
  }
  var layout = {
    title: { text: 'Win rate across finishing tiers (3-pt, with 95% CI)', font: { size: 14 } },
    xaxis: { title: '', tickfont: { size: 12, color: UI.text2 } },
    yaxis: { title: { text: 'Win Rate (3pt) %' }, gridcolor: UI.grid,
             tickfont: { size: 11, color: UI.muted } },
    height: 380, margin: { l: 70, r: 30, t: 60, b: 50 },
    legend: { orientation: 'h', yanchor: 'bottom', y: 1.02, xanchor: 'center', x: 0.5 },
    paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: UI.surface,
    font: { family: UI.font, color: UI.text, size: 12 },
    hoverlabel: { bgcolor: 'white', bordercolor: UI.border, font: { size: 12, color: UI.ink } },
    shapes: [{ type: 'line', x0: -0.5, x1: 2.5, y0: 50, y1: 50, xref: 'x', yref: 'y',
               line: { color: UI.rule, width: 1, dash: 'dot' } }],
  };
  Plotly.newPlot(el, [build(pa), build(pb)], layout, { responsive: true });
}

function renderCompare() {
  var data = compareDataPer[currentPlatform];
  if (!data) return;
  var sa = document.getElementById('compare-deck-a');
  var sb = document.getElementById('compare-deck-b');
  var skillTog = document.getElementById('compare-skill-toggle');
  var diffTog = document.getElementById('compare-diff-toggle');
  if (!sa || !sb || !skillTog || !diffTog) return;
  var a = sa.value, b = sb.value, skill = skillTog.checked, diff = diffTog.checked;
  renderH2HCard(data, a, b);
  renderProfileCard(data, 'compare-profile-a', a);
  renderProfileCard(data, 'compare-profile-b', b);
  renderDivergenceChart(a, b, skill, diff);
  renderSkillTierChart(data, a, b);
}

function refreshCompare() {
  if (!document.getElementById('compare-deck-a')) return;
  populateCompareDropdowns(currentPlatform);
  var sa = document.getElementById('compare-deck-a');
  if (sa && !sa.dataset.bound) {
    sa.dataset.bound = '1';
    sa.addEventListener('change', renderCompare);
    document.getElementById('compare-deck-b').addEventListener('change', renderCompare);
    document.getElementById('compare-skill-toggle').addEventListener('change', renderCompare);
    document.getElementById('compare-diff-toggle').addEventListener('change', renderCompare);
  }
  renderCompare();
}


// === Meta Simulator ==========================================================
// Expected WR for a deck = its matchup win rates against the simulated field,
// weighted by each opponent's share of that field and renormalised over the
// opponents it actually has data against (that's the coverage column).
var simulatorDataPer = __SIM__JSON__;
var simValues = {};          // platform (not view: a field survives the raw toggle) -> {deck: raw input value}
var simMode = 'percentage';
var simTier = 'All';

function simData() { return simulatorDataPer[currentPlatform]; }

function simDefaultValues(plat) {
  var d = simulatorDataPer[plat], out = {};
  if (!d) return out;
  d.decks.forEach(function(k) { out[k] = d.usage[k] != null ? d.usage[k] : 0; });
  return out;
}

function simEnsureValues() {
  var p = basePlatform(currentPlatform);
  if (!simValues[p]) simValues[p] = simDefaultValues(currentPlatform);
  return simValues[p];
}

function simFieldShares(vals, decks) {
  var total = 0, raw = {};
  decks.forEach(function(k) {
    var v = parseFloat(vals[k]);
    if (isNaN(v) || v <= 0) v = 0;
    raw[k] = v; total += v;
  });
  var shares = {};
  decks.forEach(function(k) { shares[k] = total > 0 ? raw[k] / total : 0; });
  return { shares: shares, total: total };
}

function simExpected(deck, shares, wrRows, gameRows, minSample) {
  var row = wrRows[deck] || {};
  var games = (gameRows || {})[deck] || {};
  var num = 0, cov = 0, thin = 0, best = null, worst = null;
  Object.keys(shares).forEach(function(opp) {
    var s = shares[opp];
    if (!s) return;
    var w = row[opp];
    if (w == null) return;
    num += s * w; cov += s;
    // How much of the field rests on a matchup thinner than MIN_SAMPLE games.
    var n = games[opp];
    if (n == null || n < minSample) thin += s;
    if (!best || w > best.wr) best = { opp: opp, wr: w, share: s, n: n };
    if (!worst || w < worst.wr) worst = { opp: opp, wr: w, share: s, n: n };
  });
  if (cov <= 0) return null;
  return { wr: num / cov, coverage: cov, thin: thin / cov, best: best, worst: worst };
}

function buildSimInputs() {
  var d = simData(), wrap = document.getElementById('sim-inputs');
  if (!d || !wrap) return;
  var vals = simEnsureValues();
  var coreSet = {};
  d.core.forEach(function(k) { coreSet[k] = true; });
  wrap.innerHTML = d.decks.map(function(k) {
    var v = vals[k];
    var shown = (v === '' || v == null) ? '' : v;
    return '<div class="sim-row">' +
      '<span class="sim-swatch" style="background:' + d.colors[k] + '"></span>' +
      '<span class="sim-deck" title="' + k + '">' + d.names[k] +
        (coreSet[k] ? '' : '<span class="sim-badge">S</span>') + '</span>' +
      '<input class="sim-input" type="number" min="0" step="0.1" data-deck="' + k +
        '" value="' + shown + '">' +
      '<span class="sim-use">' + (d.usage[k] != null ? d.usage[k].toFixed(1) : '0.0') + '%</span>' +
      '</div>';
  }).join('');
  wrap.querySelectorAll('.sim-input').forEach(function(el) {
    el.addEventListener('input', function() {
      simEnsureValues()[this.getAttribute('data-deck')] = this.value;
      renderSimResults();
    });
  });
}

function renderSimResults() {
  var d = simData(), el = document.getElementById('sim-results');
  var statusEl = document.getElementById('sim-status');
  if (!d || !el) return;
  var vals = simEnsureValues();
  var f = simFieldShares(vals, d.decks);
  if (f.total <= 0) {
    el.innerHTML = '<div class="h2h-no-data">The field is empty &mdash; enter a value for at ' +
                   'least one deck, or press Default to load the real use rates.</div>';
    if (statusEl) { statusEl.className = 'sim-status warn'; statusEl.textContent = 'Field total: 0'; }
    return;
  }
  var inField = d.decks.filter(function(k) { return f.shares[k] > 0; }).length;
  if (statusEl) {
    if (simMode === 'percentage') {
      var off = Math.abs(f.total - 100) > 0.05;
      statusEl.className = 'sim-status' + (off ? ' warn' : '');
      statusEl.textContent = 'Field total: ' + f.total.toFixed(1) + '% across ' + inField + ' decks'
        + (off ? ' \u2014 does not add up to 100%, shares are normalised for the maths.' : '.');
    } else {
      statusEl.className = 'sim-status';
      statusEl.textContent = 'Total weight: ' + f.total.toFixed(1) + ' across ' + inField +
        ' decks \u2014 normalised to 100% of the field.';
    }
  }
  var wrRows = d.wr[simTier] || {};
  var gameRows = (d.games || {})[simTier] || {};
  var minSample = d.minSample || 0;
  var defShares = simFieldShares(simDefaultValues(currentPlatform), d.decks).shares;
  var onlyEl = document.getElementById('sim-only-field');
  var onlyField = !!(onlyEl && onlyEl.checked);
  var rows = [];
  d.decks.forEach(function(k) {
    if (onlyField && !(f.shares[k] > 0)) return;
    var e = simExpected(k, f.shares, wrRows, gameRows, minSample);
    if (!e) return;
    var base = simExpected(k, defShares, wrRows, gameRows, minSample);
    rows.push({
      deck: k, share: f.shares[k] * 100, wr: e.wr, cov: e.coverage * 100,
      thin: e.thin * 100, best: e.best, worst: e.worst,
      delta: base ? e.wr - base.wr : null,
    });
  });
  if (rows.length === 0) {
    el.innerHTML = '<div class="h2h-no-data">No deck has matchup data against this field.</div>';
    return;
  }
  rows.sort(function(a, b) { return b.wr - a.wr; });
  var mu = function(m) {
    if (!m) return '<td>&mdash;</td>';
    var thin = (m.n == null || m.n < minSample);
    return '<td>' + d.names[m.opp] + ' <span style="color:' + (thin ? UI.warn : UI.faint) +
           ';font-size:11px">' + m.wr.toFixed(1) + '% &middot; ' +
           (m.n != null ? m.n : 0) + ' games' + (thin ? ' (thin)' : '') + '</span></td>';
  };
  var body = rows.map(function(r, i) {
    var dcls = r.delta == null ? 'delta-neutral'
             : (Math.abs(r.delta) < 0.5 ? 'delta-neutral' : (r.delta > 0 ? 'delta-pos' : 'delta-neg'));
    var dtxt = r.delta == null ? '&mdash;' : (r.delta >= 0 ? '+' : '') + r.delta.toFixed(1) + ' pp';
    return '<tr class="' + (r.share > 0 ? '' : 'sim-out-of-field') + '">' +
      '<td class="sim-rank">' + (i + 1) + '</td>' +
      '<td class="sim-deck-cell" style="border-left-color:' + d.colors[r.deck] + '">' +
        d.names[r.deck] + '</td>' +
      '<td class="num">' + r.share.toFixed(1) + '%</td>' +
      '<td class="num sim-wr" style="color:' + (r.wr >= 50 ? FAV_COLOR : UNFAV_COLOR) + '">' +
        r.wr.toFixed(1) + '%</td>' +
      '<td class="num ' + dcls + '">' + dtxt + '</td>' +
      '<td class="num">' + r.cov.toFixed(0) + '%' +
        (r.thin > 5 ? '<span class="sim-thin" title="share of the field resting on matchups with ' +
         'fewer than ' + minSample + ' games">' + r.thin.toFixed(0) + '% thin</span>' : '') + '</td>' +
      mu(r.best) + mu(r.worst) +
      '</tr>';
  }).join('');
  el.innerHTML = '<div class="compare-table-wrap"><table class="compare-table sim-table">' +
    '<thead><tr><th>#</th><th>Deck</th><th class="num">Field %</th>' +
    '<th class="num">Expected WR</th><th class="num">&Delta; vs actual meta</th>' +
    '<th class="num">Coverage</th><th>Best matchup</th><th>Worst matchup</th></tr></thead>' +
    '<tbody>' + body + '</tbody></table></div>';
}

function setSimMode(mode) {
  if (mode === simMode) return;
  var d = simData();
  if (d && mode === 'percentage') {
    // Weights carry over as the percentages they already imply (3/1/1 -> 60/20/20).
    var vals = simEnsureValues();
    var f = simFieldShares(vals, d.decks);
    if (f.total > 0) {
      d.decks.forEach(function(k) {
        var s = f.shares[k] * 100;
        vals[k] = s > 0 ? Math.round(s * 10) / 10 : '';
      });
    }
  }
  simMode = mode;
  document.querySelectorAll('.simmode-btn').forEach(function(el) {
    el.classList.toggle('active', el.getAttribute('data-mode') === mode);
  });
  buildSimInputs();
  renderSimResults();
}

function simClear() {
  var d = simData();
  if (!d) return;
  var vals = simEnsureValues();
  d.decks.forEach(function(k) { vals[k] = ''; });
  buildSimInputs();
  renderSimResults();
}

function simDefault() {
  if (!simData()) return;
  simValues[basePlatform(currentPlatform)] = simDefaultValues(currentPlatform);
  buildSimInputs();
  renderSimResults();
}

function refreshSimulator() {
  if (!document.getElementById('sim-inputs')) return;
  var tierSel = document.getElementById('sim-tier-select');
  if (tierSel && !tierSel.dataset.bound) {
    tierSel.dataset.bound = '1';
    tierSel.addEventListener('change', function() { simTier = this.value; renderSimResults(); });
  }
  var onlyEl = document.getElementById('sim-only-field');
  if (onlyEl && !onlyEl.dataset.bound) {
    onlyEl.dataset.bound = '1';
    onlyEl.addEventListener('change', renderSimResults);
  }
  buildSimInputs();
  renderSimResults();
}

function switchRolesAggregation(agg) {
  document.querySelectorAll('.roles-aggregation-content').forEach(function(el) {
    el.style.display = (el.getAttribute('data-agg') === agg) ? 'block' : 'none';
  });
  document.querySelectorAll('.aggtab-btn').forEach(function(el) {
    el.classList.toggle('active', el.getAttribute('data-agg') === agg);
  });
  document.querySelectorAll('.roles-aggregation-content[data-agg="' + agg + '"] .js-plotly-plot')
    .forEach(function(el) { try { Plotly.Plots.resize(el); } catch(e) {} });
  window.dispatchEvent(new Event('resize'));
}

function switchRolesTaxonomy(tax) {
  document.querySelectorAll('.roles-taxonomy-content').forEach(function(el) {
    el.style.display = (el.getAttribute('data-tax') === tax) ? 'block' : 'none';
  });
  document.querySelectorAll('.subtab-btn').forEach(function(el) {
    el.classList.toggle('active', el.getAttribute('data-tax') === tax);
  });
  document.querySelectorAll('.roles-taxonomy-content[data-tax="' + tax + '"] .js-plotly-plot')
    .forEach(function(el) { try { Plotly.Plots.resize(el); } catch(e) {} });
  window.dispatchEvent(new Event('resize'));
}

// Grouped navigation. A tab's button carries its group and a stable slug; the slug
// goes in the URL hash so a view can be linked (#win-rate), whatever the tab order.
function showGroup(group) {
  document.querySelectorAll('.nav-group, .nav-tabs').forEach(function(el) {
    el.classList.toggle('active', el.getAttribute('data-group') === group);
  });
}
function openTab(evt, tabId, keepHash) {
  var btn = document.querySelector('.tab-btn[data-tab="' + tabId + '"]');
  if (!btn) return;
  document.querySelectorAll('.tab-content').forEach(el => el.style.display = 'none');
  document.querySelectorAll('.tab-btn').forEach(el => el.classList.remove('active'));
  document.getElementById(tabId).style.display = 'block';
  btn.classList.add('active');
  showGroup(btn.getAttribute('data-group'));
  if (!keepHash && history.replaceState) history.replaceState(null, '', '#' + btn.getAttribute('data-slug'));
  window.dispatchEvent(new Event('resize'));
}
function openGroup(group) {
  var first = document.querySelector('.nav-tabs[data-group="' + group + '"] .tab-btn');
  if (first) openTab(null, first.getAttribute('data-tab'));
}
function openTabFromHash() {
  var slug = decodeURIComponent((location.hash || '').slice(1));
  if (!/^[a-z0-9-]+$/.test(slug)) return;
  var btn = document.querySelector('.tab-btn[data-slug="' + slug + '"]');
  if (btn) openTab(null, btn.getAttribute('data-tab'), true);
}
window.addEventListener('hashchange', openTabFromHash);

function applyPlatform(plat) {
  currentPlatform = plat;
  document.body.classList.toggle('raw-wr', isRawView(plat));
  // data-platform lists every view a block serves ("all all-raw" when both show the same charts).
  document.querySelectorAll('.platform-content').forEach(el => {
    el.style.display = (el.getAttribute('data-platform') || '').split(' ').indexOf(plat) >= 0 ? 'block' : 'none';
  });
  ['consist-detail-panel', 'sleeper-detail-panel'].forEach(function(id) {
    var detailPanel = document.getElementById(id);
    if (detailPanel) detailPanel.style.display = 'none';
  });
  // Resize every Plotly figure in the newly-visible blocks (figures rendered
  // while their container was display:none initialize at width 0 and stay
  // misaligned until explicitly resized).
  document.querySelectorAll('.platform-content[data-platform~="' + plat + '"] .js-plotly-plot')
    .forEach(function(el) { try { Plotly.Plots.resize(el); } catch (e) {} });
  window.dispatchEvent(new Event('resize'));
  bindConsistHandlers();
  bindSleeperHandlers();
  bindDeckClicks();
  bindRowSorts();
  populateDeckPicker(plat);
  if (currentDeckDetail) renderDeckDetail();   // same deck, new data source
  refreshCompare();
  refreshSimulator();
}

// Shared by the Consistency and Sleepers tabs: one horizontal bar per matchup,
// signed by net impact, red below the line and green above it.
function renderMatchupBreakdown(chartEl, breakdown, xTitle) {
  var labels = breakdown.map(function(m) { return m.opp_short; });
  var impacts = breakdown.map(function(m) { return m.net_impact; });
  var colors = breakdown.map(function(m) { return m.net_impact >= 0 ? FAV_COLOR : UNFAV_COLOR; });
  var raw = isRawView(currentPlatform);   // in a raw view the adjusted rate IS the raw one
  var hoverTexts = breakdown.map(function(m) {
    return '<b>vs ' + m.opponent + '</b><br>' +
           'Win Rate (' + (raw ? 'raw' : 'adjusted') + '): ' + m.win_rate + '%<br>' +
           (!raw && m.raw_wr != null ? 'Raw: ' + m.raw_wr + '%<br>' : '') +
           'Meta Share: ' + m.opp_pop + '%<br>' +
           'Games: ' + m.games + '<br>' +
           'Net Impact: ' + (m.net_impact >= 0 ? '+' : '') + m.net_impact.toFixed(3);
  });
  var trace = { y: labels, x: impacts, type: 'bar', orientation: 'h',
                marker: { color: colors, line: { width: 0 } },
                text: hoverTexts, hoverinfo: 'text' };
  var layout = {
    font: { family: UI.font, color: UI.text, size: 12 },
    paper_bgcolor: 'rgba(0,0,0,0)', plot_bgcolor: UI.surface,
    xaxis: { title: xTitle,
             zeroline: true, zerolinewidth: 2, zerolinecolor: UI.rule,
             gridcolor: UI.grid, linecolor: UI.rule,
             tickfont: { size: 11, color: UI.muted } },
    yaxis: { title: '', automargin: true, gridcolor: UI.grid, linecolor: UI.rule,
             tickfont: { size: 11, color: UI.muted } },
    height: Math.max(400, breakdown.length * 28),
    margin: { l: 140, r: 30, t: 10, b: 50 },
    hoverlabel: { bgcolor: 'white', bordercolor: UI.border,
                  font: { size: 12, color: UI.ink } },
  };
  Plotly.newPlot(chartEl, [trace], layout, {responsive: true});
}

// The sleeper scatter carries one trace per verdict, so the deck is read from
// customdata rather than a trace index. All-tier only -- no tier selector.
function bindSleeperHandlers() {
  var scatterEl = document.getElementById('sleeper-scatter-' + currentPlatform);
  if (!scatterEl || scatterEl._sleeperBound) return;
  scatterEl._sleeperBound = true;
  scatterEl.on('plotly_click', function(data) {
    var deckName = data.points[0].customdata[0];
    var perPlat = sleeperBreakdownPer[currentPlatform] || {};
    var breakdown = perPlat[deckName];
    if (!breakdown || breakdown.length === 0) return;
    var panel = document.getElementById('sleeper-detail-panel');
    document.getElementById('sleeper-detail-title').textContent =
      deckName + ' - vs the core meta';
    document.getElementById('sleeper-detail-open').setAttribute('data-deck', deckName);
    panel.style.display = 'block';
    renderMatchupBreakdown(
      document.getElementById('sleeper-detail-chart'), breakdown,
      'Net Impact vs the core meta (positive = favorable, negative = unfavorable)');
    panel.scrollIntoView({ behavior: 'smooth', block: 'nearest' });
  });
}

// Each view has its own scatter, drawn for the All tier; `_consistTier` records
// which tier it shows now, so a view switch can bring it in line with the select.
function showConsistTier(scatterEl, tier) {
  var d = consistScatterDataPer[currentPlatform][tier];
  if (!d || (scatterEl._consistTier || 'All') === tier) return;
  scatterEl._consistTier = tier;
  Plotly.restyle(scatterEl, {
    x: [d.x], y: [d.y], text: [d.text],
    'marker.color': [d.marker_colors], customdata: [d.customdata],
  }, [0]);
  Plotly.relayout(scatterEl, {
    'shapes[0].x1': d.max_val, 'shapes[0].y1': d.max_val,
  });
}

function bindConsistHandlers() {
  var scatterEl = document.getElementById('consist-scatter-' + currentPlatform);
  if (!scatterEl) return;
  showConsistTier(scatterEl, currentConsistTier);
  var tierSelect = document.getElementById('consist-tier-select');
  if (tierSelect) {
    var newSel = tierSelect.cloneNode(true);
    tierSelect.parentNode.replaceChild(newSel, tierSelect);
    newSel.value = currentConsistTier;
    newSel.addEventListener('change', function() {
      currentConsistTier = this.value;
      showConsistTier(scatterEl, currentConsistTier);
      var detailPanel = document.getElementById('consist-detail-panel');
      if (detailPanel) detailPanel.style.display = 'none';
    });
  }
  // The <select> above is cloned to drop its old listener, so it has to be
  // rebound every time; the scatter div itself persists, so registering its
  // click handler again would stack duplicates on every platform switch.
  if (scatterEl._consistBound) return;
  scatterEl._consistBound = true;
  scatterEl.on('plotly_click', function(data) {
    var deckName = data.points[0].customdata[0];
    var tierData = matchupBreakdownPer[currentPlatform][currentConsistTier];
    var breakdown = tierData && tierData[deckName];
    if (!breakdown || breakdown.length === 0) return;
    var panel = document.getElementById('consist-detail-panel');
    document.getElementById('consist-detail-title').textContent =
      deckName + ' - ' + tierLabels[currentConsistTier];
    document.getElementById('consist-detail-open').setAttribute('data-deck', deckName);
    panel.style.display = 'block';
    renderMatchupBreakdown(
      document.getElementById('consist-detail-chart'), breakdown,
      'Net Impact (positive = favorable, negative = unfavorable)');
    panel.scrollIntoView({ behavior: 'smooth', block: 'nearest' });
  });
}

// ---------------------------------------------------------------------------
// Deck detail drawer: one deck's numbers from every tab, on one card.
// ---------------------------------------------------------------------------
var deckDetailPer = __DD__JSON__;
var shortToSlug = __S2S__JSON__;
var ddCfg = __DDCFG__JSON__;
var currentDeckDetail = null;

function ddNA(msg) { return '<span class="dd-na">' + msg + '</span>'; }
function ddNum(v, digits, suffix) {
  return (v === null || v === undefined) ? ddNA('&mdash;') : v.toFixed(digits) + (suffix || '');
}
function ddSigned(v, digits, suffix) {
  if (v === null || v === undefined) return ddNA('&mdash;');
  var cls = v > 0 ? 'dd-pos' : (v < 0 ? 'dd-neg' : '');
  return '<span class="' + cls + '">' + (v > 0 ? '+' : '') + v.toFixed(digits) + (suffix || '') + '</span>';
}
function ddPill(text, color) {
  return '<span class="dd-pill" style="background:' + (color || UI.faint) + '">' + text + '</span>';
}
function ddHeld(v) {
  if (v === null || v === undefined) return '';
  return ' <span class="' + (v < 50 ? 'dd-warn' : 'dd-na') + '">held in ' + v.toFixed(0) + '% of resamples</span>';
}
function ddSection(title, rows) {
  return '<div class="dd-section"><div class="dd-section-title">' + title + '</div><div class="dd-grid">' +
    rows.map(function(r) { return '<span class="lbl">' + r[0] + '</span><span class="val">' + r[1] + '</span>'; }).join('') +
    '</div></div>';
}
// Rows for a Top Cut / Late Rounds record: win rate over decided games against
// what the deck's opponents in that slice predict, and the leftover shift.
function ddShift(s, minGames, label) {
  if (!s || !s.enough) {
    return [[label, ddNA((s ? s.games : 0) + ' decided games; the tab needs ' + minGames)]];
  }
  return [
    [label + ': actual vs expected', s.actual.toFixed(1) + '% vs ' + s.expected.toFixed(1) + '% ' +
      ddNA('(' + s.games + ' games)')],
    [label + ': adjusted shift', ddSigned(s.adj, 1, 'pp') + ' ' + ddNA('&plusmn; ' + (1.96 * s.se).toFixed(1))],
  ];
}

function renderDeckDetail() {
  if (!currentDeckDetail) return;
  var drawer = document.getElementById('deck-drawer');
  var title = document.getElementById('deck-drawer-title');
  var sub = document.getElementById('deck-drawer-sub');
  var body = document.getElementById('deck-drawer-body');
  var chartTitle = document.getElementById('deck-drawer-chart-title');
  var chart = document.getElementById('deck-drawer-chart');
  try { Plotly.purge(chart); } catch (e) {}
  chart.innerHTML = '';
  var d = (deckDetailPer[currentPlatform] || {})[currentDeckDetail];
  if (!d) {
    title.textContent = currentDeckDetail;
    title.style.color = UI.ink;
    sub.textContent = '';
    body.innerHTML = ddNA('No recorded games for this deck on this data source.');
    chartTitle.textContent = '';
    return;
  }
  drawer.style.borderLeftColor = d.color;
  title.textContent = d.short;
  title.style.color = d.color;
  sub.innerHTML = [d.name, d.family ? 'family: ' + d.family : null,
                   d.core ? 'core meta' : 'sleeper candidate'].filter(Boolean).join(' &middot; ');

  var o = d.overview, sl = d.sleeper, html = '';
  var tierHtml = o ? '<span class="tier-badge" style="background:' + ddCfg.tier[o.tier][0] + ';color:' +
                     ddCfg.tier[o.tier][1] + '">' + o.tier + '</span>' : ddNA('core only');
  var wr = o ? o.wr : (sl ? sl.wr : null);
  var wrSub = o && o.wr_lo !== null ? '95% ' + o.wr_lo.toFixed(1) + '&ndash;' + o.wr_hi.toFixed(1)
            : (sl ? 'vs the core, 95% ' + sl.lo.toFixed(1) + '&ndash;' + sl.hi.toFixed(1) : '');
  var kpis = [
    ['Tier', tierHtml, o && o.held !== null ? 'held in ' + o.held.toFixed(0) + '% of resamples' : ''],
    ['Win rate (3-pt)', ddNum(wr, 1, '%'), wrSub],
    ['Share of players', ddNum(d.share, 2, '%'), d.use_rate !== null ? 'Limitless use ' + d.use_rate.toFixed(2) + '%' : ''],
  ];
  html += '<div class="dd-kpis">' + kpis.map(function(k) {
    return '<div class="dd-kpi"><div class="l">' + k[0] + '</div><div class="v">' + k[1] + '</div><div class="s">' + k[2] + '</div></div>';
  }).join('') + '</div>';

  if (o) {
    html += ddSection('Performance', [
      ['All / Top 50% / Top 25%', ddNum(o.wr, 1, '%') + ' / ' + ddNum(o.wr_u50, 1, '%') + ' / ' + ddNum(o.wr_u25, 1, '%')],
      ['Games (All tier)', o.games !== null ? o.games.toLocaleString() : ddNA('&mdash;')],
      ['Simulated Swiss', ddNum(o.mean_mp, 1, ' MP') + ' &middot; cut ' + ddNum(o.p_cut, 1, '%')],
      ['Tier score', ddNum(o.score, 1)],
    ]);
  }
  if (sl) {
    html += ddSection('Sleeper scorecard', [
      ['Verdict', ddPill(sl.verdict, ddCfg.sleeper[sl.verdict])],
      ['WR vs core meta', sl.wr.toFixed(1) + '% ' + ddNA('(' + sl.lo.toFixed(1) + '&ndash;' + sl.hi.toFixed(1) + ')')],
      ['Coverage / games', sl.coverage + '% of core field &middot; ' + sl.games.toLocaleString() + ' games'],
      ['Would rank', '#' + sl.would_rank + ' among the core'],
    ]);
  }
  if (d.consistency) {
    var c = d.consistency;
    html += ddSection('Matchup spread', [
      ['Favorable / unfavorable', ddNum(c.fav, 2) + ' / ' + ddNum(c.unfav, 2) + ' ' + ddNA('(' + c.n_fav + ' / ' + c.n_unfav + ' matchups)')],
      ['Net impact', ddSigned(c.net, 2)],
      ['Matchup spread (&sigma;)', ddNum(c.spread, 1, 'pp')],
    ]);
  }
  if (d.roles) {
    var r = d.roles;
    html += ddSection('Meta roles', [
      ['Meta position', ddPill(r.meta, ddCfg.meta[r.meta]) + ddHeld(r.meta_held)],
      ['Tournament strategy', ddPill(r.tournament, ddCfg.tourn[r.tournament]) + ddHeld(r.tourn_held)],
      ['Share trend', r.trend && r.trend !== 'Unknown'
        ? r.trend + (r.pp_per_wk !== null ? ' ' + ddNA('(' + (r.pp_per_wk > 0 ? '+' : '') + r.pp_per_wk.toFixed(2) + 'pp/wk)') : '')
        : ddNA('too few weeks to fit')],
    ]);
  }
  var skillRows = [];
  if (o) skillRows.push(['Top 25% &minus; All ' + ddNA('(not a skill measure)'), ddSigned(o.skill_delta, 1, 'pp')]);
  skillRows.push(['Finish Curve slope', d.skill_curve
    ? ddSigned(d.skill_curve.slope, 1, 'pp/decade') + ' ' + ddNA('&plusmn; ' + (1.96 * d.skill_curve.se).toFixed(1))
    : ddNA('below the Finish Curve band/game bar')]);
  skillRows = skillRows.concat(ddShift(d.late, ddCfg.minLate, 'Rounds ' + ddCfg.lateRound + '+'));
  html += ddSection('Finishing position &amp; late rounds', skillRows);
  html += ddSection('Top cut', ddShift(d.cut, ddCfg.minCut, 'Cut'));
  if (d.conversion && d.conversion.length) {
    html += ddSection('Conversion ' + ddNA('(finishes / expected)'), d.conversion.map(function(c) {
      if (c.ratio === null) return [c.label, ddNA('no events of that size')];
      var cls = c.sig === 'up' ? 'dd-pos' : (c.sig === 'down' ? 'dd-neg' : '');
      return [c.label, '<span class="' + cls + '">' + c.ratio.toFixed(2) + '&times;</span> ' +
        ddNA(c.share.toFixed(1) + '% of finishes vs ' + c.exp_share.toFixed(1) + '% expected' +
             (c.reliable ? '' : '; layer is noise here'))];
    }));
  } else {
    html += ddSection('Conversion', [['Finishes / expected', ddNA('too few players in this window&rsquo;s standings')]]);
  }
  if (d.nash) {
    var n = d.nash;
    html += ddSection('Mispricing', [
      ['Verdict', ddPill(n.verdict, ddCfg.nash[n.verdict])],
      ['Share vs equilibrium', n.share.toFixed(2) + '% vs ' + n.median.toFixed(2) + '% ' +
        ddNA('(90% ' + n.q05.toFixed(1) + '&ndash;' + n.q95.toFixed(1) + ')')],
      ['In equilibrium', n.support + '% of draws &middot; wants more than share in ' + n.p_under + '%'],
      ['Edge vs current field', ddSigned(n.edge, 2, 'pp')],
    ]);
  } else {
    html += ddSection('Mispricing', [['Equilibrium', ddNA('fewer than ' + ddCfg.nashMin + ' games on this data source')]]);
  }
  if (d.twins && d.twins.length) {
    html += ddSection('Strategic twins', d.twins.map(function(t) {
      return ['<a href="#" class="deck-link" data-deck="' + t.deck + '">' + t.short + '</a>',
              'r = ' + t.r.toFixed(2) + (t.mutual ? '' : ' ' + ddNA('(one-way)'))];
    }));
  }
  body.innerHTML = html;

  var breakdown = d.core ? ((matchupBreakdownPer[currentPlatform] || {}).All || {})[d.deck]
                         : (sleeperBreakdownPer[currentPlatform] || {})[d.deck];
  if (breakdown && breakdown.length) {
    chartTitle.textContent = d.core ? 'Matchups: net impact vs the field (All players)'
                                    : 'Matchups: net impact vs the core meta';
    renderMatchupBreakdown(chart, breakdown, 'Net impact (positive = favorable)');
  } else {
    chartTitle.textContent = '';
  }
}

function openDeckDetail(slug) {
  currentDeckDetail = slug;
  document.getElementById('deck-drawer').classList.add('open');
  document.getElementById('deck-drawer').setAttribute('aria-hidden', 'false');
  document.getElementById('deck-drawer-backdrop').classList.add('open');
  var sel = document.getElementById('deck-detail-select');
  if (sel) sel.value = slug;
  renderDeckDetail();
  document.getElementById('deck-drawer').scrollTop = 0;
}

function closeDeckDetail() {
  currentDeckDetail = null;
  document.getElementById('deck-drawer').classList.remove('open');
  document.getElementById('deck-drawer').setAttribute('aria-hidden', 'true');
  document.getElementById('deck-drawer-backdrop').classList.remove('open');
  var sel = document.getElementById('deck-detail-select');
  if (sel) sel.value = '';
}

function populateDeckPicker(plat) {
  var sel = document.getElementById('deck-detail-select');
  if (!sel) return;
  var decks = deckDetailPer[plat] || {};
  var core = [], sleepers = [];
  Object.keys(decks).forEach(function(k) { (decks[k].core ? core : sleepers).push(decks[k]); });
  function opts(list) {
    return list.map(function(x) { return '<option value="' + x.deck + '">' + x.short + '</option>'; }).join('');
  }
  sel.innerHTML = '<option value="">Choose a deck&hellip;</option>' +
    (core.length ? '<optgroup label="Core meta">' + opts(core) + '</optgroup>' : '') +
    (sleepers.length ? '<optgroup label="Sleeper candidates">' + opts(sleepers) + '</optgroup>' : '');
  sel.value = currentDeckDetail && decks[currentDeckDetail] ? currentDeckDetail : '';
}

// A clicked mark names its deck in one of a few places depending on the chart:
// a slug in customdata, a short name on the axis, in the text label, or as the
// trace name. Accept whichever resolves to a deck on this platform.
function deckFromPoint(pt) {
  var known = deckDetailPer[currentPlatform] || {};
  var cands = [];
  if (pt.customdata !== undefined && pt.customdata !== null) {
    cands.push(Array.isArray(pt.customdata) ? pt.customdata[0] : pt.customdata);
  }
  cands.push(pt.y, pt.x, pt.text, pt.label, pt.data && pt.data.name);
  for (var i = 0; i < cands.length; i++) {
    if (typeof cands[i] !== 'string') continue;
    var c = cands[i].replace(/<[^>]+>/g, '').trim();
    if (known[c]) return c;
    if (shortToSlug[c] && known[shortToSlug[c]]) return shortToSlug[c];
  }
  return null;
}

// Figures opt in from Python with layout.meta.deckClick. Guarded per element:
// Plotly's .on() has no dedupe, and this runs again on every platform switch.
function bindDeckClicks() {
  document.querySelectorAll('.js-plotly-plot').forEach(function(gd) {
    if (gd._deckBound || !gd.layout || !gd.layout.meta || !gd.layout.meta.deckClick) return;
    gd._deckBound = true;
    gd.on('plotly_click', function(ev) {
      var slug = ev && ev.points && ev.points.length ? deckFromPoint(ev.points[0]) : null;
      if (slug) openDeckDetail(slug);
    });
  });
}

// Heatmaps opt in from Python with layout.meta.sortRows: their column headers are
// annotations named 'sort:<column>' ('sort:-1' is the built order), and a click sorts
// the rows by that column, high to low, then low to high on a second click. Rows with no
// value sort last either way. z, text, customdata and y move together, so hovers and
// deck clicks still point at the right row. Guarded per element like bindDeckClicks.
function bindRowSorts() {
  document.querySelectorAll('.js-plotly-plot').forEach(function(gd) {
    var cfg = gd.layout && gd.layout.meta && gd.layout.meta.sortRows;
    if (gd._sortBound || !cfg || !gd.data || !gd.data.length) return;
    gd._sortBound = true;
    var t = gd.data[0];
    var orig = {z: t.z.slice(), text: t.text.slice(), customdata: t.customdata.slice(), y: t.y.slice()};
    var state = {col: -1, desc: true};
    gd.on('plotly_clickannotation', function(ev) {
      var name = ev && ev.annotation ? String(ev.annotation.name || '') : '';
      if (name.indexOf('sort:') !== 0) return;
      var col = parseInt(name.slice(5), 10);
      state.desc = state.col === col ? !state.desc : true;
      state.col = col;
      var order = orig.y.map(function(_, i) { return i; });
      if (col >= 0) {
        order.sort(function(i, j) {
          var a = orig.customdata[i][col][cfg.key], b = orig.customdata[j][col][cfg.key];
          if (a == null || b == null) return (a == null) - (b == null) || i - j;
          return (state.desc ? b - a : a - b) || i - j;
        });
      } else if (!state.desc) {
        order.reverse();
      }
      var pick = function(arr) { return order.map(function(i) { return arr[i]; }); };
      Plotly.restyle(gd, {z: [pick(orig.z)], text: [pick(orig.text)],
                          customdata: [pick(orig.customdata)], y: [pick(orig.y)]}, [0]);
      var anns = (gd.layout.annotations || []).map(function(a) {
        var n = String(a.name || '');
        if (n.indexOf('sort:') !== 0) return a;
        var c = parseInt(n.slice(5), 10);
        var label = c >= 0 ? cfg.labels[c] : cfg['default'];
        var mark = c !== state.col ? cfg.marks.none : (state.desc ? cfg.marks.desc : cfg.marks.asc);
        return Object.assign({}, a, {text: label + ' ' + mark});
      });
      Plotly.relayout(gd, {annotations: anns});
    });
  });
}

document.addEventListener('click', function(ev) {
  var el = ev.target.closest ? ev.target.closest('[data-deck]') : null;
  if (!el) return;
  var slug = el.getAttribute('data-deck');
  if (!slug || !(deckDetailPer[currentPlatform] || {})[slug]) return;
  ev.preventDefault();
  openDeckDetail(slug);
});

document.addEventListener('keydown', function(ev) {
  if (ev.key === 'Escape' && currentDeckDetail) { closeDeckDetail(); return; }
  if (ev.key === 'Enter' && ev.target.classList && ev.target.classList.contains('ov-row')) {
    openDeckDetail(ev.target.getAttribute('data-deck'));
  }
});

document.addEventListener('DOMContentLoaded', function() {
  document.getElementById('platform-select').addEventListener('change', function() {
    applyPlatform(currentView());
  });
  document.getElementById('raw-wr-toggle').addEventListener('change', function() {
    applyPlatform(currentView());
  });
  document.getElementById('deck-detail-select').addEventListener('change', function() {
    if (this.value) openDeckDetail(this.value); else closeDeckDetail();
  });
  document.getElementById('deck-drawer-close').addEventListener('click', closeDeckDetail);
  document.getElementById('deck-drawer-backdrop').addEventListener('click', closeDeckDetail);
  populateDeckPicker(currentPlatform);
  bindConsistHandlers();
  bindSleeperHandlers();
  bindDeckClicks();
  bindRowSorts();
  refreshCompare();
  refreshSimulator();
  // A browser can restore the controls' last state on reload; show what they say.
  if (currentView() !== currentPlatform) applyPlatform(currentView());
  openTabFromHash();
});
// Figures are drawn by inline scripts; bind again once everything has run.
window.addEventListener('load', bindDeckClicks);
</script>
</body>
</html>"""

sim_json = jsonlib.dumps(simulator_data_per)

JS_HTML = (JS_HTML
    .replace('__UI__JSON__', jsonlib.dumps({**THEME, 'font': FONT_STACK}))
    .replace('__SIM__JSON__', sim_json)
    .replace('__DEFAULT_PLATFORM__', DEFAULT_PLATFORM)
    .replace('__RAW_SUFFIX__', RAW_SUFFIX)
    .replace('__MB__JSON__', mb_json)
    .replace('__SB__JSON__', sb_json)
    .replace('__CSD__JSON__', csd_json)
    .replace('__CMP__JSON__', cd_json)
    .replace('__DD__JSON__', jsonlib.dumps(deck_detail_per))
    .replace('__S2S__JSON__', jsonlib.dumps(short_to_slug))
    .replace('__DDCFG__JSON__', jsonlib.dumps({
        'tier': TIER_BADGE, 'meta': META_ROLE_COLORS, 'tourn': TOURN_ROLE_COLORS,
        'nash': NASH_VERDICT_COLORS, 'sleeper': SLEEPER_VERDICT_COLORS,
        'minCut': MIN_CUT_GAMES, 'minLate': MIN_LATE_GAMES, 'lateRound': LATE_ROUND,
        'nashMin': NASH_MIN_GAMES}))
    .replace('__FAV__', FAVORABLE_COLOR)
    .replace('__UNFAV__', UNFAVORABLE_COLOR))

html_template = HEAD_HTML + CSS_HTML.replace('<style>', '<style>\n' + THEME_CSS, 1) + BODY_HTML + JS_HTML

# dist/ is the published site (scripts/build_site.py adds the phone page beside it).
# A historical build (scripts/run_notebook.py) writes beside its own data instead.
output_path = os.environ.get('META_DASHBOARD_PATH') or 'dist/index.html'
os.makedirs(os.path.dirname(os.path.abspath(output_path)), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    f.write(html_template)

file_size = os.path.getsize(output_path) / (1024 * 1024)
print(f"Dashboard saved to {output_path} ({file_size:.1f} MB)")
print(f"Open in a browser; switch Data Source dropdown to filter by platform.")
